# SafeViChi v2 · SafeVi-DualORC-CRF

[![Architecture](https://img.shields.io/badge/Nhánh_2-SafeVi--DualORC--CRF-1d4ed8)](#sec-4)
[![Task](https://img.shields.io/badge/Tác_vụ-5_Nhãn_+_Bằng_chứng-0f766e)](#sec-4)
[![Dataset Commit](https://img.shields.io/badge/Data_Commit-371a0f1c-475569)](#sec-2)
[![Test Lock](https://img.shields.io/badge/Test_Suite-Locked_(1.670_mẫu)-b91c1c)](#sec-6)
[![License: MIT](https://img.shields.io/badge/License-MIT-15803d)](../LICENSE)

> **Tóm tắt:** Notebook huấn luyện, hiệu chuẩn và đánh giá nhánh kiến trúc song song hạng nhẹ **SafeVi-DualORC-CRF** cho bài toán phát hiện tin nhắn tiếng Việt nguy hại chạy cục bộ trên thiết bị (`< 2 GB`), phân loại **5 nhãn theo tình huống** (`AN_TOAN`, `XUC_PHAM`, `QUAY_ROI`, `DE_DOA`, `LUA_DAO`) và trích xuất **cụm từ bằng chứng chép nguyên văn (verbatim)**. Quy trình phân hoạch (`split_ids.json`), hiệu chuẩn và đánh giá trên bộ test khóa được thiết kế đồng nhất với Nhánh 1 ([`v2_guardmerge_colab.ipynb`](v2_guardmerge_colab.ipynb)) tại commit dữ liệu cố định `371a0f1caf3f7b62f62b87dbf557c66eda9c544a`.

---

### Hồ sơ thí nghiệm (Experiment Specification)

| Hạng mục | Thông số kỹ thuật |
| :--- | :--- |
| **Kiến trúc** | `SafeVi-DualORC-CRF` (Encoder + UTF-8 Byte-CNN + HAT + BIO-CRF mức từ + ORC) |
| **Backbone ứng viên (`BACKBONE_PRESET`)** | `visobert` (98M, 12 lớp) · `bamibert` (102M, 12 lớp, 1.024/2.048 tok, `lr_backbone=5e-5`) · `mmbert_base` (307M, 22 lớp, `lr_backbone=4.5e-5`) · `mmbert_small` (140M, 22 lớp) |
| **Nguồn dữ liệu khóa (`DATA_REV`)** | Commit `371a0f1caf3f7b62f62b87dbf557c66eda9c544a` (`so_sanh.jsonl`: 13.392 đoạn hội thoại `C0`, 1–8 lượt) |
| **Bộ Test đánh giá (Khóa cứng)** | `test_day_du.jsonl` (`334 C0`) + `test_perturb_C1.jsonl` (`668 C1`) + `test_perturb_C2.jsonl` (`668 C2`) = **`1.670` mẫu** |
| **Chế độ suy luận song song** | **Dòng `#5`** (đầu vào thô) & **Dòng `#6`** (chuẩn hóa 5 tầng giữ dòng `\n`, ánh xạ ngược span `non_verbatim == 0`) |
| **Tính tái hiện (Reproducibility)** | `SEED = 42`, `split_seed = 42`, ghim phiên bản thư viện, cơ chế khóa 1 lần `REAL_TEST_LOCK.json` & xác thực SHA-256 `calibration.json` |
| **Yêu cầu phần cứng** | Google Colab / Kaggle GPU CUDA (`>= 12 GiB` VRAM khuyến nghị, T4 / L4 / A100; ổ tạm `>= 4 GiB`) |

---

### Mục lục (Table of Contents)

| Mục | Nội dung | Tệp xuất chính |
| :---: | :--- | :--- |
| **1** | [Thiết lập môi trường, Kiểm tra GPU & Bảng điều khiển Cấu hình](#sec-1) | `config.json`, `environment.json` |
| **2** | [Nạp dữ liệu ghim SHA-256 & Phân hoạch sạch không rò rỉ](#sec-2) | `split_ids.json` |
| **3** | [Tiền xử lý hội thoại, Tăng cường C1 phi điểm mù & DataLoader](#sec-3) | Tự kiểm định Offset / BIO / Aug |
| **4** | [Kiến trúc `SafeVi-DualORC-CRF` & Kiểm định toán học](#sec-4) | Kiểm định CRF $3^W$ & Padding |
| **5** | [Huấn luyện mô hình, Hàm mất mát đa nhiệm & Chọn Checkpoint](#sec-5) | `model_best.safetensors`, `last.pt`, `history.json` |
| **6** | [Hiệu chuẩn xác suất, Đánh giá phát triển & Cổng khóa Bộ Test](#sec-6) | `calibration.json`, `dev_metrics.json`, `REAL_TEST_LOCK.json`, `real_test_*` |
| **7** | [Kiểm thử độ bền (Stress Tests), Ngữ cảnh đa lượt & Benchmark](#sec-7) | `*_stress_tests.json`, `*_context_checks.json`, `benchmark.json` |
| **8** | [Lượng tử hóa INT8 trên CPU & Xuất đồ thị ONNX](#sec-8) | `int8_comparison.json`, `dualorc_core.onnx`, `crf_params.json` |
| **9** | [Đối chiếu liên nhánh (`GuardMerge`) & `VietSafe-HierBi-Cascade`](#sec-9) | `cascade_vs_guardmerge_*.json` |
| **10** | [Phụ lục: Quy trình thực nghiệm, Đối chiếu API & Tái kiểm chứng CLI](#sec-10) | Lệnh `python -m src.v2.eval.evaluate` |

---

### Sơ đồ luồng thực nghiệm (End-to-End Pipeline)

```text
  [1. Môi trường & Cấu hình] ──▶ [2. Dữ liệu SHA-256 & Split sạch] ──▶ [3. Tokenizer + C1 Aug + LEG]
                                                                                  │
  [6. Hiệu chuẩn (cal) & Khóa Test] ◀── [5. Train: LLRD + JS + FGM] ◀── [4. DualORC-CRF + Verify]
               │
               ├──▶ [7. Stress (5 bộ + #6), Context & Benchmark VRAM/Latency]
               ├──▶ [8. Nén INT8 CPU (PerRowInt8Embedding) & Xuất ONNX]
               └──▶ [9. Kiểm định McNemar & Bộ định tuyến UCCI Cascade]
```

### Chiến lược phân hoạch dữ liệu (3 chế độ không rò rỉ)

| Chế độ (`DATA_SCHEME`) | `fit` | `dev` | `cal` | Mục đích sử dụng |
| :--- | :---: | :---: | :---: | :--- |
| **`dev_80_10_10`** *(mặc định)* | 10.713 (80%) | 1.340 (10%) | 1.339 (10%) | So sánh ứng viên song song & chạy ablation; khớp 100% ID với Nhánh 1. |
| **`final_90_10`** | 12.053 (90%) | — | 1.339 (10%) | Huấn luyện toàn phần đủ `EPOCHS` cố định (không dùng `cal` để chọn epoch). |
| **`final_100_human_cal`** | 13.392 (100%) | — | `HUMAN_CAL_PATH` | Huấn luyện trên toàn bộ `so_sanh.jsonl` khi có tập validation người gán độc lập (khóa chặn 6 tệp train/test/C1/C2). |

### 4 Trụ cột Thiết kế Kiến trúc & Huấn luyện

1. **Backbone Encoder đa dạng + Depth-Normalized LLRD + UTF-8 Byte-CNN:** Hỗ trợ chuyển đổi qua `BACKBONE_PRESET` (`visobert`, `bamibert` với `lr_backbone=5e-5`, `mmbert_base` với `lr_backbone=4.5e-5`, `mmbert_small`). Cơ chế **Depth-Normalized LLRD (`LLRD = None`)** duy trì tỷ lệ LR tầng đáy/đỉnh cố định (`~36,8%`) nhất quán cho cả encoder 12 lớp (`0,920`) và 22 lớp (`0,956`).
2. **Tăng cường trực tuyến phi điểm mù & Dual-View JS Consistency:** Áp dụng `ACCENT_AUG = 0,30`, `SURFACE_AUG = 0,20` (từ `>= 3` ký tự, ngân sách theo độ dài dòng, chỉ dùng quy tắc C1 đơn nghĩa `a/e/i/o`, `.-_`, kéo dài `>= 3`, 17 teencode đơn nghĩa). Giữ **hoàn toàn ngoài train** 6 kiểu điểm mù `C2` (`blindspot.py`), `go_sai`, `dinh_tu`, `5/9/8`, `·` và teencode/phương ngữ held-out. Đánh giá song song **Dòng `#5` (thô)** và **Dòng `#6` (+ chuẩn hóa 5 tầng giữ dòng `\n`)** khi `EVAL_WITH_NORMALIZER = True`.
3. **Bằng chứng BIO-CRF + LEG mức từ & Hiệu chuẩn sát phân phối thật:** Áp `LEG_POLAR` trực tiếp ở mức từ, chọn bộ giải mã bằng chứng chống phình cụm trên `cal` kèm fallback cùng dòng (`fallback_min_prob`), báo cáo song song F1 mức từ và **F1 theo ý** ([`DO_DO_BANG_CHUNG.md`](../docs/v2/DO_DO_BANG_CHUNG.md)). Tiêu chí chọn checkpoint `real_aligned` sử dụng công thức tổng hợp `0.50 * macro_f1 + 0.15 * that_macro_f1 + 0.25 * that_acc + 0.10 * ev_f1`; hiệu chuẩn nhiệt độ trên `internal_llm` áp dụng trần trọng số `cap = 3.0` và biên `T >= 0.74`.
4. **Đo VRAM sạch & Đóng gói On-Device INT8 / ONNX:** Giải phóng toàn bộ `opt`/`scheduler`/`scaler`/FGM trước `benchmark()`, nén 4× bảng từ vựng bằng `PerRowInt8Embedding` (scale FP32), hỗ trợ xuất ONNX opset 17 và kiểm định McNemar/UCCI Cascade.

---

<a id="sec-1"></a>
## 1. Thiết lập môi trường, Kiểm tra GPU & Bảng điều khiển Cấu hình

### 1.1. Cài đặt các gói phụ thuộc đã ghim phiên bản

Chạy ô dưới đây để cài đúng bộ thư viện ghim (`transformers==4.57.6`, `tokenizers==0.22.2`, `safetensors==0.6.2`, `huggingface-hub==0.36.0`, `scikit-learn==1.9.1`). Nếu môi trường vừa thay đổi gói có sẵn, chọn **Runtime → Restart session** trước khi chạy tiếp.

In [ ]:
%pip -q install "transformers==4.57.6" "tokenizers==0.22.2" "safetensors==0.6.2" "huggingface-hub==0.36.0" "scikit-learn==1.9.1" "sentencepiece>=0.2.0"

### 1.2. Kiểm tra phần cứng GPU & Bảng điều khiển Cấu hình Thí nghiệm

Hai ô code tiếp theo thực hiện tuần tự:
1. **Ô 1.2a (Kiểm tra phần cứng):** Xác nhận GPU CUDA, dung lượng VRAM và ổ đĩa tạm (`/content` tối thiểu `4 GiB`). Nếu dừng do thiếu GPU, vào **Runtime → Change runtime type → GPU** rồi chạy lại.
2. **Ô 1.2b (Bảng điều khiển cấu hình & Khởi tạo phiên):** Nạp thư viện, khai báo toàn bộ tham số thí nghiệm, xác thực phiên bản gói ghim, cố định `SEED = 42`, gắn Google Drive và sinh thư mục `RUN` theo mã băm SHA-256 của `config.json`.

| Tham số trọng tâm | Mặc định | Ý nghĩa & Lưu ý an toàn |
| :--- | :---: | :--- |
| `RUN_MODE` | `"smoke"` | `"smoke"` kiểm thử nhanh 1 epoch (~15 giây); đổi `"full"` để huấn luyện đầy đủ 5 epoch. |
| `BACKBONE_PRESET` | `"visobert"` | Chọn một trong 4 backbone ứng viên: `"visobert"`, `"bamibert"`, `"mmbert_base"`, hoặc `"mmbert_small"`. |
| `DATA_SCHEME` | `"dev_80_10_10"` | `"dev_80_10_10"` (so sánh ứng viên) · `"final_90_10"` · `"final_100_human_cal"`. |
| `EVALUATE_REAL_TEST` | `False` | **Khóa cứng** bộ test 1.670 mẫu; không nằm trong `RUN` hash để mở khóa **đúng 1 lần duy nhất** trên cùng checkpoint đã đóng băng `calibration.json`. |
| `EVAL_WITH_NORMALIZER` | `True` | Chấm song song **Dòng `#5` (thô)** và **Dòng `#6` (+ chuẩn hóa 5 tầng giữ dòng `\n`)**. |

> **Lưu ý tái hiện:** Khi khôi phục phiên hoặc mở khóa test trên một runtime Colab khác, ghim `BACKBONE_REV` theo mã SHA in ở cuối Ô 1.2b để giữ nguyên 100% `RUN` hash.

In [ ]:
#@title 1.2a. Kiểm tra GPU CUDA, VRAM & Dung lượng đĩa tạm

from pathlib import Path
import shutil
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Chưa có GPU CUDA. Chọn Runtime → Change runtime type → GPU, sau đó chạy lại.")

gpu_name = torch.cuda.get_device_name(0)
gpu_gib = torch.cuda.get_device_properties(0).total_memory / 2**30
free_gib = shutil.disk_usage("/content" if Path("/content").exists() else ".").free / 2**30
print(f"GPU: {gpu_name} | VRAM: {gpu_gib:.1f} GiB | Ổ tạm còn trống: {free_gib:.1f} GiB")
if free_gib < 4.0:
    raise RuntimeError("Ổ tạm còn dưới 4 GiB. Xóa bớt tệp trước khi tải trọng số.")
if gpu_gib < 12.0:
    print("CẢNH BÁO: VRAM dưới 12 GiB. Notebook tự giảm BATCH và bật gradient checkpointing.")

In [ ]:
#@title 1.2b. Bảng điều khiển Cấu hình Thí nghiệm & Khởi tạo Phiên (RUN Hash)
# ==============================================================================
# 1. NHẬP THƯ VIỆN & GÓI PHỤ THUỘC
# ==============================================================================
import os, sys, gc, json, hashlib, random, math, time, platform, unicodedata, re, copy
from pathlib import Path
from collections import Counter, defaultdict
from contextlib import nullcontext
from itertools import islice
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import classification_report, confusion_matrix, f1_score, log_loss
from sklearn.isotonic import IsotonicRegression
from scipy.optimize import minimize_scalar
from scipy.special import softmax
from scipy.stats import binomtest
from huggingface_hub import HfApi
from safetensors.torch import save_file, load_file
from transformers import AutoConfig, AutoModel, AutoTokenizer
import transformers, requests, sklearn, scipy, tokenizers, huggingface_hub, safetensors

# ==============================================================================
# 2. CẤU HÌNH THAM SỐ THÍ NGHIỆM
# ==============================================================================
#@markdown **Chế độ chạy & Backbone**
RUN_MODE = "smoke" #@param ["smoke", "full"]  # "smoke" = kiểm thử nhanh 1 epoch; đổi thành "full" để train thật
USE_DRIVE = True #@param {type:"boolean"}
BACKBONE_PRESET = "visobert" #@param ["visobert", "bamibert", "mmbert_base", "mmbert_small"]  # "visobert" (98M, 12 lớp) | "bamibert" (102M, 12 lớp, ngữ cảnh 1024/2048) | "mmbert_base" (307M, 22 lớp) | "mmbert_small" (140M, 22 lớp)
BACKBONE_REV = None            # None = tự tra cứu & ghim SHA mới nhất từ HF Hub khi khởi chạy

#@markdown **Chiến lược dữ liệu, Chọn checkpoint & Hiệu chuẩn**
# Chiến lược dữ liệu sạch, chọn checkpoint, hiệu chuẩn & giải mã bằng chứng
DATA_SCHEME = "dev_80_10_10" #@param ["dev_80_10_10", "final_90_10", "final_100_human_cal", "legacy_70_10_10_10"]  # 80/10/10 | 90/10 | 100% + human_cal | legacy
CKPT_SELECT_MODE = "real_aligned" #@param ["real_aligned", "macro_only"]  # "real_aligned" (0.50 macro + 0.15 that_f1 + 0.25 that_acc + 0.10 ev) | "macro_only"
CAL_WEIGHT_MODE = "real_aligned" #@param ["real_aligned", "uniform"]  # "real_aligned" (T: 70% thật + 30% sinh khó; bằng chứng: cân bằng 5 lớp) | "uniform"
HUMAN_CAL_PATH = None              # Đường dẫn JSONL validation người gán độc lập (tách khỏi train & test); bắt buộc nếu DATA_SCHEME="final_100_human_cal"
EVIDENCE_DECODE_MODE = "auto" #@param ["auto", "marginal", "crf"]  # "auto" (chọn tốt nhất không phình cụm trên cal) | "marginal" | "crf"
QUANTIZE_EMBEDDINGS = True #@param {type:"boolean"}  # Nén cả nn.Embedding (INT8 theo hàng) trong ô INT8 CPU (giảm 750 -> 188 MiB trên mmbert_base)

#@markdown **Khóa cứng Bộ Test & Chấm song song #5/#6 (KHÔNG đưa vào RUN hash)**
# Khóa cứng tập test thật (KHÔNG đưa vào config hash để mở khóa 1 lần trên đúng RUN đã train!)
EVALUATE_AUDIT = False #@param {type:"boolean"}  # Chỉ dùng nếu DATA_SCHEME == "legacy_70_10_10_10"
EVALUATE_REAL_TEST = False #@param {type:"boolean"}  # KHÓA CỨNG: chỉ bật True ĐÚNG 1 LẦN DUY NHẤT sau khi hoàn tất chọn cấu hình trên dev và đóng băng calibration.json
RUN_REAL_TEST_PERTURB = True #@param {type:"boolean"}  # Khi mở khóa real_test, chấm trọn bộ test_perturb_C1 (668) + test_perturb_C2 (668) và 5 bộ stress
EVAL_WITH_NORMALIZER = True #@param {type:"boolean"}  # Tự động chấm thêm Dòng #6 (#5 + chuẩn hóa 5 tầng giữ dòng, ánh xạ ngược span verbatim; KHÔNG đưa vào hash)

#@markdown **Cờ kiến trúc & Ablation (Yêu cầu #3)**
# Cờ kiến trúc (tắt từng cờ để chạy ablation Yêu cầu #3)
USE_BYTE_CNN = True #@param {type:"boolean"}  # Luồng song song Word-Level Multi-Scale UTF-8 Byte-CNN
USE_HAT_ZOOM = True #@param {type:"boolean"}  # HAT Turn Self-Attention + Turn-to-Word Cross-Attention
USE_BIO_CRF = True #@param {type:"boolean"}  # Word-Level Partial-Label BIO-CRF có ràng buộc biên dòng \n
USE_ORC = True #@param {type:"boolean"}  # Chuỗi suy luận 4 bước ORC + FlexGuard severity + trộn 70/30
NUM_SUBCENTERS = 3             # Multi-Prototype Sub-Centers/lớp (1 để ablation đơn tâm)

# Cờ hàm mất mát & huấn luyện bền vững (giữ C2 điểm mù và held-out C1 hoàn toàn ngoài train)
SPAN_WEIGHT = 0.45             # Trọng số nhánh trích xuất bằng chứng (0.0 để tắt)
SPAN_DICE_WEIGHT = 0.30        # Soft Dice phụ trợ trên xác suất biên P(B)+P(I)
LEG_POLAR_WEIGHT = 0.15        # Trọng số phân cực từ vựng LEG áp ở MỨC TỪ (tính duy nhất trên fit; 0.0 để tắt)
ORC_AUX_WEIGHT = 0.20          # Tổng trọng số nhánh phụ Harm BCE + Ordinal Severity
MUTEX_WEIGHT = 0.10            # Soft Mutual-Exclusion (SingProbe) chống Safe & Risk cùng cao
PROTO_WEIGHT = 0.08            # Prototype-Regularized Supervised Contrastive (0.0 để tắt)
CONSISTENCY_WEIGHT = 0.25      # Dual-View JS Consistency giữa bản gốc & bản bỏ dấu/C1
STRATUM_WEIGHT = True          # Tăng nhẹ trọng số nhóm câu thật hiếm (QUAY_ROI|that, DE_DOA|that)
REAL_RARE_CAP = 2.5            # Trần trọng số nhóm thật hiếm sau chuẩn hóa
LABEL_SMOOTHING = 0.02         # Làm mượt nhẹ để giữ hiệu chuẩn nhiệt độ sắc nét
FOCAL_GAMMA = 1.0              # Focal gamma cho phân loại (0.0 = Cross-Entropy thường)
ACCENT_AUG = 0.30              # Tỷ lệ cặp ghép bỏ dấu mỗi epoch (0.0 để ablation)
SURFACE_AUG = 0.20             # Tỷ lệ cặp ghép biến dạng C1 đơn nghĩa phi điểm mù mỗi epoch (0.0 để ablation)
TOKEN_DROP_PROB = 0.08         # Xác suất che ngẫu nhiên subword → <unk>/<mask> khi train
FGM_EPS = 0.5                  # Nhiễu đối kháng embedding FGM (bật từ epoch 2; 0.0 để tắt)
LLRD = None                    # None = tự chuẩn hóa theo chiều sâu từ LLRD_BOTTOM_RATIO (0.92 cho 12 lớp, 0.9555 cho 22 lớp)
LLRD_BOTTOM_RATIO = 0.92 ** 12 # Tỷ lệ LR tầng đáy / tầng đỉnh cố định (~0.3677) cho mọi backbone

#@markdown **Tùy chọn Đóng gói On-Device & So sánh Liên nhánh**
# Siêu tham số tối ưu & tùy chọn đánh giá / so sánh
SEED = 42
WIDTH = 256
MAX_WORD_BYTES = 24
MAX_SUBWORDS_PER_WORD = 12
EPOCHS = 5 if RUN_MODE == "full" else 1
RUN_INT8_CPU = False #@param {type:"boolean"}  # Bật True để đánh giá lượng tử hóa INT8 trên CPU & đo dung lượng tệp
EXPORT_ONNX = False #@param {type:"boolean"}  # Bật True để xuất mô hình ONNX (cần gói onnx/onnxruntime)
GUARDMERGE_RUN_DIR = None      # Đường dẫn thư mục kết quả Nhánh 1 để chạy McNemar + UCCI Cascade


# ==============================================================================
# 3. THÔNG SỐ BACKBONE & DEPTH-NORMALIZED LLRD
# ==============================================================================
BACKBONE_SPECS = {
    "visobert": dict(repo="uitnlp/visobert", max_len=512, num_layers=12, lr_backbone=3e-5, lr_head=3e-4),
    "bamibert": dict(repo="Qualcomm-AI-Research/BamiBERT", max_len=1024, num_layers=12, lr_backbone=5e-5, lr_head=3e-4),
    "mmbert_base": dict(repo="jhu-clsp/mmBERT-base", max_len=512, num_layers=22, lr_backbone=4.5e-5, lr_head=3e-4),
    "mmbert_small": dict(repo="jhu-clsp/mmBERT-small", max_len=512, num_layers=22, lr_backbone=5e-5, lr_head=4e-4),
}
assert BACKBONE_PRESET in BACKBONE_SPECS, f"BACKBONE_PRESET không hợp lệ: {BACKBONE_PRESET}"
SPEC = BACKBONE_SPECS[BACKBONE_PRESET]
BACKBONE_ID = SPEC["repo"]
MAX_TOKENS = SPEC["max_len"]
LR_BACKBONE = SPEC["lr_backbone"]
LR_HEAD = SPEC["lr_head"]

def resolve_effective_llrd(num_layers, llrd=LLRD, bottom_ratio=LLRD_BOTTOM_RATIO):
    if llrd is not None:
        return float(llrd)
    return round(float(bottom_ratio) ** (1.0 / max(int(num_layers), 1)), 6)

LLRD_EFFECTIVE = resolve_effective_llrd(SPEC["num_layers"], LLRD, LLRD_BOTTOM_RATIO)


# ==============================================================================
# 4. KIỂM TRA PHẦN CỨNG, RÀNG BUỘC CẤU HÌNH & THƯ VIỆN GHIM
# ==============================================================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    raise RuntimeError("Notebook cần GPU. Chọn Runtime → Change runtime type → GPU rồi chạy lại.")
gpu_gib = torch.cuda.get_device_properties(0).total_memory / 2**30
GRAD_CKPT = gpu_gib < 12.0 or (BACKBONE_PRESET == "mmbert_base" and gpu_gib < 14.0)
BATCH, ACCUM = (16, 1) if (gpu_gib >= 12.0 and BACKBONE_PRESET != "mmbert_base") else (8, 2)
EVAL_BATCH = 32
MAX_MICROBATCH_TOKENS = 4096 if gpu_gib >= 12.0 else 2048

assert RUN_MODE in {"full", "smoke"}
assert DATA_SCHEME in {"dev_80_10_10", "final_90_10", "final_100_human_cal", "legacy_70_10_10_10"}
if DATA_SCHEME == "final_100_human_cal" and not HUMAN_CAL_PATH:
    raise ValueError("DATA_SCHEME='final_100_human_cal' bắt buộc phải đặt HUMAN_CAL_PATH (validation người gán độc lập) để không rò rỉ hiệu chuẩn!")
assert CKPT_SELECT_MODE in {"real_aligned", "macro_only"}
assert CAL_WEIGHT_MODE in {"real_aligned", "uniform"}
assert EVIDENCE_DECODE_MODE in {"auto", "marginal", "crf"}
assert 0 <= ACCENT_AUG <= 1 and 0 <= SURFACE_AUG <= 1 and (ACCENT_AUG + SURFACE_AUG) <= 1.0
assert 0 <= SPAN_DICE_WEIGHT <= 1 and 0 <= TOKEN_DROP_PROB < 0.5
assert 0 <= LABEL_SMOOTHING < 0.5 and REAL_RARE_CAP >= 1.0 and NUM_SUBCENTERS >= 1
assert SPAN_WEIGHT >= 0 and CONSISTENCY_WEIGHT >= 0 and FGM_EPS >= 0
assert 0.5 <= LLRD_EFFECTIVE <= 1.0 and 0.05 <= LLRD_BOTTOM_RATIO <= 1.0

native_bf16 = DEVICE.type == "cuda" and torch.cuda.is_bf16_supported(including_emulation=False)
if native_bf16:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
DTYPE = (torch.bfloat16 if native_bf16 else torch.float16) if DEVICE.type == "cuda" else torch.float32

pinned = {
    "transformers": (transformers.__version__, "4.57.6"),
    "tokenizers": (tokenizers.__version__, "0.22.2"),
    "huggingface-hub": (huggingface_hub.__version__, "0.36.0"),
    "safetensors": (safetensors.__version__, "0.6.2"),
    "scikit-learn": (sklearn.__version__, "1.9.1"),
}
for package, (active, expected) in pinned.items():
    if active != expected:
        raise RuntimeError(f"{package} đang dùng {active}, cần {expected}. Chạy ô cài đặt, khởi động lại kernel rồi Run all.")


# ==============================================================================
# 5. CỐ ĐỊNH SEED, GẮN DRIVE & KHỞI TẠO THƯ MỤC RUN (CONFIG / ENV)
# ==============================================================================
def seed_all(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)
seed_all(SEED)

if BACKBONE_REV is None:
    BACKBONE_REV = HfApi().model_info(BACKBONE_ID).sha

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
ROOT = Path("/content/drive/MyDrive/SafeViChi/dualorc_crf") if USE_DRIVE else Path.cwd() / "dualorc_runs"
DATA_REV = "371a0f1caf3f7b62f62b87dbf557c66eda9c544a"
LABELS = ["AN_TOAN", "XUC_PHAM", "QUAY_ROI", "DE_DOA", "LUA_DAO"]
SEVERITY_TARGETS = [0.0, 0.45, 0.65, 1.00, 0.85]  # Thứ tự ưu tiên: DE_DOA > LUA_DAO > QUAY_ROI > XUC_PHAM > AN_TOAN

config = dict(
    arch="SafeVi-DualORC-CRF",
    schema=3,
    backbone_preset=BACKBONE_PRESET,
    backbone=BACKBONE_ID,
    backbone_rev=BACKBONE_REV,
    max_tokens=MAX_TOKENS,
    data_scheme=DATA_SCHEME,
    ckpt_select_mode=CKPT_SELECT_MODE,
    ckpt_select_weights=(([0.50, 0.15, 0.25, 0.10] if SPAN_WEIGHT > 0 else [0.55, 0.20, 0.25, 0.0]) if CKPT_SELECT_MODE == "real_aligned" else None),
    cal_weight_mode=CAL_WEIGHT_MODE,
    human_cal_sha256=(hashlib.sha256(Path(HUMAN_CAL_PATH).read_bytes().removeprefix(b"\xef\xbb\xbf").replace(b"\r\n", b"\n")).hexdigest() if HUMAN_CAL_PATH else None),
    evidence_decode_mode=EVIDENCE_DECODE_MODE,
    use_byte_cnn=USE_BYTE_CNN,
    use_hat_zoom=USE_HAT_ZOOM,
    use_bio_crf=USE_BIO_CRF,
    use_orc=USE_ORC,
    num_subcenters=NUM_SUBCENTERS,
    span_weight=SPAN_WEIGHT,
    span_dice=SPAN_DICE_WEIGHT,
    leg_polar_weight=LEG_POLAR_WEIGHT,
    orc_aux_weight=ORC_AUX_WEIGHT,
    mutex_weight=MUTEX_WEIGHT,
    proto_weight=PROTO_WEIGHT,
    consistency_weight=CONSISTENCY_WEIGHT,
    stratum_weight=STRATUM_WEIGHT,
    real_rare_cap=REAL_RARE_CAP,
    label_smoothing=LABEL_SMOOTHING,
    focal_gamma=FOCAL_GAMMA,
    accent_aug=ACCENT_AUG,
    surface_aug=SURFACE_AUG,
    token_drop_prob=TOKEN_DROP_PROB,
    fgm_eps=FGM_EPS,
    llrd=LLRD,
    llrd_bottom_ratio=round(LLRD_BOTTOM_RATIO, 6),
    llrd_effective=LLRD_EFFECTIVE,
    dataset=DATA_REV,
    mode=RUN_MODE,
    seed=SEED,
    split_seed=42,
    epochs=EPOCHS,
    batch=BATCH,
    accum=ACCUM,
    max_microbatch_tokens=MAX_MICROBATCH_TOKENS,
    grad_ckpt=GRAD_CKPT,
    width=WIDTH,
    lr_backbone=LR_BACKBONE,
    lr_head=LR_HEAD,
    labels=LABELS,
    dtype=str(DTYPE),
)

def digest(b): return hashlib.sha256(b).hexdigest()
RUN = ROOT / digest(json.dumps(config, sort_keys=True).encode())[:14]
RUN.mkdir(parents=True, exist_ok=True)

def write_json(path, obj):
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf8")
    os.replace(tmp, path)

write_json(RUN / "config.json", config)
environment = dict(
    python=sys.version,
    torch=str(torch.__version__),
    cuda=torch.version.cuda,
    device=torch.cuda.get_device_name() if DEVICE.type == "cuda" else platform.processor(),
    capability=torch.cuda.get_device_capability() if DEVICE.type == "cuda" else None,
    gpu_gib=round(gpu_gib, 2),
    native_bf16=native_bf16,
    dtype=str(DTYPE),
    fp16_scaler=DTYPE == torch.float16,
    backbone=BACKBONE_ID,
    backbone_rev=BACKBONE_REV,
    llrd_effective=LLRD_EFFECTIVE,
    data_scheme=DATA_SCHEME,
    numpy=np.__version__,
    scipy=scipy.__version__,
    pandas=pd.__version__,
    requests=requests.__version__,
    **{name: active for name, (active, _) in pinned.items()},
)
write_json(RUN / "environment.json", environment)
print(RUN, f"BACKBONE={BACKBONE_PRESET} ({BACKBONE_REV[:8]}) SCHEME={DATA_SCHEME} LLRD_EFF={LLRD_EFFECTIVE:.4f} BATCH={BATCH}x{ACCUM}", environment)
print(f"Ghi chú khôi phục/mở khóa test trên phiên khác: ghim BACKBONE_REV = {BACKBONE_REV!r} để giữ nguyên RUN hash.")

---
<a id="sec-2"></a>
## 2. Nạp dữ liệu ghim SHA-256 & Phân hoạch sạch không rò rỉ

Ô này tải dữ liệu sạch từ commit cố định `371a0f1caf3f7b62f62b87dbf557c66eda9c544a`, đối chiếu **SHA-256** cho cả `so_sanh.jsonl` và `train.jsonl` (`13.392` đoạn `C0`), đồng thời kiểm tra toàn vẹn cấu trúc hội thoại (1–8 lượt, 5 nhãn hợp lệ, bằng chứng verbatim, 0 kịch bản held-out `LD7/LD12/DD6/QR6`).

| Bảo đảm tính toàn vẹn | Cơ chế áp dụng trong ô 2.1 |
| :--- | :--- |
| **Khớp 100% phân hoạch liên nhánh** | `StratifiedGroupKFold(n_splits=10, random_state=42)` phân tầng theo `(nhãn, loại)` và gom nhóm theo `split_group`, khớp hoàn toàn `split_ids.json` với Nhánh 1 (`GuardMerge`). |
| **Cách ly tuyệt đối Bộ Test đánh giá** | Cả 3 tệp `test_day_du.jsonl` (334 C0), `test_perturb_C1.jsonl` (668 C1) và `test_perturb_C2.jsonl` (668 C2) **không được nạp vào bộ nhớ ở bước này** — chỉ đọc ở Mục 6 sau khi đã đóng băng checkpoint và `calibration.json`. |
| **Chống rò rỉ `HUMAN_CAL_PATH`** | Kiểm tra SHA-256 chặn ngay lập tức nếu trỏ nhầm vào bất kỳ tệp nào trong 6 tệp `train`/`test`/`C1`/`C2` đã ghim (kể cả bản đổi tên, CRLF hoặc UTF-8 BOM). |

In [ ]:
#@title 2.1. Tải dữ liệu ghim SHA-256, Kiểm tra toàn vẹn & Chia tập StratifiedGroupKFold
# ==============================================================================
# 1. PHIÊN HTTP RETRY & BẢNG BĂM SHA-256 GHIM
# ==============================================================================
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

DATA_HTTP = requests.Session()
DATA_HTTP.mount("https://", HTTPAdapter(max_retries=Retry(total=3, connect=3, read=3,
    backoff_factor=1, status_forcelist=(429, 500, 502, 503, 504))))
DATA_URL = f"https://raw.githubusercontent.com/thaoanhlehoang/SafeViChi/{DATA_REV}/data/v2/"
DATA_SHA256 = {
    "so_sanh.jsonl": "c2c5046fcd2e3853e76ecc79c7ea70eaad79743380e03185bc32610ebc813102",
    "train.jsonl": "4a31dcee5b93392f1b5d6fba778d9121604aa59beaa8252b3026ce1112ca334f",
    "test.jsonl": "9ebdad75d0b1bc60df7d1cacd761dd74c5eb6e64d678692f4d2889204033c115",
    "test_day_du.jsonl": "7652e102a6e8b6b9ce9c99633333afe639642e65f41f2d6edc005a8e75085463",
    "test_perturb_C1.jsonl": "1ae3cadecac4df407273d23a87c84201f603311a4994acaaebfd3915736585a5",
    "test_perturb_C2.jsonl": "8b2a24234dda9e855c91d86ae03d21a16710be2dc2569d121d29d5b7f693823b",
}
TEENCODE_DICT_SHA256 = "c7f345574974668c2b319955c23df11b5d6ac3055e743cb1b4e40ec326ff1c14"

def _matches_blob_sha(raw_bytes, expected):
    clean = raw_bytes.removeprefix(b"\xef\xbb\xbf")
    return digest(raw_bytes) == expected or digest(clean) == expected or digest(clean.replace(b"\r\n", b"\n")) == expected

LOCKED_DATA_FILES = (
    "test.jsonl", "test_day_du.jsonl", "test_perturb_C1.jsonl", "test_perturb_C2.jsonl",
    "so_sanh.jsonl", "train.jsonl",
)

# ==============================================================================
# 2. KHÓA AN TOÀN HUMAN_CAL_PATH & HÀM TẢI TỆP DỮ LIỆU
# ==============================================================================
if HUMAN_CAL_PATH:
    _hcal_raw = Path(HUMAN_CAL_PATH).read_bytes()
    for _locked_name in LOCKED_DATA_FILES:
        if _matches_blob_sha(_hcal_raw, DATA_SHA256[_locked_name]):
            raise ValueError(f"KHÓA AN TOÀN: HUMAN_CAL_PATH trùng SHA-256 của {_locked_name} — từ chối để chống rò rỉ!")
    del _hcal_raw, _locked_name

def fetch_teencode_dict(expected=TEENCODE_DICT_SHA256):
    """Tải từ điển teencode v1 (643 mục) đã ghim SHA-256 phục vụ đánh giá Dòng #6 (chuẩn hóa đầu vào)."""
    local_candidates = (
        Path.cwd() / "src" / "shared" / "normalization" / "teencode_dict.json",
        Path.cwd().parent / "src" / "shared" / "normalization" / "teencode_dict.json",
        Path.cwd() / "src" / "normalization" / "teencode_dict.json",
        Path.cwd().parent / "src" / "normalization" / "teencode_dict.json",
    )
    for cand in local_candidates:
        if cand.exists() and _matches_blob_sha(cand.read_bytes(), expected):
            return json.loads(cand.read_text(encoding="utf-8-sig"))
    cache = Path.cwd() / ".safevichi_data" / DATA_REV
    cache.mkdir(parents=True, exist_ok=True)
    path = cache / "teencode_dict.json"
    if not path.exists() or not _matches_blob_sha(path.read_bytes(), expected):
        base_raw = f"https://raw.githubusercontent.com/thaoanhlehoang/SafeViChi/{DATA_REV}/"
        last_err = None
        for rel in ("src/normalization/teencode_dict.json", "src/shared/normalization/teencode_dict.json"):
            try:
                res = DATA_HTTP.get(base_raw + rel, timeout=(15, 60))
                if res.status_code == 200 and _matches_blob_sha(res.content, expected):
                    tmp_p = path.with_suffix(f".{os.getpid()}.tmp")
                    tmp_p.write_bytes(res.content)
                    os.replace(tmp_p, path)
                    break
                last_err = f"HTTP {res.status_code} ({rel})"
            except Exception as exc:
                last_err = str(exc)
        else:
            raise RuntimeError(f"Không tải được teencode_dict.json khớp SHA-256: {last_err}")
    return json.loads(path.read_text(encoding="utf-8-sig"))

def fetch_data(name, expected=None):
    expected = expected or DATA_SHA256[name]
    for candidate in (Path.cwd() / "data" / "v2" / name, Path.cwd().parent / "data" / "v2" / name):
        if candidate.exists() and _matches_blob_sha(candidate.read_bytes(), expected):
            return [json.loads(line) for line in candidate.read_text(encoding="utf-8-sig").splitlines() if line.strip()]
    cache = Path.cwd() / ".safevichi_data" / DATA_REV
    cache.mkdir(parents=True, exist_ok=True)
    path = cache / name
    if not path.exists() or not _matches_blob_sha(path.read_bytes(), expected):
        res = DATA_HTTP.get(DATA_URL + name, timeout=(15, 180))
        res.raise_for_status()
        assert _matches_blob_sha(res.content, expected), f"Checksum sai: {name}"
        tmp_p = path.with_suffix(f".{os.getpid()}.tmp")
        tmp_p.write_bytes(res.content)
        os.replace(tmp_p, path)
    return [json.loads(line) for line in path.read_text(encoding="utf-8-sig").splitlines() if line.strip()]


# ==============================================================================
# 3. XÁC THỰC TẬP HUẤN LUYỆN SẠCH (13.392 ĐOẠN C0)
# ==============================================================================
rows = fetch_data("so_sanh.jsonl")
train_rows = fetch_data("train.jsonl")
assert len(rows) == len(train_rows) == 13392
assert len({r["id"] for r in rows}) == len({r["nhom_trung"] for r in rows}) == len(rows)
for r, t in zip(rows, train_rows):
    assert t == {k: r[k] for k in ("id", "text", "nhan", "bang_chung")}
    assert r["nhan"] in LABELS and r["bien_dang"] == "C0"
    assert r["so_dong"] == len(r["text"].splitlines()) and 1 <= r["so_dong"] <= 8
    assert all(any(p in line for line in r["text"].splitlines()) for p in r["bang_chung"])
    assert (r.get("meta") or {}).get("kich_ban") not in {"LD7", "LD12", "DD6", "QR6"}, f"Rò rỉ kịch bản ngoài train: {r['id']}"
del train_rows


# ==============================================================================
# 4. PHÂN HOẠCH STRATIFIEDGROUPKFOLD & LƯU SPLIT_IDS.JSON
# ==============================================================================
def split_group(r):
    meta = r.get("meta") or {}
    if meta.get("loai") == "sinh":
        return "prompt/" + r["nguon_id"].rsplit(":", 1)[0].removesuffix("-SD")
    grp = r.get("nhom_trung")
    return "real/" + (str(grp) if grp else str(r["id"]))

def make_splits(all_rows, scheme=DATA_SCHEME, split_seed=42):
    groups = np.array([split_group(r) for r in all_rows])
    strata = [r["nhan"] + "/" + r["meta"]["loai"] for r in all_rows]
    fold = np.full(len(all_rows), -1)
    for i, (_, ix) in enumerate(StratifiedGroupKFold(10, shuffle=True, random_state=split_seed).split(groups, strata, groups)):
        fold[ix] = i
    if scheme == "dev_80_10_10":
        masks = {"fit": (fold >= 3) | (fold == 0), "dev": fold == 2, "cal": fold == 1}
    elif scheme == "final_90_10":
        masks = {"fit": fold != 1, "cal": fold == 1}
    elif scheme == "final_100_human_cal":
        masks = {"fit": np.ones(len(all_rows), dtype=bool)}
    elif scheme == "legacy_70_10_10_10":
        masks = {"fit": fold >= 3, "dev": fold == 2, "cal": fold == 1, "audit": fold == 0}
    else:
        raise ValueError(f"DATA_SCHEME không hợp lệ: {scheme}")
    out = {k: [all_rows[i] for i in np.flatnonzero(mask)] for k, mask in masks.items()}
    assert len(set.union(*(set(map(split_group, v)) for v in out.values()))) == sum(len(set(map(split_group, v))) for v in out.values())
    assert sum(len(v) for v in out.values()) == len(all_rows)
    return out

parts = make_splits(rows, scheme=DATA_SCHEME, split_seed=config["split_seed"])

if RUN_MODE == "smoke":
    for k, v in parts.items():
        per_label = 8 if k == "fit" else 2
        parts[k] = [r for label in LABELS for r in [x for x in v if x["nhan"] == label][:per_label]]
for v in parts.values():
    assert set(r["nhan"] for r in v) == set(LABELS)

write_json(RUN / "split_ids.json", {"scheme": DATA_SCHEME, **{k: [r["id"] for r in v] for k, v in parts.items()}})
print(f"DATA_SCHEME = {DATA_SCHEME} | Phân bổ số mẫu mỗi split:")
print(pd.DataFrame({k: Counter(r["nhan"] for r in v) for k, v in parts.items()}))
print(pd.crosstab([r["nhan"] for r in rows], [r["meta"]["loai"] for r in rows]))

---
<a id="sec-3"></a>
## 3. Tiền xử lý hội thoại, Tăng cường C1 phi điểm mù & DataLoader

Ô này chuẩn bị toàn bộ dữ liệu đầu vào mức subword–từ–byte và tự kiểm định căn chỉnh offset trước khi khởi tạo mô hình:

| Thành phần | Thiết kế kỹ thuật |
| :--- | :--- |
| **Mã hóa chung toàn hội thoại** | Đóng gói hội thoại 1–8 lượt thành `[BOS] turn_0 [SEP] turn_1 [SEP] ... [SEP]` (tối đa `MAX_TOKENS`: `512` cho `ViSoBERT`/`mmBERT`, `1.024` cho `BamiBERT`, tự co đều giữa các lượt nếu vượt ngưỡng) kèm ánh xạ chính xác `ký tự → từ → subword → byte UTF-8`. |
| **Nhãn BIO-CRF mức từ** | Gán `O=0, B-EV=1, I-EV=2, masked=-100`; bảo đảm không bao giờ sinh `I-EV` ở đầu lượt mới (`word_turn_start=True`); vị trí lặp mơ hồ (`unknown`) hoặc mẫu nguy hại thiếu bằng chứng gán `-100` để CRF biên hóa chính xác. |
| **Trọng số phân cực LEG (`δ_t`)** | Tính duy nhất trên `parts["fit"]` theo `arXiv:2602.15853`, hoàn toàn không nhìn thấy `dev / cal / audit`. |
| **Tăng cường phi điểm mù & Dual-View** | Dùng đúng bộ quy tắc C1 đơn nghĩa khớp Nhánh 1 (tuyệt đối không dùng bất kỳ kỹ thuật nào của `blindspot.py` hay C1 held-out) và ghép cặp `(bản sạch, bản bỏ dấu/C1)` trong cùng microbatch để tối ưu hàm nhất quán phân kỳ JS. |

In [ ]:
#@title 3.1. Token hóa, Tăng cường C1 phi điểm mù, LEG, Collate & Tự kiểm định
# ==============================================================================
# 1. KHỞI TẠO FAST TOKENIZER & TOKEN ĐẶC BIỆT
# ==============================================================================
tok = AutoTokenizer.from_pretrained(BACKBONE_ID, revision=BACKBONE_REV, use_fast=True)
assert getattr(tok, "is_fast", False), f"Cần fast tokenizer có return_offsets_mapping cho {BACKBONE_ID}"

BOS_ID = tok.cls_token_id if tok.cls_token_id is not None else tok.bos_token_id
SEP_ID = tok.sep_token_id if tok.sep_token_id is not None else tok.eos_token_id
PAD_ID = tok.pad_token_id if tok.pad_token_id is not None else 0
MASK_DROP_ID = tok.unk_token_id if tok.unk_token_id is not None else (tok.mask_token_id if tok.mask_token_id is not None else PAD_ID)
assert BOS_ID is not None and SEP_ID is not None and PAD_ID is not None
print(f"Tokenizer: {tok.__class__.__name__} | vocab={len(tok)} | BOS={BOS_ID} SEP={SEP_ID} PAD={PAD_ID} DROP={MASK_DROP_ID}")


# ==============================================================================
# 2. CHUẨN HÓA BỎ DẤU & TĂNG CƯỜNG BỀ MẶT C1 PHI ĐIỂM MÙ
# ==============================================================================
def strip_accents(text):
    return "".join(c for c in unicodedata.normalize("NFD", text) if unicodedata.category(c) != "Mn").translate(
        str.maketrans({"đ": "d", "Đ": "D"}))

# Tập quy tắc C1 đơn nghĩa giống hệt Nhánh 1 (giữ hoàn toàn blindspot và heldout ngoài train)
C1_TEENCODE_MAP = {
    "không": ("ko", "k", "hông"),
    "được": ("đc", "dc"),
    "biết": ("bt", "bít"),
    "với": ("vs",),
    "người": ("ng",),
    "mình": ("mk",),
    "gì": ("j",),
    "vậy": ("zậy",),
    "rồi": ("rùi",),
    "cũng": ("cx",),
    "phải": ("fải",),
    "nhiều": ("nhìu",),
    "yêu": ("iu",),
    "thôi": ("thui",),
    "hiểu": ("hỉu",),
    "quá": ("qá", "wá"),
    "chưa": ("chưaaa",),
}
C1_LOOKALIKE = {"a": ("4", "@"), "e": ("3",), "i": ("1",), "o": ("0",)}
C1_SEPS = (".", "-", "_")
_ALPHA_CORE_RE = re.compile(r"^[a-zA-ZÀ-ỹ]+$")

def _split_affix(token):
    i, j = 0, len(token)
    while i < j and not (token[i].isalnum() or unicodedata.combining(token[i])):
        i += 1
    while j > i and not (token[j - 1].isalnum() or unicodedata.combining(token[j - 1])):
        j -= 1
    return token[:i], token[i:j], token[j:]

def _map_pos(pos, segs, is_end):
    for sa, sb, oa, ob, same in segs:
        if (pos < sb) if not is_end else (pos <= sb):
            if pos <= sa:
                return oa
            if same:
                return oa + (pos - sa)
            return ob if is_end else oa
    return segs[-1][3] if segs else 0

def transform_line_with_spans(line, spans, core_fn):
    segs, parts_out, cur = [], [], 0
    for m in re.finditer(r"\S+|\s+", line):
        a, b = m.span()
        tok_str = m.group(0)
        if tok_str.isspace():
            parts_out.append(tok_str)
            segs.append((a, b, cur, cur + len(tok_str), True))
            cur += len(tok_str)
            continue
        lead, core, trail = _split_affix(tok_str)
        pos = a
        if lead:
            parts_out.append(lead)
            segs.append((pos, pos + len(lead), cur, cur + len(lead), True))
            cur += len(lead)
            pos += len(lead)
        if core:
            new_core = core_fn(core) or core
            parts_out.append(new_core)
            segs.append((pos, pos + len(core), cur, cur + len(new_core), new_core == core))
            cur += len(new_core)
            pos += len(core)
        if trail:
            parts_out.append(trail)
            segs.append((pos, pos + len(trail), cur, cur + len(trail), True))
            cur += len(trail)
    out_line = "".join(parts_out)
    new_spans = []
    for sa, sb in spans:
        na = _map_pos(sa, segs, False)
        nb = _map_pos(sb, segs, True)
        if na < nb:
            new_spans.append((na, nb))
    return out_line, new_spans

def _apply_line_transforms(row, line_fn_at):
    lines = row["text"].split("\n")
    unique_phrases = list(dict.fromkeys(row["bang_chung"]))
    new_lines, new_phrases, seen = [], [], set()
    for idx_line, line in enumerate(lines):
        line_spans = []
        for ph in unique_phrases:
            for m in re.finditer("(?=" + re.escape(ph) + ")", line):
                line_spans.append((m.start(), m.start() + len(ph)))
        out_line, mapped_spans = transform_line_with_spans(line, line_spans, line_fn_at(idx_line, line))
        new_lines.append(out_line)
        for na, nb in mapped_spans:
            sub = out_line[na:nb]
            if sub and sub not in seen:
                seen.add(sub)
                new_phrases.append(sub)
    return dict(row, text="\n".join(new_lines), bang_chung=new_phrases)

def augment_accents(row, rng):
    lines = row["text"].split("\n")
    accented_idx = [i for i, l in enumerate(lines) if strip_accents(l) != l]
    if not accented_idx:
        return row
    if len(lines) > 1 and len(accented_idx) > 1 and rng.random() < 0.40:
        k = rng.randint(1, len(accented_idx) - 1)
        chosen = set(rng.sample(accented_idx, k))
    else:
        chosen = set(accented_idx)
    return _apply_line_transforms(row, lambda idx, _: (strip_accents if idx in chosen else (lambda c: c)))

def _c1_lookalike_word(core, rng):
    if len(core) < 3 or not _ALPHA_CORE_RE.match(core):
        return core
    idxs = [i for i in range(1, len(core)) if core[i].lower() in C1_LOOKALIKE]
    if not idxs:
        return core
    max_k = max(1, (len(core) - 1) // 2)  # bảo đảm số chữ cái > số ký tự đổi được (khớp normalizer.py)
    picks = set(rng.sample(idxs, min(len(idxs), rng.choice((1, 2)), max_k)))
    return "".join(rng.choice(C1_LOOKALIKE[ch.lower()]) if i in picks else ch for i, ch in enumerate(core))

def _c1_sep_word(core, rng):
    if len(core) < 3 or not _ALPHA_CORE_RE.match(core):
        return core
    cuts = sorted(rng.sample(range(1, len(core)), min(len(core) - 1, rng.choice((1, 2)))))
    sep = rng.choice(C1_SEPS)
    out, prev = [], 0
    for c in cuts:
        out.append(core[prev:c])
        out.append(sep)
        prev = c
    out.append(core[prev:])
    return "".join(out)

def _c1_elongate_word(core, rng):
    if len(core) < 2 or not _ALPHA_CORE_RE.match(core):
        return core
    vowels = [i for i, c in enumerate(core) if c.lower() in "aeiouyăâêôơưáàảãạắằẳẵặấầẩẫậéèẻẽẹếềểễệíìỉĩịóòỏõọốồổỗộớờởỡợúùủũụứừửữựýỳỷỹỵ"]
    pos = rng.choice(vowels) if vowels else len(core) - 1
    ch = core[pos]
    extra = rng.choice((2, 3))
    return core[:pos + 1] + (ch * extra) + core[pos + 1:]

def augment_surface(row, rng):
    do_lower = rng.random() < 0.50
    do_teencode = rng.random() < 0.75
    do_strip = rng.random() < 0.40
    obfuscate_mode = rng.choice(("teencode_strip", "lookalike", "sep", "elongate"))
    def make_fn(idx, line):
        n_w = max(1, len(line.split()))
        budget = [min(6, max(2, round(n_w * rng.uniform(0.25, 0.45)))) if obfuscate_mode != "teencode_strip" else 0]
        def fn(core):
            w = core
            if do_teencode:
                alts = C1_TEENCODE_MAP.get(w.lower())
                if alts and rng.random() < 0.45:
                    rep = rng.choice(alts)
                    w = rep.capitalize() if (w[0].isupper() and not do_lower) else rep
            if do_strip:
                w = strip_accents(w)
            if do_lower:
                w = w.lower()
            if budget[0] > 0 and len(w) >= 3 and _ALPHA_CORE_RE.match(w) and rng.random() < 0.55:
                if obfuscate_mode == "lookalike":
                    cand = _c1_lookalike_word(w, rng)
                elif obfuscate_mode == "sep":
                    cand = _c1_sep_word(w, rng)
                else:
                    cand = _c1_elongate_word(w, rng)
                if cand != w:
                    w = cand
                    budget[0] -= 1
            return w
        return fn
    return _apply_line_transforms(row, make_fn)


# ==============================================================================
# 3. KHỚP VỊ TRÍ BẰNG CHỨNG & TRỌNG SỐ PHÂN CỰC TỪ LEG (CHỈ TRÊN FIT)
# ==============================================================================
def annotations(row):
    text = row["text"]
    yes, unknown = [], []
    for phrase in set(row["bang_chung"]):
        positions = [m.start() for m in re.finditer("(?=" + re.escape(phrase) + ")", text)]
        target = yes if len(positions) == 1 else unknown
        target.extend((s, s + len(phrase)) for s in positions)
    return yes, unknown

def overlaps(interval, spans):
    a, b = interval
    return any(a < end and start < b for start, end in spans)

def _norm_polar_word(w):
    return strip_accents(w.lower().strip(".,!?;:\"'()[]{}…"))

def compute_leg_polarization(fit_rows):
    """Điểm phân cực từ vựng LEG (arXiv:2602.15853) tính CHỈ trên fit (0 rò rỉ)."""
    c_safe, c_harm = Counter(), Counter()
    for r in fit_rows:
        is_harm = r["nhan"] != "AN_TOAN"
        seen = set()
        for m in re.finditer(r"\S+", r["text"]):
            k = _norm_polar_word(m.group(0))
            if k and k not in seen:
                seen.add(k)
                (c_harm if is_harm else c_safe)[k] += 1
    polar = {}
    for w in set(c_safe) | set(c_harm):
        s, h = c_safe[w], c_harm[w]
        tot = s + h
        if tot >= 5:
            # delta ∈ [0, 1); chỉ giữ hướng phân cực (h > s -> +delta cho harm, s > h -> -delta cho safe)
            polar[w] = (h - s) / float(tot)
    return polar

LEG_POLAR = compute_leg_polarization(parts["fit"]) if LEG_POLAR_WEIGHT > 0 else {}


# ==============================================================================
# 4. ĐÓNG GÓI MẪU HỘI THOẠI (PREPARE) & NHÃN BIO MỨC TỪ
# ==============================================================================
def _allocate_turn_budgets(turn_lens, budget):
    """Co đều các lượt quá dài để tổng subword ≤ budget mà mọi lượt vẫn được giữ."""
    n_t = len(turn_lens)
    if sum(turn_lens) <= budget:
        return list(turn_lens)
    # Ưu tiên giữ phần đuôi lượt dài (thường chứa tín hiệu chốt); phân bổ vòng tròn tối thiểu 12 token/lượt.
    caps = [0] * n_t
    rem = budget
    active = list(range(n_t))
    while active and rem > 0:
        share = max(1, rem // len(active))
        next_active = []
        for i in active:
            need = turn_lens[i] - caps[i]
            take = min(need, share, rem)
            caps[i] += take
            rem -= take
            if caps[i] < turn_lens[i]:
                next_active.append(i)
        if not next_active or len(next_active) == len(active) and share == 1 and rem == 0:
            break
        active = next_active
    return caps

def prepare(row):
    text = row["text"]
    yes, unknown = annotations(row)
    skip = row["nhan"] != "AN_TOAN" and not yes and not unknown
    lines = text.split("\n")
    n_turns = len(lines)
    assert 1 <= n_turns <= 8, f"Số lượt ngoài 1..8: {row['id']}"

    # 1) Tách danh sách từ ngắt khoảng trắng và gán nhãn BIO mức từ
    words_meta = []  # (char_start, char_end, turn_id, recency, is_turn_start, bio_label, bin_gold, byte_ids, polar_w)
    cursor = 0
    for t_idx, line in enumerate(lines):
        recency = n_turns - 1 - t_idx
        prev_span_set = set()
        first_in_turn = True
        for m in re.finditer(r"\S+", line):
            a, b = cursor + m.start(), cursor + m.end()
            loc = (a, b)
            raw_w = m.group(0)
            b_arr = [x + 1 for x in raw_w.encode("utf-8")[:MAX_WORD_BYTES]]
            if not b_arr:
                b_arr = [1]
            # Xác định nhãn BIO & nhãn nhị phân chuẩn chấm evidence_stats
            if skip or overlaps(loc, unknown):
                bio = -100
                bin_g = -1
                cur_spans = set()
            else:
                cur_spans = {idx_s for idx_s, sp in enumerate(yes) if a < sp[1] and sp[0] < b}
                if not cur_spans:
                    bio = 0
                    bin_g = 0
                else:
                    bin_g = 1
                    # I-EV chỉ khi cùng lượt (không phải đầu lượt) VÀ chia sẻ ít nhất 1 cụm bằng chứng với từ liền trước
                    bio = 2 if (not first_in_turn and bool(cur_spans & prev_span_set)) else 1
            # Trọng số phân cực LEG (chỉ tăng nhẹ trọng số khi từ phân cực đồng hướng với nhãn câu)
            pw = 1.0
            if LEG_POLAR and bio in (0, 1, 2):
                p_val = LEG_POLAR.get(_norm_polar_word(raw_w), 0.0)
                if bio > 0 and p_val > 0:
                    pw = 1.0 + LEG_POLAR_WEIGHT * p_val
                elif bio == 0 and p_val < 0:
                    pw = 1.0 + LEG_POLAR_WEIGHT * (-p_val)
            words_meta.append((a, b, t_idx, recency, first_in_turn, bio, bin_g, b_arr, pw))
            prev_span_set = cur_spans
            first_in_turn = False
        cursor += len(line) + 1

    if not words_meta:
        raise ValueError(f"Văn bản không có từ: {row['id']}")

    # 2) Token hóa từng lượt rồi ghép thành 1 chuỗi [BOS] t0 [SEP] t1 [SEP] ... [SEP]
    turn_enc = []
    cursor = 0
    for line in lines:
        enc = tok(line, add_special_tokens=False, return_offsets_mapping=True, truncation=False)
        t_ids = enc["input_ids"]
        t_offs = []
        for sa, sb in enc["offset_mapping"]:
            while sa < sb and sb < len(line) and unicodedata.combining(line[sb]):
                sb += 1
            t_offs.append((cursor + sa, cursor + sb))
        turn_enc.append((t_ids, t_offs))
        cursor += len(line) + 1

    # Ngân sách subword thực (trừ 1 BOS + n_turns SEP)
    budget = max(16, MAX_TOKENS - (1 + n_turns))
    caps = _allocate_turn_budgets([len(x[0]) for x in turn_enc], budget)
    seq_ids = [BOS_ID]
    seq_offs = [(0, 0)]
    for (t_ids, t_offs), cap in zip(turn_enc, caps):
        if len(t_ids) > cap:
            # Giữ đầu + đuôi của lượt dài (ưu tiên phần đuôi chứa ý đồ)
            head = cap // 3
            tail = cap - head
            kept_ids = t_ids[:head] + t_ids[-tail:] if tail > 0 else t_ids[:head]
            kept_offs = t_offs[:head] + t_offs[-tail:] if tail > 0 else t_offs[:head]
        else:
            kept_ids, kept_offs = t_ids, t_offs
        seq_ids.extend(kept_ids)
        seq_offs.extend(kept_offs)
        seq_ids.append(SEP_ID)
        seq_offs.append((0, 0))
    assert len(seq_ids) <= MAX_TOKENS

    # 3) Ánh xạ từng từ sang chỉ số subword trong seq_ids
    word_sub_idxs = []
    pos_tok = 1
    n_seq = len(seq_offs)
    for a, b, *_ in words_meta:
        # Di chuyển con trỏ gần tới vị trí từ
        while pos_tok < n_seq and (seq_offs[pos_tok][0] == seq_offs[pos_tok][1] or seq_offs[pos_tok][1] <= a):
            pos_tok += 1
        idxs = []
        j = pos_tok
        while j < n_seq:
            sa, sb = seq_offs[j]
            if sa == sb:
                j += 1
                continue
            if sa >= b:
                break
            if sb > a:
                idxs.append(j)
            j += 1
        word_sub_idxs.append(idxs[:MAX_SUBWORDS_PER_WORD])

    return dict(
        row=row,
        label=LABELS.index(row["nhan"]),
        ids=seq_ids,
        n_tokens=len(seq_ids),
        n_words=len(words_meta),
        words_meta=words_meta,
        word_sub_idxs=word_sub_idxs,
    )


# ==============================================================================
# 5. COLLATE, BUCKETBATCHSAMPLER & CHIA MICROBATCH GIỮ NGUYÊN CẶP
# ==============================================================================
def collate(items):
    """Nhận danh sách item: mỗi item là 1 dict prepare(row) hoặc tuple (clean, pert) cho Dual-View."""
    examples = []
    pairs = []
    for it in items:
        if isinstance(it, tuple):
            i0 = len(examples)
            examples.append(it[0])
            examples.append(it[1])
            pairs.append((i0, i0 + 1))
        else:
            examples.append(it)

    B = len(examples)
    max_L = max(e["n_tokens"] for e in examples)
    max_W = max(e["n_words"] for e in examples)
    max_S = max(1, max((len(s) for e in examples for s in e["word_sub_idxs"]), default=1))

    ids = torch.full((B, max_L), PAD_ID, dtype=torch.long)
    attn_mask = torch.zeros((B, max_L), dtype=torch.bool)
    word_sub_idx = torch.zeros((B, max_W, max_S), dtype=torch.long)
    word_sub_mask = torch.zeros((B, max_W, max_S), dtype=torch.bool)
    word_mask = torch.zeros((B, max_W), dtype=torch.bool)
    word_turn = torch.zeros((B, max_W), dtype=torch.long)
    word_recency = torch.zeros((B, max_W), dtype=torch.long)
    word_turn_start = torch.zeros((B, max_W), dtype=torch.bool)
    word_bytes = torch.zeros((B, max_W, MAX_WORD_BYTES), dtype=torch.long)
    bio_targets = torch.full((B, max_W), -100, dtype=torch.long)
    polar_weights = torch.ones((B, max_W), dtype=torch.float32)

    for i, e in enumerate(examples):
        L = e["n_tokens"]
        ids[i, :L] = torch.tensor(e["ids"], dtype=torch.long)
        attn_mask[i, :L] = True
        for w_i, (meta, sub_idxs) in enumerate(zip(e["words_meta"], e["word_sub_idxs"])):
            _, _, t_idx, recency, is_ts, bio, _, b_arr, pw = meta
            word_mask[i, w_i] = True
            word_turn[i, w_i] = t_idx
            word_recency[i, w_i] = recency
            word_turn_start[i, w_i] = is_ts
            bio_targets[i, w_i] = bio
            polar_weights[i, w_i] = pw
            word_bytes[i, w_i, :len(b_arr)] = torch.tensor(b_arr, dtype=torch.long)
            if sub_idxs:
                word_sub_idx[i, w_i, :len(sub_idxs)] = torch.tensor(sub_idxs, dtype=torch.long)
                word_sub_mask[i, w_i, :len(sub_idxs)] = True

    return dict(
        ids=ids,
        attn_mask=attn_mask,
        word_sub_idx=word_sub_idx,
        word_sub_mask=word_sub_mask,
        word_mask=word_mask,
        word_turn=word_turn,
        word_recency=word_recency,
        word_turn_start=word_turn_start,
        word_bytes=word_bytes,
        bio_targets=bio_targets,
        polar_weights=polar_weights,
        labels=torch.tensor([e["label"] for e in examples], dtype=torch.long),
        examples=examples,
        items=items,
        pairs=pairs,
    )

def _item_len(it):
    if isinstance(it, tuple):
        return max(it[0]["n_tokens"], it[1]["n_tokens"])
    return it["n_tokens"]

class BucketBatchSampler:
    def __init__(self, data, batch_size, seed=SEED, epoch=0, mega_mult=32):
        self.data = data
        self.batch_size = batch_size
        self.seed = seed
        self.epoch = epoch
        self.mega_mult = mega_mult
    def __iter__(self):
        rng = random.Random(self.seed + self.epoch)
        idx = list(range(len(self.data)))
        rng.shuffle(idx)
        mega = self.batch_size * self.mega_mult
        batches_idx = []
        for s in range(0, len(idx), mega):
            pool = idx[s:s + mega]
            pool.sort(key=lambda i: _item_len(self.data[i]))
            for j in range(0, len(pool), self.batch_size):
                batches_idx.append(pool[j:j + self.batch_size])
        rng.shuffle(batches_idx)
        return iter(batches_idx)
    def __len__(self):
        return math.ceil(len(self.data) / self.batch_size)

def split_microbatch(batch, max_tokens=MAX_MICROBATCH_TOKENS):
    items = batch["items"]
    if batch["ids"].numel() <= max_tokens or len(items) <= 1:
        return [batch]
    mid = len(items) // 2
    return (
        split_microbatch(collate(items[:mid]), max_tokens)
        + split_microbatch(collate(items[mid:]), max_tokens)
    )


# ==============================================================================
# 6. TỰ KIỂM ĐỊNH OFFSET / BIO / AUGMENTATION & MÃ HÓA DỮ LIỆU
# ==============================================================================
# Tự kiểm thử căn chỉnh offset, BIO không vượt dòng và tăng cường phi điểm mù
for form in ("NFC", "NFD", "bo_dau"):
    needle = unicodedata.normalize("NFD" if form == "NFD" else "NFC", "dữ dằn")
    if form == "bo_dau":
        needle = strip_accents(needle)
    sample = "Chào bạn!\nĐừng nói " + needle + " quá nhé"
    e = prepare(dict(id="bio-check", text=sample, nhan="XUC_PHAM", bang_chung=[needle]))
    bios = [m[5] for m in e["words_meta"]]
    ts = [m[4] for m in e["words_meta"]]
    assert bios.count(1) == 1 and bios.count(2) == 1, f"Sai BIO: {bios}"
    assert all(not (is_t and b == 2) for is_t, b in zip(ts, bios))

aug_probe = dict(
    id="aug-check",
    text='Chào bạn nhé!\nKhông được nói "đồ ngu" với người ta như vậy!',
    nhan="XUC_PHAM",
    bang_chung=["đồ ngu"],
    so_dong=2,
)
aug_rng = random.Random(SEED)
for _ in range(12):
    for aug_row in (augment_accents(aug_probe, aug_rng), augment_surface(aug_probe, aug_rng)):
        assert aug_row["bang_chung"] and all(any(p in l for l in aug_row["text"].splitlines()) for p in aug_row["bang_chung"])
        assert not re.search(r"[\u200b\u200c\u200d\ufeffаеосрухкѕіАЕОСРУХКВНМ]", aug_row["text"])
        p_clean, p_aug = prepare(aug_probe), prepare(aug_row)
        assert p_clean["n_words"] == p_aug["n_words"]
        assert any(m[5] == 1 for m in p_aug["words_meta"])

encoded = {k: [prepare(r) for r in v] for k, v in parts.items()}

def batches(data, shuffle=False, epoch=0, batch_size=BATCH):
    if shuffle:
        return DataLoader(
            data,
            batch_sampler=BucketBatchSampler(data, batch_size, seed=SEED, epoch=epoch),
            collate_fn=collate,
            num_workers=0,
        )
    return DataLoader(data, batch_size=batch_size, shuffle=False, collate_fn=collate, num_workers=0)

def to_device(batch, device):
    return {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in batch.items()}

print("Thống kê token/từ mỗi tập:", {
    k: dict(n=len(v), max_tok=max(e["n_tokens"] for e in v), p95_tok=int(np.percentile([e["n_tokens"] for e in v], 95)))
    for k, v in encoded.items()
})

---
<a id="sec-4"></a>
## 4. Kiến trúc `SafeVi-DualORC-CRF` & Kiểm định toán học

Toàn bộ các tầng head (Word Aggregation, UTF-8 Byte-CNN, HAT Turn-to-Word Cross-Attention, Linear-Chain BIO-CRF và Ordered Reasoning Chain) chạy **hoàn toàn ở FP32 (`autocast(enabled=False)`)** để bảo toàn độ chính xác số học của Forward–Backward / LogSumExp trên Tesla T4, trong khi backbone Transformer tận dụng AMP FP16/BF16.

```text
  Hội thoại (1–8 lượt)
         │
         ├──▶ [Backbone Transformer (AMP FP16/BF16)] ──▶ Gộp Subword → Từ
         │                                                      │
         └──▶ [Word-Level UTF-8 Byte-CNN (k=3,5,7)] ────────────┼──▶ Cổng trộn (FP32)
                                                                ▼
                                           [HAT: Turn Self-Attn + Turn-to-Word Cross-Attn]
                                                   │                              │
                                                   ▼                              ▼
                                  [ORC 4 bước + Multi-Prototype]      [Turn-Aware BIO-CRF]
                                  → 5 nhãn + Harm + Severity          → Cụm bằng chứng verbatim
```

### Tự kiểm định toán học trước huấn luyện (chạy tự động cuối ô 4.1)
1. **Chính xác toán học `TurnAwareBIOCRF`:** So khớp `log_Z`, phân phối biên Forward–Backward và giải mã Viterbi với duyệt vét cạn toàn bộ $3^W$ trạng thái; xác nhận mẫu toàn `-100` cho `NLL = 0,0` và ràng buộc cấm `I-EV` vượt ranh giới `\n`.
2. **Bất biến tuyệt đối với Padding:** So khớp mọi đầu ra (`logits`, `emissions`, `ev_prob`, `harm_logit`, `severity`) trên mẫu 1 lượt và đa lượt khi đặt đơn lẻ so với khi gộp batch có padding.

In [ ]:
#@title 4.1. Định nghĩa kiến trúc SafeViDualORCCRF & Kiểm định toán học (CRF / Padding)
# ==============================================================================
# 1. TẢI BACKBONE ENCODER
# ==============================================================================
def load_backbone():
    cfg = AutoConfig.from_pretrained(BACKBONE_ID, revision=BACKBONE_REV)
    # Tắt torch.compile nội bộ trên ModernBERT để tương thích ổn định với Tesla T4 / shape động
    if hasattr(cfg, "reference_compile"):
        cfg.reference_compile = False
    if hasattr(cfg, "use_cache"):
        cfg.use_cache = False
    kwargs = dict(revision=BACKBONE_REV, config=cfg)
    if getattr(cfg, "model_type", "") in {"xlm-roberta", "roberta", "bert"}:
        kwargs["add_pooling_layer"] = False
    if getattr(cfg, "model_type", "") == "modernbert":
        kwargs["attn_implementation"] = "sdpa"
    model = AutoModel.from_pretrained(BACKBONE_ID, **kwargs)
    return model



# ==============================================================================
# 2. LỚP TURN-AWARE LINEAR-CHAIN BIO-CRF (FP32)
# ==============================================================================
class TurnAwareBIOCRF(nn.Module):
    """Linear-Chain CRF 3 trạng thái [0=O, 1=B-EV, 2=I-EV] có:
    - Ràng buộc cứng cấm I-EV ở đầu chuỗi và ở đầu mỗi lượt thoại mới (\n).
    - Hỗ trợ Partial-Label NLL chính xác cho các vị trí mơ hồ (-100) qua Forward trên tập trạng thái hợp lệ.
    - Thuật toán Forward-Backward trả về xác suất biên chính xác P(B-EV) + P(I-EV) làm tín hiệu cho nhánh ORC.
    - Giải mã Viterbi hỗ trợ độ lệch nền delta_O để hiệu chuẩn Precision/Recall bằng chứng độc lập.
    """
    NEG_INF = -1e4

    def __init__(self):
        super().__init__()
        self.start_trans = nn.Parameter(torch.tensor([0.2, 0.0, -5.0]))
        self.end_trans = nn.Parameter(torch.zeros(3))
        # Khởi tạo thiên nhẹ O->O và B->I để khớp phân phối cụm từ tiếng Việt
        self.trans = nn.Parameter(torch.tensor([
            [ 0.5,  0.0, -5.0],  # O -> O, B, I (O->I luôn bị chặn cứng)
            [-0.2, -0.3,  0.6],  # B -> O, B, I
            [ 0.0, -0.4,  0.6],  # I -> O, B, I
        ]))

    def _effective_trans(self):
        t = self.trans.clone()
        # Cấm tuyệt đối chuyển O -> I-EV
        t[0, 2] = self.NEG_INF
        return t

    def _effective_start(self):
        s = self.start_trans.clone()
        s[2] = self.NEG_INF
        return s

    def _step_trans(self, trans, is_turn_start_t):
        """(B, 3, 3): nếu đầu lượt mới (\n), chặn cứng mọi chuyển tiếp vào I-EV (trạng thái 2)."""
        B = is_turn_start_t.size(0)
        step = trans.unsqueeze(0).expand(B, 3, 3).clone()
        if is_turn_start_t.any():
            step[is_turn_start_t, :, 2] = self.NEG_INF
        return step

    def forward_alpha(self, emissions, mask, turn_start, allowed=None):
        """Trả về log_Z (B,) và alpha_all (B, W, 3) bằng thuật toán Forward."""
        B, W, _ = emissions.shape
        trans = self._effective_trans()
        start = self._effective_start()
        emit = emissions
        if allowed is not None:
            emit = emit.masked_fill(~allowed, self.NEG_INF)

        alpha = start.unsqueeze(0) + emit[:, 0]  # (B, 3)
        alphas = [alpha]
        for t in range(1, W):
            m_t = mask[:, t].unsqueeze(-1)  # (B, 1)
            step_tr = self._step_trans(trans, turn_start[:, t])  # (B, 3, 3)
            # score[b, i, j] = alpha[b, i] + step_tr[b, i, j] + emit[b, t, j]
            nxt = torch.logsumexp(alpha.unsqueeze(2) + step_tr, dim=1) + emit[:, t]
            alpha = torch.where(m_t, nxt, alpha)
            alphas.append(alpha)

        log_z = torch.logsumexp(alpha + self.end_trans.unsqueeze(0), dim=-1)
        return log_z, torch.stack(alphas, dim=1)

    def marginals(self, emissions, mask, turn_start):
        """Forward-Backward chính xác -> xác suất biên (B, W, 3) và log_Z (B,)."""
        B, W, _ = emissions.shape
        trans = self._effective_trans()
        log_z, alphas = self.forward_alpha(emissions, mask, turn_start, allowed=None)

        beta = self.end_trans.unsqueeze(0).expand(B, 3).clone()
        betas = [None] * W
        betas[W - 1] = beta
        for t in range(W - 2, -1, -1):
            m_next = mask[:, t + 1].unsqueeze(-1)
            step_tr = self._step_trans(trans, turn_start[:, t + 1])  # (B, 3, 3)
            # prev[b, i] = logsumexp_j (step_tr[b, i, j] + emit[b, t+1, j] + beta[b, j])
            nxt_beta = torch.logsumexp(step_tr + (emissions[:, t + 1] + beta).unsqueeze(1), dim=2)
            beta = torch.where(m_next, nxt_beta, beta)
            betas[t] = beta

        betas = torch.stack(betas, dim=1)
        log_marg = alphas + betas - log_z[:, None, None]
        probs = log_marg.exp()
        probs = probs / probs.sum(dim=-1, keepdim=True).clamp_min(1e-8)
        probs = probs * mask.unsqueeze(-1)
        return probs, log_z

    def nll(self, emissions, targets, mask, turn_start, polar_weights=None):
        """Partial-Label CRF NLL chuẩn hóa theo số từ có nhãn hợp lệ trong mỗi mẫu.
        Khi truyền polar_weights (từ LEG trên fit), áp trọng số phân cực trực tiếp ở MỨC TỪ
        qua per-word emission CE với phần vượt (polar_weights - 1.0) thay vì làm phẳng cả câu."""
        B, W, _ = emissions.shape
        known = mask & (targets >= 0)
        # Bảng allowed (B, W, 3): từ biết nhãn chỉ mở đúng 1 trạng thái; từ mơ hồ (-100) mở cả 3.
        allowed = torch.ones((B, W, 3), dtype=torch.bool, device=emissions.device)
        if known.any():
            safe_t = targets.clamp(0, 2)
            one_hot = F.one_hot(safe_t, num_classes=3).bool()
            allowed = torch.where(known.unsqueeze(-1), one_hot, allowed)
        # An toàn tuyệt đối: không bao giờ khóa hết trạng thái trên đầu lượt (vì gold I-EV không bao giờ ở đầu lượt)
        log_z_all, _ = self.forward_alpha(emissions, mask, turn_start, allowed=None)
        log_z_gold, _ = self.forward_alpha(emissions, mask, turn_start, allowed=allowed)
        seq_nll = (log_z_all - log_z_gold).clamp_min(0.0)
        valid_words = known.sum(dim=1).float()
        has_known = valid_words > 0
        if not has_known.any():
            return emissions.sum() * 0.0
        per_sample = seq_nll / valid_words.clamp_min(1.0)
        if polar_weights is not None:
            w_excess = (polar_weights - 1.0).clamp_min(0.0) * known.float()
            if w_excess.any():
                word_ce = F.cross_entropy(
                    emissions.reshape(-1, 3),
                    targets.masked_fill(~known, -100).reshape(-1),
                    ignore_index=-100,
                    reduction="none",
                ).view(B, W)
                per_sample = per_sample + (word_ce * w_excess).sum(dim=1) / valid_words.clamp_min(1.0)
        return per_sample[has_known].mean()

    @torch.no_grad()
    def decode(self, emissions, mask, turn_start, o_bias=0.0):
        """Giải mã Viterbi với độ lệch o_bias cộng vào nhãn O (điều chỉnh Precision/Recall)."""
        B, W, _ = emissions.shape
        emit = emissions.clone()
        if o_bias != 0.0:
            emit[:, :, 0] = emit[:, :, 0] + float(o_bias)
        trans = self._effective_trans()
        start = self._effective_start()

        score = start.unsqueeze(0) + emit[:, 0]
        backptrs = []
        for t in range(1, W):
            m_t = mask[:, t].unsqueeze(-1)
            step_tr = self._step_trans(trans, turn_start[:, t])
            cand = score.unsqueeze(2) + step_tr  # (B, 3, 3)
            best_prev_score, best_prev_state = cand.max(dim=1)
            nxt = best_prev_score + emit[:, t]
            score = torch.where(m_t, nxt, score)
            backptrs.append(best_prev_state)

        score = score + self.end_trans.unsqueeze(0)
        best_last = score.argmax(dim=-1)  # (B,)
        lengths = mask.sum(dim=1).tolist()
        bp_cpu = [bp.cpu().tolist() for bp in backptrs]
        last_cpu = best_last.cpu().tolist()
        paths = []
        for b_i, length in enumerate(lengths):
            length = int(length)
            if length == 0:
                paths.append([])
                continue
            seq = [0] * length
            cur = last_cpu[b_i]
            seq[length - 1] = cur
            for t in range(length - 2, -1, -1):
                cur = bp_cpu[t][b_i][cur]
                seq[t] = cur
            paths.append(seq)
        return paths



# ==============================================================================
# 3. LUỒNG SONG SONG WORD-LEVEL UTF-8 BYTE-CNN
# ==============================================================================
class WordByteCNN(nn.Module):
    """Luồng UTF-8 Byte-CNN đa tỷ lệ (k=2, 3, 4) ở mức từ (~60K params)."""
    def __init__(self, out_dim, emb_dim=32, filters=48):
        super().__init__()
        self.byte_emb = nn.Embedding(257, emb_dim, padding_idx=0)
        self.conv2 = nn.Conv1d(emb_dim, filters, kernel_size=2)
        self.conv3 = nn.Conv1d(emb_dim, filters, kernel_size=3, padding=1)
        self.conv4 = nn.Conv1d(emb_dim, filters, kernel_size=4)
        self.proj = nn.Linear(filters * 3, out_dim)

    def forward(self, word_bytes, word_mask):
        # word_bytes: (B, W, B_max)
        B, W, L = word_bytes.shape
        flat_b = word_bytes.view(B * W, L)
        valid_b = (flat_b != 0)
        x = self.byte_emb(flat_b).transpose(1, 2)  # (B*W, E, L)
        x = x * valid_b.unsqueeze(1).to(x.dtype)

        def pool_conv(conv, x_in, pad_right=0, pad_left=0):
            if pad_left or pad_right:
                x_in = F.pad(x_in, (pad_left, pad_right))
            y = F.gelu(conv(x_in))  # (B*W, F, L)
            y = y.masked_fill(~valid_b.unsqueeze(1), -1e4)
            pooled = y.max(dim=-1).values
            has_any = valid_b.any(dim=-1, keepdim=True)
            return torch.where(has_any, pooled, torch.zeros_like(pooled))

        f2 = pool_conv(self.conv2, x, pad_right=1)
        f3 = pool_conv(self.conv3, x)
        f4 = pool_conv(self.conv4, x, pad_left=1, pad_right=2)
        feat = self.proj(torch.cat([f2, f3, f4], dim=-1)).view(B, W, -1)
        return feat * word_mask.unsqueeze(-1).to(feat.dtype)



# ==============================================================================
# 4. KIẾN TRÚC TỔNG HỢP: SafeViDualORCCRF
# ==============================================================================
class SafeViDualORCCRF(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        d_model = backbone.config.hidden_size
        self.d = WIDTH

        self.in_norm = nn.LayerNorm(d_model)
        self.sub_attn = nn.Linear(d_model, 1)
        self.sub_proj = nn.Linear(d_model, WIDTH)

        if USE_BYTE_CNN:
            self.byte_cnn = WordByteCNN(WIDTH)
            self.char_gate = nn.Linear(WIDTH * 2, WIDTH)
            nn.init.zeros_(self.char_gate.weight)
            nn.init.constant_(self.char_gate.bias, -2.0)  # Khởi đầu ~0.12 để không nhiễu ViSoBERT sớm
        self.fuse_norm = nn.LayerNorm(WIDTH)

        # Nhúng cấu trúc hội thoại luân phiên (std=0.02 để không át tín hiệu ngữ nghĩa)
        self.emb_parity = nn.Embedding(2, WIDTH)
        self.emb_turn = nn.Embedding(8, WIDTH)
        self.emb_recency = nn.Embedding(8, WIDTH)
        for emb in (self.emb_parity, self.emb_turn, self.emb_recency):
            nn.init.normal_(emb.weight, std=0.02)

        # Gộp lượt & khối HAT Turn-to-Word Cross-Attention
        self.word_turn_attn = nn.Linear(WIDTH, 1)
        if USE_HAT_ZOOM:
            self.turn_self_attn = nn.MultiheadAttention(WIDTH, 4, batch_first=True, dropout=0.1)
            self.t_norm1 = nn.LayerNorm(WIDTH)
            self.turn_word_xattn = nn.MultiheadAttention(WIDTH, 4, batch_first=True, dropout=0.1)
            self.t_norm2 = nn.LayerNorm(WIDTH)
            self.turn_ffn = nn.Sequential(
                nn.Linear(WIDTH, WIDTH * 2),
                nn.GELU(),
                nn.Dropout(0.1),
                nn.Linear(WIDTH * 2, WIDTH),
            )
            self.t_norm3 = nn.LayerNorm(WIDTH)

        # Đầu trích xuất bằng chứng mức từ (Conv đa tỷ lệ k=3, 5 + Partial-Label BIO-CRF)
        self.span_conv3 = nn.Conv1d(WIDTH, WIDTH, kernel_size=3, padding=1, groups=WIDTH, bias=False)
        self.span_conv5 = nn.Conv1d(WIDTH, WIDTH, kernel_size=5, padding=2, groups=WIDTH, bias=False)
        self.span_mix = nn.Linear(WIDTH * 2, WIDTH)
        self.span_norm = nn.LayerNorm(WIDTH)
        if USE_BIO_CRF:
            self.bio_head = nn.Linear(WIDTH, 3)
            self.crf = TurnAwareBIOCRF()
        else:
            self.bin_span_head = nn.Linear(WIDTH, 1)

        # Chuỗi ORC 4 bước + Multi-Prototype Sub-Centers (M tâm con/lớp)
        self.ctx_attn = nn.Linear(WIDTH, 1)
        self.ev_norm = nn.LayerNorm(WIDTH)
        self.harm_head = nn.Sequential(nn.Linear(WIDTH * 2, WIDTH // 2), nn.GELU(), nn.Dropout(0.1), nn.Linear(WIDTH // 2, 1))
        self.severity_head = nn.Sequential(nn.Linear(WIDTH * 2, WIDTH // 2), nn.GELU(), nn.Dropout(0.1), nn.Linear(WIDTH // 2, 1))

        self.M = NUM_SUBCENTERS
        self.queries = nn.Parameter(torch.randn(len(LABELS), self.M, WIDTH) / math.sqrt(WIDTH))
        self.class_vectors = nn.Parameter(torch.randn(len(LABELS), self.M, WIDTH) / math.sqrt(WIDTH))
        self.sub_bias = nn.Parameter(torch.zeros(len(LABELS), self.M))

        if USE_ORC:
            self.orc_cls_head = nn.Sequential(
                nn.Linear(WIDTH * 2 + 2, WIDTH),
                nn.GELU(),
                nn.Dropout(0.15),
                nn.Linear(WIDTH, len(LABELS)),
            )
            self.orc_blend_logit = nn.Parameter(torch.tensor(math.log(0.30 / 0.70)))

        if PROTO_WEIGHT > 0:
            self.prototypes = nn.Parameter(torch.randn(len(LABELS), self.M, WIDTH) / math.sqrt(WIDTH))
        self.drop = nn.Dropout(0.1)

    def _apply_token_drop(self, ids, attn_mask):
        if not (self.training and TOKEN_DROP_PROB > 0):
            return ids
        # Không che BOS/SEP/PAD
        droppable = attn_mask & (ids != BOS_ID) & (ids != SEP_ID) & (ids != PAD_ID)
        drop_mask = droppable & (torch.rand_like(ids, dtype=torch.float32) < TOKEN_DROP_PROB)
        return torch.where(drop_mask, torch.full_like(ids, MASK_DROP_ID), ids)

    def forward(self, b, with_aux=False):
        ids = self._apply_token_drop(b["ids"], b["attn_mask"])
        hidden = self.backbone(input_ids=ids, attention_mask=b["attn_mask"].long()).last_hidden_state

        with torch.autocast(device_type=hidden.device.type, enabled=False):
            h_tok = self.in_norm(hidden.float())  # (B, L, d_model)
            B, W, S = b["word_sub_idx"].shape
            w_mask = b["word_mask"]
            sub_mask = b["word_sub_mask"]

            # Gom subword về mức từ (0.5 mean + 0.5 subword-attention)
            flat_idx = b["word_sub_idx"].view(B, W * S)
            gathered = h_tok.gather(1, flat_idx.unsqueeze(-1).expand(-1, -1, h_tok.size(-1))).view(B, W, S, -1)
            sm_f = sub_mask.unsqueeze(-1).float()
            sub_cnt = sm_f.sum(dim=2).clamp_min(1.0)
            sub_mean = (gathered * sm_f).sum(dim=2) / sub_cnt
            s_score = self.sub_attn(gathered).squeeze(-1).clamp(-20.0, 20.0).masked_fill(~sub_mask, -1e4)
            s_w = F.softmax(s_score, dim=-1) * sub_mask.float()
            s_w = s_w / s_w.sum(dim=-1, keepdim=True).clamp_min(1e-6)
            sub_att = (gathered * s_w.unsqueeze(-1)).sum(dim=2)
            has_sub = sub_mask.any(dim=-1, keepdim=True).float()
            u_sub = self.sub_proj(0.5 * sub_mean + 0.5 * sub_att) * has_sub  # (B, W, D)

            # Hợp nhất luồng UTF-8 Byte-CNN
            if USE_BYTE_CNN:
                u_char = self.byte_cnn(b["word_bytes"], w_mask)
                gate = torch.sigmoid(self.char_gate(torch.cat([u_sub, u_char], dim=-1)))
                w_feat = self.fuse_norm(u_sub + gate * u_char)
            else:
                w_feat = self.fuse_norm(u_sub)

            # Nhúng cấu trúc hội thoại luân phiên (A/B parity + vị trí + recency)
            parity = b["word_turn"] % 2
            struct_emb = self.emb_parity(parity) + self.emb_turn(b["word_turn"]) + self.emb_recency(b["word_recency"])
            w_feat = (w_feat + struct_emb) * w_mask.unsqueeze(-1).float()

            # Gộp lượt T ∈ (B, 8, D) tương thích ONNX (không dùng scatter_reduce)
            turn_ids = torch.arange(8, device=w_feat.device)[None, :, None]  # (1, 8, 1)
            belongs = (b["word_turn"][:, None, :] == turn_ids) & w_mask[:, None, :]  # (B, 8, W)
            t_counts = belongs.sum(dim=-1)  # (B, 8)
            turn_present = t_counts > 0
            t_mean = torch.einsum("btw,bwd->btd", belongs.float(), w_feat) / t_counts.clamp_min(1).unsqueeze(-1).float()
            wt_score = self.word_turn_attn(w_feat).squeeze(-1).clamp(-20.0, 20.0)  # (B, W)
            wt_masked = wt_score[:, None, :].expand(-1, 8, -1).masked_fill(~belongs, -1e4)
            wt_w = F.softmax(wt_masked, dim=-1) * belongs.float()
            wt_w = wt_w / wt_w.sum(dim=-1, keepdim=True).clamp_min(1e-6)
            t_att = torch.einsum("btw,bwd->btd", wt_w, w_feat)
            turns = (0.5 * t_mean + 0.5 * t_att) * turn_present.unsqueeze(-1).float()

            # HAT: Turn Self-Attention + Turn-to-Word Cross-Attention Zoom-In
            if USE_HAT_ZOOM:
                t_self = self.turn_self_attn(turns, turns, turns, key_padding_mask=~turn_present, need_weights=False)[0]
                turns = self.t_norm1(turns + self.drop(t_self))
                t_zoom = self.turn_word_xattn(turns, w_feat, w_feat, key_padding_mask=~w_mask, need_weights=False)[0]
                turns = self.t_norm2(turns + self.drop(t_zoom))
                turns = self.t_norm3(turns + self.turn_ffn(turns))
                turns = turns * turn_present.unsqueeze(-1).float()

            # Truyền ngữ cảnh lượt trở lại từng từ
            t_for_w = turns.gather(1, b["word_turn"].unsqueeze(-1).expand(-1, -1, self.d))
            w_ctx = torch.tanh(w_feat + t_for_w) * w_mask.unsqueeze(-1).float()

            # Trích xuất đặc trưng cục bộ đa tỷ lệ (k=3, 5) cho đầu bằng chứng
            wc_t = w_ctx.transpose(1, 2)
            c3 = self.span_conv3(wc_t).transpose(1, 2) * w_mask.unsqueeze(-1).float()
            c5 = self.span_conv5(wc_t).transpose(1, 2) * w_mask.unsqueeze(-1).float()
            span_feat = self.span_norm(w_ctx + F.gelu(self.span_mix(self.drop(torch.cat([c3, c5], dim=-1))))) * w_mask.unsqueeze(-1).float()

            # Đầu bằng chứng: Partial-Label BIO-CRF hoặc Binary Sigmoid (ablation)
            # Dùng xác suất emission song song (0 vòng lặp) trong forward để tăng tốc ~3x và giữ ONNX động hoàn toàn theo W;
            # BIO-CRF NLL chính xác chạy trong hàm mất mát, còn Viterbi + Forward-Backward marginals chạy ở suy luận/hiệu chuẩn.
            if USE_BIO_CRF:
                emissions = self.bio_head(self.drop(span_feat))  # (B, W, 3)
                ev_prob = F.softmax(emissions, dim=-1)[:, :, 1:].sum(dim=-1) * w_mask.float()
            else:
                emissions = self.bin_span_head(self.drop(span_feat)).squeeze(-1)  # (B, W)
                ev_prob = emissions.sigmoid() * w_mask.float()

            # ORC Bước 1: Context Vector z_ctx
            ctx_sc = self.ctx_attn(turns).squeeze(-1).clamp(-20.0, 20.0).masked_fill(~turn_present, -1e4)
            ctx_w = F.softmax(ctx_sc, dim=-1) * turn_present.float()
            z_ctx = (turns * ctx_w.unsqueeze(-1)).sum(dim=1)  # (B, D)

            # ORC Bước 2: Evidence Indicator Vector z_ev (chuẩn hóa theo mass_sum + 1.0)
            ev_sum = ev_prob.sum(dim=1, keepdim=True)  # (B, 1)
            z_ev = self.ev_norm((ev_prob.unsqueeze(-1) * span_feat).sum(dim=1) / (ev_sum + 1.0))

            # ORC Bước 3: Harm Gate & Điểm nghiêm trọng liên tục FlexGuard [0, 1]
            orc_pair = torch.cat([z_ctx, z_ev], dim=-1)
            harm_logit = self.harm_head(self.drop(orc_pair)).squeeze(-1)
            severity = torch.sigmoid(self.severity_head(self.drop(orc_pair)).squeeze(-1))

            # ORC Bước 4: Multi-Prototype ML-Decoder (M tâm con/lớp) trên bộ nhớ [turns; z_ev]
            mem = torch.cat([turns, z_ev.unsqueeze(1)], dim=1)  # (B, 9, D)
            mem_mask = torch.cat([turn_present, torch.ones((B, 1), dtype=torch.bool, device=turns.device)], dim=1)
            q_scores = torch.einsum("btd,lmd->blmt", mem, self.queries) / math.sqrt(self.d)
            q_att = q_scores.masked_fill(~mem_mask[:, None, None, :], float("-inf")).softmax(dim=-1)
            q_pooled = torch.einsum("blmt,btd->blmd", q_att, mem)
            sub_logits = (self.drop(q_pooled) * self.class_vectors).sum(dim=-1) + self.sub_bias  # (B, 5, M)
            logits_direct = torch.logsumexp(sub_logits, dim=-1) - math.log(self.M)

            if USE_ORC:
                orc_in = torch.cat([z_ctx, z_ev, harm_logit.sigmoid().unsqueeze(-1), severity.unsqueeze(-1)], dim=-1)
                logits_orc = self.orc_cls_head(self.drop(orc_in))
                beta = torch.sigmoid(self.orc_blend_logit)
                logits = (1.0 - beta) * logits_direct + beta * logits_orc
            else:
                logits = logits_direct

            proto_logits = None
            if with_aux and PROTO_WEIGHT > 0:
                doc_n = F.normalize(z_ctx, dim=-1)  # (B, D)
                p_n = F.normalize(self.prototypes, dim=-1)  # (5, M, D)
                sim = torch.einsum("bd,lmd->blm", doc_n, p_n) / 0.15
                proto_logits = sim.max(dim=-1).values  # Sub-center max similarity

            out = dict(logits=logits.float(), emissions=emissions.float(), ev_prob=ev_prob.float(),
                       harm_logit=harm_logit.float(), severity=severity.float(), proto=proto_logits)
            return out



# ==============================================================================
# 5. KHỞI TẠO MÔ HÌNH & NGỮ CẢNH AMP
# ==============================================================================
def build_model():
    seed_all(SEED)
    net = SafeViDualORCCRF(load_backbone()).to(DEVICE)
    if GRAD_CKPT and hasattr(net.backbone, "gradient_checkpointing_enable"):
        net.backbone.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    return net

def amp():
    return torch.autocast("cuda", dtype=DTYPE) if DEVICE.type == "cuda" else nullcontext()

model = build_model()


# ==============================================================================
# 6. TỰ KIỂM ĐỊNH TOÁN HỌC (BIO-CRF 3^W & BẤT BIẾN PADDING)
# ==============================================================================
# Kiểm định 1: Tính chính xác toán học của TurnAwareBIOCRF so với vét cạn 3^W
if USE_BIO_CRF:
    with torch.no_grad():
        crf_t = model.crf
        W_test = 5
        em_t = torch.randn(1, W_test, 3, device=DEVICE)
        mk_t = torch.ones(1, W_test, dtype=torch.bool, device=DEVICE)
        ts_t = torch.tensor([[True, False, False, True, False]], device=DEVICE)  # Lượt 2 bắt đầu ở vị trí 3
        marg_t, logz_t = crf_t.marginals(em_t, mk_t, ts_t)
        vit_t = crf_t.decode(em_t, mk_t, ts_t)[0]
        # Duyệt vét cạn mọi chuỗi hợp lệ
        tr_eff = crf_t._effective_trans().cpu()
        st_eff = crf_t._effective_start().cpu()
        en_eff = crf_t.end_trans.cpu()
        em_c = em_t[0].cpu()
        ts_c = ts_t[0].cpu().tolist()
        valid_scores, valid_paths = [], []
        from itertools import product
        for path in product((0, 1, 2), repeat=W_test):
            if path[0] == 2:
                continue
            ok = True
            sc = float(st_eff[path[0]] + em_c[0, path[0]])
            for t in range(1, W_test):
                if (path[t - 1] == 0 and path[t] == 2) or (ts_c[t] and path[t] == 2):
                    ok = False
                    break
                sc += float(tr_eff[path[t - 1], path[t]] + em_c[t, path[t]])
            if ok:
                sc += float(en_eff[path[-1]])
                valid_scores.append(sc)
                valid_paths.append(list(path))
        bf_logz = float(torch.logsumexp(torch.tensor(valid_scores), dim=0))
        bf_best = valid_paths[int(np.argmax(valid_scores))]
        assert abs(float(logz_t[0]) - bf_logz) < 1e-4, "CRF log_Z lệch vét cạn"
        assert vit_t == bf_best and vit_t[3] != 2, f"Viterbi lệch vét cạn: {vit_t} vs {bf_best}"
        # Mẫu toàn -100 phải cho NLL == 0.0
        nll_masked = float(crf_t.nll(em_t, torch.full((1, W_test), -100, device=DEVICE), mk_t, ts_t))
        assert abs(nll_masked) < 1e-6, f"NLL toàn -100 khác 0: {nll_masked}"
    # Kiểm định 1b: LEG polar_weights tác động đúng MỨC TỪ (không làm phẳng sang từ trung tính)
    em_g1 = em_t.detach().clone().requires_grad_(True)
    em_g2 = em_t.detach().clone().requires_grad_(True)
    y_leg = torch.tensor([[0, 1, 2, 0, 0]], device=DEVICE)
    pw_base = torch.ones((1, W_test), device=DEVICE)
    pw_boost = torch.tensor([[1.0, 1.0, 1.5, 1.0, 1.0]], device=DEVICE)
    crf_t.nll(em_g1, y_leg, mk_t, ts_t, polar_weights=pw_base).backward()
    crf_t.nll(em_g2, y_leg, mk_t, ts_t, polar_weights=pw_boost).backward()
    assert float((em_g2.grad[0, 0] - em_g1.grad[0, 0]).abs().max()) < 1e-6, "LEG bị rò gradient sang từ trung tính!"
    assert float((em_g2.grad[0, 2] - em_g1.grad[0, 2]).abs().max()) > 1e-3, "LEG không tác động lên từ phân cực!"
    model.zero_grad(set_to_none=True)
    del crf_t, em_t, mk_t, ts_t, marg_t, logz_t, vit_t, em_g1, em_g2, y_leg, pw_base, pw_boost

# Kiểm định 2: Bất biến tuyệt đối với padding trên 1 lượt và đa lượt
with torch.no_grad(), amp():
    model.eval()
    short = prepare(dict(id="short", text="Cảm ơn bạn", nhan="AN_TOAN", bang_chung=[]))
    multi = prepare(dict(id="multi", text="Chào bạn nhé\nChiều gặp ở thư viện", nhan="AN_TOAN", bang_chung=[]))
    long_ex = prepare(dict(id="long", text="Chiều nay mình gặp nhau ở thư viện trường lúc mấy giờ để làm tiếp bài tập nhóm vậy bạn?\nNhớ mang theo máy tính và tài liệu bản in nhé.", nhan="AN_TOAN", bang_chung=[]))
    atol = {torch.float32: 1e-4, torch.float16: 2e-2, torch.bfloat16: 5e-2}[DTYPE]
    for target in (short, multi):
        single = to_device(collate([target]), DEVICE)
        padded = to_device(collate([target, long_ex]), DEVICE)
        o1 = model(single)
        o2 = model(padded)
        nw = target["n_words"]
        torch.testing.assert_close(o1["logits"][0], o2["logits"][0], atol=atol, rtol=atol)
        torch.testing.assert_close(o1["ev_prob"][0, :nw], o2["ev_prob"][0, :nw], atol=atol, rtol=atol)
        torch.testing.assert_close(o1["severity"][0], o2["severity"][0], atol=atol, rtol=atol)
    del short, multi, long_ex, target, single, padded, o1, o2
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

tot_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
head_p = sum(p.numel() for k, p in model.named_parameters() if p.requires_grad and not k.startswith("backbone."))
print(f"Kiểm định CRF + Padding PASSED | Tổng tham số học: {tot_p:,} (Backbone: {tot_p - head_p:,} | Head: {head_p:,})")

---
<a id="sec-5"></a>
## 5. Huấn luyện mô hình, Hàm mất mát đa nhiệm & Chọn Checkpoint

### Bảng thành phần hàm mất mát & Tối ưu hóa

| Thành phần | Cấu hình & Vai trò |
| :--- | :--- |
| **Phân loại 5 nhãn** | Trọng số phân tầng có kiểm soát (`REAL_RARE_CAP = 2,5`) + Focal Cross-Entropy (`FOCAL_GAMMA = 1,0`, `LABEL_SMOOTHING = 0,02`) + Soft Mutual-Exclusion (`L_mutex`) + Prototype Contrastive (`PROTO_WEIGHT = 0,08`). |
| **Chuỗi ORC phụ trợ** | Binary Harm BCE (`y != AN_TOAN`) + Smooth-L1 trên điểm mức độ ưu tiên `[0.0, 0.45, 0.65, 1.00, 0.85]`. |
| **Trích xuất bằng chứng** | Partial-Label BIO-CRF NLL + **LEG phân cực áp trực tiếp ở mức từ** (per-word emission CE + weighted Soft Dice trên `P(B) + P(I)`). |
| **Chống né lọc & Mất dấu** | Dual-View JS Consistency (`CONSISTENCY_WEIGHT = 0,25`) trên cặp `(gốc, biến dạng C1/bỏ dấu)` + FGM đóng gói cục bộ (giải phóng ngay `g_grad`, không rò rỉ VRAM). |
| **Tối ưu hóa LLRD** | **Depth-Normalized LLRD** tự điều chỉnh theo chiều sâu backbone (`0,920` cho 12 lớp `ViSoBERT`/`BamiBERT`, `0,956` cho 22 lớp `mmBERT`). |

### Tiêu chí chọn Checkpoint & Quản lý bộ nhớ GPU
- **Chế độ `dev_80_10_10` (`CKPT_SELECT_MODE = "real_aligned"`):** Sử dụng công thức tổng hợp `0.50 * macro_f1 + 0.15 * that_macro_f1 + 0.25 * that_acc + 0.10 * ev_f1` khi `SPAN_WEIGHT > 0` (hoặc `0.55 * macro_f1 + 0.20 * that_macro_f1 + 0.25 * that_acc` khi `SPAN_WEIGHT == 0`). Trọng số `ckpt_select_weights` được ghi vào `config.json` để bảo đảm mỗi thư mục `RUN` tương ứng duy nhất với một cấu hình thực nghiệm.
- **Chế độ `final_90_10` / `final_100_human_cal`:** Huấn luyện đủ `EPOCHS` cố định, hoàn toàn không dùng `cal` để chọn epoch.
- **Dọn sạch VRAM sau train:** Cuối ô giải phóng toàn bộ `opt`, `scheduler`, `scaler` và gradient khỏi GPU để phép đo VRAM suy luận ở Mục 7 phản ánh đúng dung lượng triển khai.

In [ ]:
#@title 5.1. Hàm mất mát đa nhiệm, Suy luận/Chuẩn hóa #6, Giải mã bằng chứng & Huấn luyện
# ==============================================================================
# 1. TRỌNG SỐ PHÂN TẦNG & HÀM MẤT MÁT ĐA NHIỆM (FOCAL / MUTEX / ORC / CRF / JS)
# ==============================================================================
def compute_weights(data):
    lc = Counter(e["label"] for e in data)
    n_tot = len(data)
    raw_lbl = {k: (n_tot / lc[k]) ** 0.5 for k in range(len(LABELS))}
    mean_lbl = sum(raw_lbl[e["label"]] for e in data) / max(n_tot, 1)
    w_lbl = {k: v / mean_lbl for k, v in raw_lbl.items()}
    if STRATUM_WEIGHT:
        sc = Counter((e["label"], e["row"]["meta"]["loai"]) for e in data)
        raw = {}
        for (lbl, loai), c in sc.items():
            base = w_lbl[lbl]
            if loai == "that" and c < 0.25 * lc[lbl]:
                mult = min(2.0, (lc[lbl] / c) ** 0.25)
                raw[(lbl, loai)] = base * mult
            else:
                raw[(lbl, loai)] = base
        mw = sum(raw[(e["label"], e["row"]["meta"]["loai"])] for e in data) / max(n_tot, 1)
        sw = {k: v / mw for k, v in raw.items()}
        for _ in range(5):
            sw = {k: min(REAL_RARE_CAP, v) for k, v in sw.items()}
            mw = sum(sw[(e["label"], e["row"]["meta"]["loai"])] for e in data) / max(n_tot, 1)
            sw = {k: v / mw for k, v in sw.items()}
        sw = {k: min(REAL_RARE_CAP, v) for k, v in sw.items()}
    else:
        sw = w_lbl
    # pos_weight chỉ dùng khi ablation USE_BIO_CRF = False
    pos_w = sum(1 for e in data for m in e["words_meta"] if m[5] in (1, 2))
    neg_w = sum(1 for e in data for m in e["words_meta"] if m[5] == 0)
    pw = torch.tensor(min(3.0, (neg_w / max(pos_w, 1)) ** 0.5), device=DEVICE)
    return sw, pw

stratum_weights, bin_pos_weight = compute_weights(encoded["fit"])
SEVERITY_TENSOR = torch.tensor(SEVERITY_TARGETS, dtype=torch.float32, device=DEVICE)
print("Trọng số mẫu:", {str(k): round(v, 3) for k, v in sorted(stratum_weights.items())}, "| bin_pos_weight:", round(float(bin_pos_weight), 3))

def sample_weights(b):
    if STRATUM_WEIGHT:
        vals = [stratum_weights.get((e["label"], e["row"]["meta"]["loai"]), 1.0) for e in b["examples"]]
    else:
        vals = [stratum_weights.get(e["label"], 1.0) for e in b["examples"]]
    return torch.tensor(vals, dtype=torch.float32, device=DEVICE)

def accumulation_windows(loader, size):
    iterator = iter(loader)
    while window := list(islice(iterator, size)):
        yield window

def js_divergence_probs(p, q, eps=1e-6):
    p = p.clamp(eps, 1.0 - eps)
    q = q.clamp(eps, 1.0 - eps)
    m = 0.5 * (p + q)
    kl_pm = (p * (p.log() - m.log())).sum(dim=-1)
    kl_qm = (q * (q.log() - m.log())).sum(dim=-1)
    return 0.5 * (kl_pm + kl_qm)

def objective(net, out, b, class_mass, use_consistency=True, pair_count=None):
    logits = out["logits"]
    labels = b["labels"]
    sw = sample_weights(b)
    frac = sw.sum() / class_mass

    # 1) Phân loại: Focal + Stratum-Weighted Cross-Entropy
    ce = F.cross_entropy(logits, labels, reduction="none", label_smoothing=LABEL_SMOOTHING)
    if FOCAL_GAMMA > 0:
        pt = F.softmax(logits, dim=-1).gather(1, labels.unsqueeze(1)).squeeze(1).clamp(1e-6, 1.0)
        ce = ((1.0 - pt) ** FOCAL_GAMMA) * ce
    loss = (ce * sw).sum() / class_mass

    # 2) Soft Mutual-Exclusion (SingProbe): phạt Safe và max Risk cùng cao
    if MUTEX_WEIGHT > 0:
        probs_sig = logits.sigmoid()
        p_safe = probs_sig[:, 0]
        p_risk_max = probs_sig[:, 1:].max(dim=-1).values
        loss = loss + MUTEX_WEIGHT * ((p_safe * p_risk_max) * sw).sum() / class_mass

    # 3) ORC Auxiliary: Binary Harm Gate + Continuous Priority Severity
    if ORC_AUX_WEIGHT > 0:
        harm_y = (labels != 0).float()
        sev_y = SEVERITY_TENSOR[labels]
        l_harm = F.binary_cross_entropy_with_logits(out["harm_logit"], harm_y, reduction="none")
        l_sev = F.smooth_l1_loss(out["severity"], sev_y, reduction="none", beta=0.1)
        loss = loss + ORC_AUX_WEIGHT * ((0.6 * l_harm + 0.4 * l_sev) * sw).sum() / class_mass

    # 4) Prototype Contrastive Auxiliary
    if out["proto"] is not None and PROTO_WEIGHT > 0:
        p_ce = F.cross_entropy(out["proto"], labels, reduction="none")
        loss = loss + PROTO_WEIGHT * (p_ce * sw).sum() / class_mass

    # 5) Đầu bằng chứng: Partial-Label BIO-CRF (với LEG mức từ) + Weighted Soft Dice
    if SPAN_WEIGHT > 0:
        w_mask = b["word_mask"]
        bio_y = b["bio_targets"]
        known = w_mask & (bio_y >= 0)
        if USE_BIO_CRF:
            l_span_main = net.crf.nll(
                out["emissions"], bio_y, w_mask, b["word_turn_start"],
                polar_weights=(b["polar_weights"] if LEG_POLAR_WEIGHT > 0 else None),
            )
        else:
            bin_t = (bio_y > 0).float()
            bce_w = F.binary_cross_entropy_with_logits(
                out["emissions"], bin_t, pos_weight=bin_pos_weight, reduction="none"
            )
            if LEG_POLAR_WEIGHT > 0:
                bce_w = bce_w * b["polar_weights"]
            l_span_main = bce_w.masked_fill(~known, 0.0).sum() / known.sum().clamp_min(1)

        if SPAN_DICE_WEIGHT > 0 and known.any():
            ev_p = out["ev_prob"]
            bin_t = (bio_y > 0).float()
            wm = known.float() * (b["polar_weights"] if LEG_POLAR_WEIGHT > 0 else 1.0)
            inter = (ev_p * bin_t * wm).sum()
            denom = (ev_p * wm).sum() + (bin_t * wm).sum()
            dice = 1.0 - (2.0 * inter + 2.0) / (denom + 2.0)
            span_loss = (1.0 - SPAN_DICE_WEIGHT) * l_span_main + SPAN_DICE_WEIGHT * dice
        else:
            span_loss = l_span_main
        loss = loss + SPAN_WEIGHT * span_loss * frac

    # 6) Dual-View Consistency Loss (JS-divergence trên cặp clean <-> bỏ dấu / C1, bất biến theo split_microbatch & ACCUM)
    if use_consistency and CONSISTENCY_WEIGHT > 0 and b["pairs"]:
        i0 = torch.tensor([p[0] for p in b["pairs"]], device=logits.device)
        i1 = torch.tensor([p[1] for p in b["pairs"]], device=logits.device)
        p0 = F.softmax(logits[i0], dim=-1)
        p1 = F.softmax(logits[i1], dim=-1)
        pair_denom = max(pair_count if pair_count is not None else len(b["pairs"]), 1)
        l_cons = js_divergence_probs(p0, p1).sum() / pair_denom
        if SPAN_WEIGHT > 0:
            w0 = b["word_mask"][i0]
            w1 = b["word_mask"][i1]
            same_len = (w0.sum(dim=1) == w1.sum(dim=1))
            if same_len.any():
                e0 = out["ev_prob"][i0[same_len]]
                e1 = out["ev_prob"][i1[same_len]]
                wm = w0[same_len].float()
                ev_per_pair = ((e0 - e1) ** 2 * wm).sum(dim=1) / wm.sum(dim=1).clamp_min(1.0)
                l_cons = l_cons + 0.5 * ev_per_pair.sum() / pair_denom
        loss = loss + CONSISTENCY_WEIGHT * l_cons

    return loss



# ==============================================================================
# 2. TỐI ƯU HÓA DEPTH-NORMALIZED LLRD & NHIỄU ĐỐI KHÁNG FGM
# ==============================================================================
def build_llrd_param_groups(net):
    """Phân nhóm tham số với Depth-Normalized Layer-wise LR Decay (hỗ trợ cả XLM-RoBERTa lẫn ModernBERT)."""
    no_decay_keys = (".bias", "norm", "Norm", "emb_parity", "emb_turn", "emb_recency", "start_trans", "end_trans", "trans")
    def is_nd(k, p):
        return p.ndim < 2 or any(tok_k in k for tok_k in no_decay_keys)

    num_layers = getattr(net.backbone.config, "num_hidden_layers", SPEC["num_layers"])
    eff_llrd = resolve_effective_llrd(num_layers, LLRD, LLRD_BOTTOM_RATIO)
    layer_re = re.compile(r"(?:encoder\.layer|layers)\.(\d+)\.")

    grouped = defaultdict(list)
    bb_lrs = []
    for k, p in net.named_parameters():
        if not p.requires_grad:
            continue
        nd = is_nd(k, p)
        wd = 0.0 if nd else 0.01
        if not k.startswith("backbone."):
            lr = LR_HEAD
        else:
            m = layer_re.search(k)
            if m:
                depth = int(m.group(1))
                lr = LR_BACKBONE * (eff_llrd ** (num_layers - 1 - depth))
            elif "embeddings" in k:
                lr = LR_BACKBONE * (eff_llrd ** num_layers)
            else:
                lr = LR_BACKBONE
            bb_lrs.append(lr)
        grouped[(round(lr, 10), wd)].append(p)
    print(f"Depth-Normalized LLRD: num_layers={num_layers} | llrd_eff={eff_llrd:.4f} | bottom/top={eff_llrd**num_layers:.3f} | bb_lr=[{min(bb_lrs):.2e} .. {max(bb_lrs):.2e}] | head_lr={LR_HEAD:.2e}")
    return [{"params": ps, "lr": lr, "weight_decay": wd} for (lr, wd), ps in sorted(grouped.items())]


def _fgm_adversarial_step(net, emb_weight, dev_subs, class_mass, scaler):
    """Chạy 1 bước nhiễu đối kháng FGM trong phạm vi cục bộ để không rò rỉ g_grad ra biến toàn cục."""
    if emb_weight.grad is None or not dev_subs:
        return
    g_grad = emb_weight.grad.detach()
    active_ids = torch.nonzero(g_grad.abs().amax(dim=-1) > 0, as_tuple=False).squeeze(-1)
    if active_ids.numel() == 0:
        return
    g_rows = g_grad.index_select(0, active_ids).float()
    g_norm = g_rows.norm()
    if not (torch.isfinite(g_norm) and g_norm > 0):
        return
    r_rows = (FGM_EPS * g_rows / (g_norm + 1e-12)).to(emb_weight.dtype)
    with torch.no_grad():
        emb_weight.index_add_(0, active_ids, r_rows)
    try:
        for b in dev_subs:
            with amp():
                adv_loss = 0.5 * objective(net, net(b, with_aux=False), b, class_mass, use_consistency=False)
            scaler.scale(adv_loss).backward()
    finally:
        with torch.no_grad():
            emb_weight.index_add_(0, active_ids, -r_rows)
        del g_grad, active_ids, g_rows, r_rows



# ==============================================================================
# 3. SUY LUẬN GỐC (#5) & BỘ CHUẨN HÓA 5 TẦNG ÁNH XẠ NGƯỢC VERBATIM (#6)
# ==============================================================================
def word_record_from_outputs(row, words_meta, logits, ev_probs, emissions_row, severity_val):
    words = []
    for (a, b, t_idx, _, is_ts, _, bin_g, _, _), sc in zip(words_meta, ev_probs):
        words.append((a, b, float(sc), int(bin_g), int(t_idx), bool(is_ts)))
    return dict(
        id=row["id"],
        text=row["text"],
        gold=LABELS.index(row["nhan"]),
        gold_phrases=list(row.get("bang_chung") or []),
        logits=logits,
        severity=float(severity_val),
        words=words,
        emissions=emissions_row,
    )

@torch.inference_mode()
def infer(net, data, device=DEVICE, batch_size=EVAL_BATCH):
    net.eval()
    out = []
    for batch in batches(data, batch_size=batch_size):
        for sub in split_microbatch(batch):
            b = to_device(sub, device)
            with (amp() if device.type == "cuda" else nullcontext()):
                res = net(b, with_aux=False)
            logits_cpu = res["logits"].cpu().tolist()
            if USE_BIO_CRF:
                marg_p, _ = net.crf.marginals(res["emissions"], b["word_mask"], b["word_turn_start"])
                ev_cpu = (marg_p[:, :, 1] + marg_p[:, :, 2]).clamp(0.0, 1.0).cpu().tolist()
            else:
                ev_cpu = res["ev_prob"].cpu().tolist()
            sev_cpu = res["severity"].cpu().tolist()
            em_cpu = res["emissions"].cpu().numpy() if USE_BIO_CRF else None
            for i, e in enumerate(sub["examples"]):
                nw = e["n_words"]
                em_i = em_cpu[i, :nw].copy() if em_cpu is not None else None
                out.append(word_record_from_outputs(
                    e["row"], e["words_meta"], logits_cpu[i], ev_cpu[i][:nw], em_i, sev_cpu[i]
                ))
    return out

_NORM_LOOKALIKE = {"4": "a", "3": "e", "1": "i", "0": "o", "5": "s", "9": "g", "8": "b", "@": "a"}
_NORM_SEP_RE = re.compile(r"(?<=[a-zA-ZÀ-ỹ])[.\-_·~](?=[a-zA-ZÀ-ỹ])")
_NORM_ELONG_RE = re.compile(r"([a-zA-ZÀ-ỹ])\1{2,}")
_NORM_LEAD_RE = re.compile(r"^([^\w]*)")
_NORM_TRAIL_RE = re.compile(r"([^\w]*)$")

def _norm_strip_base(ch):
    s = ch.replace("đ", "d").replace("Đ", "D")
    return "".join(c for c in unicodedata.normalize("NFD", s) if not unicodedata.combining(c))

def _normalize_v1_token(tok_str, teencode_map):
    t = unicodedata.normalize("NFC", tok_str)
    if t and not t[0].isdigit():
        na = sum(1 for c in t if c.isalpha())
        nc = sum(1 for c in t if c in _NORM_LOOKALIKE)
        if nc > 0 and na > nc:
            t = "".join(_NORM_LOOKALIKE.get(c, c) for c in t)
    t = _NORM_SEP_RE.sub("", t)
    t = _NORM_ELONG_RE.sub(r"\1", t)
    if len(t) >= 2:
        chars = list(t)
        res = [chars[0]]
        for i in range(1, len(chars)):
            p, c = chars[i - 1], chars[i]
            if p != c and c.isalpha() and p.isalpha() and _norm_strip_base(p) == _norm_strip_base(c) and p != _norm_strip_base(p):
                continue
            res.append(c)
        t = "".join(res)
    lead = _NORM_LEAD_RE.match(t).group(1)
    trail = _NORM_TRAIL_RE.search(t).group(1)
    core = t[len(lead):len(t) - len(trail)] if trail else t[len(lead):]
    if core and teencode_map:
        rep = teencode_map.get(core.lower())
        if rep:
            if core[0].isupper() and rep[0].islower():
                rep = rep[0].upper() + rep[1:]
            t = lead + rep + trail
    return t.strip() or tok_str

def normalize_with_alignment(text, teencode_map):
    """Chuẩn hóa 5 tầng kiểu v1 theo từng dòng (giữ nguyên \n) và ánh xạ ngược mỗi từ về chỉ số từ văn bản gốc."""
    norm_lines, norm_to_orig, orig_idx = [], [], 0
    for line in text.split("\n"):
        out_words = []
        for m in re.finditer(r"\S+", line):
            nt = _normalize_v1_token(m.group(0), teencode_map)
            pieces = nt.split() or [m.group(0)]
            for p in pieces:
                out_words.append(p)
                norm_to_orig.append(orig_idx)
            orig_idx += 1
        norm_lines.append(" ".join(out_words))
    return "\n".join(norm_lines), norm_to_orig

def infer_normalized(net, records, teencode_map, infer_fn=None, prep_fn=None):
    """Suy luận Dòng #6 (#5 + chuẩn hóa đầu vào giữ dòng) và chiếu ngược điểm bằng chứng về văn bản gốc (100% verbatim)."""
    _prep = prepare if prep_fn is None else prep_fn
    _inf = infer if infer_fn is None else infer_fn
    norm_encoded, mappings = [], []
    for r in records:
        norm_text, norm_to_orig = normalize_with_alignment(r["text"], teencode_map)
        norm_encoded.append(_prep(dict(r, text=norm_text, bang_chung=[])))
        mappings.append(norm_to_orig)
    raw_preds = _inf(net, norm_encoded)
    projected = []
    for r, p, norm_to_orig in zip(records, raw_preds, mappings):
        yes, unknown = annotations(r)
        skip = r["nhan"] != "AN_TOAN" and not yes and not unknown
        orig_spans = []
        cursor = 0
        for t_idx, line in enumerate(r["text"].split("\n")):
            first_in_turn = True
            for m in re.finditer(r"\S+", line):
                a, b = cursor + m.start(), cursor + m.end()
                if skip or overlaps((a, b), unknown):
                    bin_g = -1
                else:
                    bin_g = int(overlaps((a, b), yes))
                orig_spans.append((a, b, bin_g, t_idx, first_in_turn))
                first_in_turn = False
            cursor += len(line) + 1
        by_orig = defaultdict(list)
        n_pw = len(p["words"])
        for j, oi in enumerate(norm_to_orig):
            if j < n_pw:
                by_orig[oi].append(j)
        orig_words, orig_em = [], []
        has_em = p.get("emissions") is not None
        for oi, (a, b, bin_g, t_idx, is_ts) in enumerate(orig_spans):
            js = by_orig.get(oi)
            if js:
                best_j = max(js, key=lambda j: p["words"][j][2])
                sc = float(p["words"][best_j][2])
                if has_em:
                    if len(js) == 1:
                        orig_em.append(p["emissions"][js[0]])
                    else:
                        em_sub = p["emissions"][js]
                        orig_em.append(np.array([
                            float(em_sub[:, 0].min()),
                            float(em_sub[:, 1].max()),
                            float(em_sub[:, 2].max()),
                        ], dtype=np.float32))
            else:
                sc = 0.0
                if has_em:
                    orig_em.append(np.array([2.0, -2.0, -2.0], dtype=np.float32))
            orig_words.append((a, b, sc, int(bin_g), int(t_idx), bool(is_ts)))
        em_arr = np.asarray(orig_em, dtype=np.float32) if (has_em and orig_em) else None
        rec_proj = dict(
            id=r["id"],
            text=r["text"],
            gold=LABELS.index(r["nhan"]),
            gold_phrases=list(r.get("bang_chung") or []),
            logits=p["logits"],
            severity=float(p.get("severity", 0.0)),
            words=orig_words,
            emissions=em_arr,
        )
        if has_em:
            rec_proj["norm_emissions"] = p["emissions"]
            rec_proj["norm_turn_starts"] = [bool(w[5]) for w in p["words"]]
            rec_proj["norm_to_orig"] = list(norm_to_orig)
        projected.append(rec_proj)
    return projected


# ==============================================================================
# 4. GIẢI MÃ BẰNG CHỨNG (BIO-CRF VITERBI / MARGINAL + HẬU XỬ LÝ / FALLBACK)
# ==============================================================================
def _viterbi_single_np(emissions_np, turn_starts, start_tr, end_tr, trans, o_bias=0.0):
    """Giải mã Viterbi thuần NumPy cho 1 mẫu (chính xác như PyTorch, dễ port sang JS/CPU)."""
    W = len(turn_starts)
    if W == 0:
        return []
    em = emissions_np.astype(np.float64, copy=True)
    em[:, 0] += float(o_bias)
    st = start_tr.astype(np.float64, copy=True)
    st[2] = -1e4
    tr = trans.astype(np.float64, copy=True)
    tr[0, 2] = -1e4

    dp = st + em[0]
    bp = np.zeros((W, 3), dtype=np.int64)
    for t in range(1, W):
        step = tr.copy()
        if turn_starts[t]:
            step[:, 2] = -1e4
        cand = dp[:, None] + step
        bp[t] = cand.argmax(axis=0)
        dp = cand[bp[t], np.arange(3)] + em[t]
    dp = dp + end_tr
    seq = [0] * W
    cur = int(dp.argmax())
    seq[W - 1] = cur
    for t in range(W - 2, -1, -1):
        cur = int(bp[t + 1, cur])
        seq[t] = cur
    return seq

def get_crf_params_np(net):
    if not USE_BIO_CRF:
        return None
    return dict(
        start=net.crf.start_trans.detach().cpu().numpy(),
        end=net.crf.end_trans.detach().cpu().numpy(),
        trans=net.crf.trans.detach().cpu().numpy(),
    )

def _apply_marginal_postproc(words, text, threshold=0.5, single_word_threshold=None, bridge_ratio=0.0):
    n_w = len(words)
    marks = [w[2] >= threshold for w in words]
    if bridge_ratio > 0 and n_w >= 3:
        bridge_thr = threshold * bridge_ratio
        bridged = marks[:]
        for i in range(1, n_w - 1):
            if not marks[i] and marks[i - 1] and marks[i + 1] and words[i][2] >= bridge_thr:
                if "\n" not in text[words[i - 1][1]:words[i + 1][0]]:
                    bridged[i] = True
        marks = bridged
    if single_word_threshold is not None and single_word_threshold > threshold:
        filtered = marks[:]
        for i in range(n_w):
            if not marks[i]:
                continue
            left_same = i > 0 and marks[i - 1] and "\n" not in text[words[i - 1][1]:words[i][0]]
            right_same = i + 1 < n_w and marks[i + 1] and "\n" not in text[words[i][1]:words[i + 1][0]]
            if not left_same and not right_same and words[i][2] < single_word_threshold:
                filtered[i] = False
        marks = filtered
    return marks

def _apply_empty_fallback(marks, words, text, fallback_min_prob=None, expand_ratio=0.70, max_words=5):
    """Fallback có ngưỡng tin cậy: chỉ chọn cụm đỉnh cùng dòng khi mẫu đã dự đoán nguy hại nhưng marks đang rỗng
    và từ có xác suất biên cao nhất đạt >= fallback_min_prob (không ép tô khi mô hình không chắc)."""
    if any(marks) or not words or fallback_min_prob is None or fallback_min_prob > 1.0:
        return marks
    probs = [w[2] for w in words]
    peak_i = int(np.argmax(probs))
    peak_p = probs[peak_i]
    if peak_p < fallback_min_prob:
        return marks
    out = list(marks)
    out[peak_i] = True
    cut = peak_p * expand_ratio
    lo = hi = peak_i
    while (hi - lo + 1) < max_words:
        can_l = lo > 0 and "\n" not in text[words[lo - 1][1]:words[lo][0]] and probs[lo - 1] >= cut
        can_r = hi + 1 < len(words) and "\n" not in text[words[hi][1]:words[hi + 1][0]] and probs[hi + 1] >= cut
        if not can_l and not can_r:
            break
        if can_l and (not can_r or probs[lo - 1] >= probs[hi + 1]):
            lo -= 1
            out[lo] = True
        else:
            hi += 1
            out[hi] = True
    return out

def _decode_item_crf_tags(item, crf_np, o_bias=0.0):
    """Giải mã Viterbi; với mẫu chuẩn hóa Dòng #6 chạy Viterbi nguyên bản trên chuỗi đã chuẩn hóa rồi chiếu ngược về từ gốc."""
    words = item["words"]
    n_w = len(words)
    if n_w == 0:
        return []
    if item.get("norm_emissions") is not None and item.get("norm_to_orig") is not None:
        norm_em = item["norm_emissions"]
        norm_ts = item.get("norm_turn_starts")
        if norm_ts is None:
            norm_ts = [i == 0 for i in range(len(norm_em))]
        norm_tags = _viterbi_single_np(
            norm_em, norm_ts, crf_np["start"], crf_np["end"], crf_np["trans"], o_bias=o_bias,
        )
        by_orig = defaultdict(list)
        for j, oi in enumerate(item["norm_to_orig"]):
            if j < len(norm_tags):
                by_orig[oi].append(j)
        tags = [0] * n_w
        for oi in range(n_w):
            js = by_orig.get(oi)
            if not js:
                continue
            sub_tags = [norm_tags[j] for j in js]
            if any(t != 0 for t in sub_tags):
                is_ts = bool(words[oi][5])
                tags[oi] = 2 if (not is_ts and sub_tags[0] == 2 and oi > 0 and tags[oi - 1] != 0) else 1
        return tags
    ts = [w[5] for w in words]
    return _viterbi_single_np(item["emissions"], ts, crf_np["start"], crf_np["end"], crf_np["trans"], o_bias=o_bias)

def select_word_marks(
    item, threshold=0.5, o_bias=0.0, crf_np=None, gate=True, mode="marginal",
    single_word_threshold=None, bridge_ratio=0.0, fallback_min_prob=None,
):
    words = item["words"]
    n_w = len(words)
    if not n_w or (gate and int(np.argmax(item["logits"])) == 0):
        return [False] * n_w
    use_crf_dec = (mode == "crf") and USE_BIO_CRF and crf_np is not None and (
        item.get("norm_emissions") is not None or item.get("emissions") is not None
    )
    if use_crf_dec:
        tags = _decode_item_crf_tags(item, crf_np, o_bias=o_bias)
        marks = [t != 0 for t in tags]
    else:
        marks = _apply_marginal_postproc(
            words, item["text"], threshold=threshold,
            single_word_threshold=single_word_threshold, bridge_ratio=bridge_ratio,
        )
    if gate and fallback_min_prob is not None:
        marks = _apply_empty_fallback(marks, words, item["text"], fallback_min_prob=fallback_min_prob)
    return marks

def extract_evidence_runs(
    item, text, threshold=0.5, o_bias=0.0, crf_np=None, gate=True, mode="marginal",
    single_word_threshold=None, bridge_ratio=0.0, fallback_min_prob=None,
):
    words = item["words"]
    if not words or (gate and int(np.argmax(item["logits"])) == 0):
        return []
    use_crf_dec = (mode == "crf") and USE_BIO_CRF and crf_np is not None and (
        item.get("norm_emissions") is not None or item.get("emissions") is not None
    )
    runs = []
    if use_crf_dec:
        tags = _decode_item_crf_tags(item, crf_np, o_bias=o_bias)
        if any(t != 0 for t in tags):
            for w, tag in zip(words, tags):
                if tag == 0:
                    continue
                a, b = w[0], w[1]
                if tag == 2 and runs and re.fullmatch(r"[^\S\n]*", text[runs[-1][1]:a]):
                    runs[-1][1] = b
                else:
                    runs.append([a, b])
            return runs
        if not gate or fallback_min_prob is None:
            return []
        marks = _apply_empty_fallback([False] * len(words), words, text, fallback_min_prob=fallback_min_prob)
    else:
        marks = select_word_marks(
            item, threshold=threshold, o_bias=o_bias, crf_np=crf_np, gate=gate, mode="marginal",
            single_word_threshold=single_word_threshold, bridge_ratio=bridge_ratio,
            fallback_min_prob=fallback_min_prob,
        )
    for w, marked in zip(words, marks):
        if not marked:
            continue
        a, b = w[0], w[1]
        if runs and re.fullmatch(r"[^\S\n]*", text[runs[-1][1]:a]):
            runs[-1][1] = b
        else:
            runs.append([a, b])
    return runs


# ==============================================================================
# 5. ĐỘ ĐO BẰNG CHỨNG (F1 MỨC TỪ & F1 THEO Ý) & TIÊU CHÍ CHỌN CHECKPOINT
# ==============================================================================
_CLAUSE_END_RE = re.compile(r"[,.;:!?…]+[\"'”’\)\]\}]*$")

def _word_line_clauses(text, words):
    out, line_id, clause_id, prev_end = [], 0, 0, 0
    for w in words:
        a, b = w[0], w[1]
        gap = text[prev_end:a]
        if "\n" in gap:
            line_id += gap.count("\n")
            clause_id += 1
        out.append((line_id, clause_id))
        if _CLAUSE_END_RE.search(text[a:b]):
            clause_id += 1
        prev_end = b
    return out

def f1_theo_y_item(text, words, marks, gold_phrases, margin=2):
    """Độ đo F1 theo ý (docs/v2/DO_DO_BANG_CHUNG.md): ±margin từ trong cùng mệnh đề và cùng dòng."""
    if not gold_phrases or not words:
        return None
    lc = _word_line_clauses(text, words)
    lines = text.split("\n")
    line_starts, pos = [], 0
    for ln in lines:
        line_starts.append(pos)
        pos += len(ln) + 1
    gold_groups, allowed = [], set()
    for ph in dict.fromkeys(p for p in gold_phrases if p):
        phrase_sets = []
        for l_idx, ln in enumerate(lines):
            base = line_starts[l_idx]
            for m in re.finditer("(?=" + re.escape(ph) + ")", ln):
                sa, sb = base + m.start(), base + m.start() + len(ph)
                g = [i for i, w in enumerate(words) if w[0] < sb and sa < w[1]]
                if not g:
                    continue
                phrase_sets.append(set(g))
                lo, hi = g[0], g[-1]
                allowed.update(g)
                lo_lc, hi_lc = lc[lo], lc[hi]
                idx, step = lo - 1, 0
                while idx >= 0 and step < margin and lc[idx] == lo_lc:
                    allowed.add(idx); idx -= 1; step += 1
                idx, step = hi + 1, 0
                while idx < len(words) and step < margin and lc[idx] == hi_lc:
                    allowed.add(idx); idx += 1; step += 1
        if phrase_sets:
            gold_groups.append(phrase_sets)
    if not gold_groups:
        return None
    pred_set = {i for i, m in enumerate(marks) if m}
    if not pred_set:
        return 0.0
    rec = sum(max(len(g & pred_set) / len(g) for g in g_list) for g_list in gold_groups) / len(gold_groups)
    prec = len(pred_set & allowed) / len(pred_set)
    return 2.0 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0

def evidence_stats(
    pred, threshold=0.5, o_bias=0.0, crf_np=None, gate=True, mode="marginal",
    single_word_threshold=None, bridge_ratio=0.0, fallback_min_prob=None, sample_weights=None,
):
    tp = fp = fn = safe_mark = safe_count = 0.0
    harm_pred = harm_empty = 0.0
    y_f1_sum = y_f1_w = 0.0
    for idx_i, item in enumerate(pred):
        wi = 1.0 if sample_weights is None else float(sample_weights[idx_i])
        marks = select_word_marks(
            item, threshold=threshold, o_bias=o_bias, crf_np=crf_np, gate=gate, mode=mode,
            single_word_threshold=single_word_threshold, bridge_ratio=bridge_ratio,
            fallback_min_prob=fallback_min_prob,
        )
        if item["gold"] == 0:
            safe_count += wi
            safe_mark += wi * int(any(marks))
        else:
            yi = f1_theo_y_item(item["text"], item["words"], marks, item.get("gold_phrases") or [])
            if yi is not None:
                y_f1_sum += wi * yi
                y_f1_w += wi
        if int(np.argmax(item["logits"])) != 0:
            harm_pred += wi
            harm_empty += wi * int(not any(marks))
        for marked, w in zip(marks, item["words"]):
            gold = w[3]
            if gold < 0:
                continue
            tp += wi * int(marked and gold == 1)
            fp += wi * int(marked and gold == 0)
            fn += wi * int(not marked and gold == 1)
    return dict(
        precision=float(tp / max(tp + fp, 1e-12)),
        recall=float(tp / max(tp + fn, 1e-12)),
        f1=float(2 * tp / max(2 * tp + fp + fn, 1e-12)),
        f1_theo_y=float(y_f1_sum / max(y_f1_w, 1e-12)) if y_f1_w > 0 else 0.0,
        safe_mark_rate=float(safe_mark / max(safe_count, 1e-12)),
        word_length_ratio=float((tp + fp) / max(tp + fn, 1e-12)),
        harm_empty_evidence_rate=float(harm_empty / max(harm_pred, 1e-12)),
    )

def macro(pred):
    return f1_score(
        [p["gold"] for p in pred],
        [int(np.argmax(p["logits"])) for p in pred],
        labels=range(len(LABELS)),
        average="macro",
        zero_division=0,
    )

def slice_summary(pred, records):
    y = np.array([p["gold"] for p in pred])
    yp = np.array([int(np.argmax(p["logits"])) for p in pred])
    out = {}
    for src in ("that", "sinh"):
        ix = [i for i, r in enumerate(records) if (r.get("meta") or {}).get("loai") == src]
        if not ix:
            continue
        present = np.unique(y[ix])
        out[f"{src}_macro_f1"] = float(f1_score(y[ix], yp[ix], average="macro", labels=present, zero_division=0))
        if src == "that":
            out["that_acc"] = float((y[ix] == yp[ix]).mean())
            for lbl in ("QUAY_ROI", "DE_DOA"):
                j = LABELS.index(lbl)
                sub = [i for i in ix if y[i] == j]
                out[f"that_{lbl.lower()}_recall"] = float((yp[sub] == j).mean()) if sub else None
        else:
            ix_any = [i for i in ix if (records[i].get("meta") or {}).get("do_lo") == "an_y"]
            if ix_any:
                p_any = np.unique(y[ix_any])
                out["sinh_an_y_macro_f1"] = float(f1_score(y[ix_any], yp[ix_any], average="macro", labels=p_any, zero_division=0))
    return out

def checkpoint_select_key(macro_f1, slices_dict, ev_f1, span_weight=SPAN_WEIGHT, mode=CKPT_SELECT_MODE):
    if mode not in {"real_aligned", "macro_only"}:
        raise ValueError(f"CKPT_SELECT_MODE không hợp lệ: {mode!r}")
    m_val = float(macro_f1)
    ev_val = float(ev_f1) if span_weight > 0 else 0.0
    if mode == "macro_only":
        return (m_val, ev_val)
    sl = slices_dict or {}
    that_f1 = float(sl["that_macro_f1"]) if sl.get("that_macro_f1") is not None else m_val
    that_acc = float(sl["that_acc"]) if sl.get("that_acc") is not None else m_val
    if span_weight > 0:
        score = 0.50 * m_val + 0.15 * that_f1 + 0.25 * that_acc + 0.10 * ev_val
    else:
        score = 0.55 * m_val + 0.20 * that_f1 + 0.25 * that_acc
    return (round(score, 6), round(m_val, 6), round(that_acc, 6), round(that_f1, 6), round(ev_val, 6))


# ==============================================================================
# 6. LƯU / KHÔI PHỤC CHECKPOINT & DỌN SẠCH BỘ NHỚ GPU
# ==============================================================================
def trainable(net):
    return {k: p.detach().cpu().clone().contiguous() for k, p in net.named_parameters() if p.requires_grad}

def restore(net, state):
    params = {k: p for k, p in net.named_parameters() if p.requires_grad}
    assert set(params) == set(state)
    with torch.no_grad():
        for key, p in params.items():
            p.copy_(state[key])

def free_training_memory(net, ns):
    """Giải phóng hoàn toàn optimizer states, scaler, gradient và biến tạm khỏi GPU trước khi đo benchmark()."""
    net.zero_grad(set_to_none=True)
    for var_name in (
        "opt", "opt_groups", "scheduler", "scaler", "all_trainable_params",
        "emb_weight", "dev_subs", "sub_batches", "window", "b", "sub",
        "running", "window_loss", "loss", "adv_loss", "g_grad", "active_ids",
        "g_rows", "r_rows", "fit_items", "loader",
    ):
        obj = ns.pop(var_name, None)
        if obj is not None and hasattr(obj, "zero_grad"):
            try:
                obj.zero_grad(set_to_none=True)
            except Exception:
                pass
        del obj
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.synchronize(DEVICE)
        torch.cuda.empty_cache()

opt_groups = build_llrd_param_groups(model)
opt = torch.optim.AdamW(opt_groups)
all_trainable_params = [p for g in opt_groups for p in g["params"]]

accented_fit_idx = [i for i, r in enumerate(parts["fit"]) if strip_accents(r["text"]) != r["text"]]
accent_aug_count = min(len(accented_fit_idx), round(len(parts["fit"]) * ACCENT_AUG))
surface_aug_count = round(len(parts["fit"]) * SURFACE_AUG)


# ==============================================================================
# 7. VÒNG LẶP HUẤN LUYỆN, LƯU CHECKPOINT & GIẢI PHÓNG VRAM
# ==============================================================================
def build_epoch_items(epoch_rng):
    n_fit = len(parts["fit"])
    acc_pick = set(epoch_rng.sample(accented_fit_idx, accent_aug_count)) if accent_aug_count else set()
    rem_pool = [i for i in range(n_fit) if i not in acc_pick]
    if len(rem_pool) >= surface_aug_count:
        surf_pick = set(epoch_rng.sample(rem_pool, surface_aug_count)) if surface_aug_count else set()
    else:
        surf_pick = set(epoch_rng.sample(range(n_fit), surface_aug_count)) if surface_aug_count else set()

    items = []
    for i, e_clean in enumerate(encoded["fit"]):
        if i in acc_pick:
            e_pert = prepare(augment_accents(parts["fit"][i], epoch_rng))
            items.append((e_clean, e_pert))
            if i in surf_pick:
                items.append(prepare(augment_surface(parts["fit"][i], epoch_rng)))
        elif i in surf_pick:
            e_pert = prepare(augment_surface(parts["fit"][i], epoch_rng))
            items.append((e_clean, e_pert))
        else:
            items.append(e_clean)
    return items

probe_items = build_epoch_items(random.Random(SEED + 1000))
steps_epoch = math.ceil(math.ceil(len(probe_items) / BATCH) / ACCUM)
del probe_items
steps = steps_epoch * EPOCHS
warm = max(1, round(steps * 0.10))

def lr_factor(i):
    if i < warm:
        return (i + 1) / warm
    progress = min(1.0, (i - warm) / max(1, steps - warm))
    return 0.05 + 0.95 * 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(opt, lr_factor)
scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda" and DTYPE == torch.float16)

last = RUN / "last.pt"
start_epoch, history, best_state, best_key = 0, [], {}, (-1.0,)
if last.exists():
    saved = torch.load(last, map_location="cpu", weights_only=True)
    assert saved["config"] == config
    start_epoch, history = saved["epoch"] + 1, saved["history"]
    best_state, best_key = saved["best"], tuple(saved["key"])
    if start_epoch < EPOCHS:
        restore(model, saved["trainable"])
        opt.load_state_dict(saved["optimizer"])
        scheduler.load_state_dict(saved["scheduler"])
        scaler.load_state_dict(saved["scaler"])
    else:
        print(f"Đã huấn luyện đủ {EPOCHS} epoch trong {last}; khôi phục trực tiếp best_state (bỏ qua nạp optimizer states lên GPU).")
    del saved

emb_weight = model.backbone.get_input_embeddings().weight
has_dev = "dev" in encoded
dev_records = [e["row"] for e in encoded["dev"]] if has_dev else []

for epoch in range(start_epoch, EPOCHS):
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats(DEVICE)
    epoch_start = time.perf_counter()
    seed_all(SEED + 100 + epoch)
    epoch_rng = random.Random(SEED + 1000 + epoch)
    fit_items = build_epoch_items(epoch_rng)
    loader = batches(fit_items, shuffle=True, epoch=epoch)
    model.train()
    opt.zero_grad(set_to_none=True)
    running = torch.zeros((), device=DEVICE)
    use_fgm_now = FGM_EPS > 0 and (epoch >= 1 or EPOCHS == 1)

    for step, window in enumerate(accumulation_windows(loader, ACCUM), start=1):
        sub_batches = [sb for batch in window for sb in split_microbatch(batch)]
        class_mass = sum(sample_weights(b).sum() for b in sub_batches).clamp_min(1e-6)
        pair_count = max(sum(len(b["pairs"]) for b in sub_batches), 1)
        window_loss = torch.zeros((), device=DEVICE)
        dev_subs = []
        for sub in sub_batches:
            b = to_device(sub, DEVICE)
            if use_fgm_now:
                dev_subs.append(b)
            with amp():
                loss = objective(model, model(b, with_aux=True), b, class_mass, use_consistency=True, pair_count=pair_count)
            scaler.scale(loss).backward()
            window_loss += loss.detach()

        if use_fgm_now:
            _fgm_adversarial_step(model, emb_weight, dev_subs, class_mass, scaler)
        dev_subs.clear()

        if not torch.isfinite(window_loss):
            raise FloatingPointError(f"Loss lỗi ở epoch {epoch + 1}, bước {step}")
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(all_trainable_params, 1.0)
        old_scale = scaler.get_scale()
        scaler.step(opt)
        scaler.update()
        if scaler.get_scale() >= old_scale:
            scheduler.step()
        opt.zero_grad(set_to_none=True)
        running += window_loss
        if step % 40 == 0 or step == steps_epoch:
            print(f"epoch {epoch + 1}/{EPOCHS} | step {step}/{steps_epoch} | loss {running.item() / step:.4f}", flush=True)

    if has_dev:
        dev = infer(model, encoded["dev"])
        crf_np_ep = get_crf_params_np(model)
        if SPAN_WEIGHT:
            ev_m = evidence_stats(dev, threshold=0.5, mode="marginal")
            ev_c = evidence_stats(dev, o_bias=0.0, crf_np=crf_np_ep, mode="crf") if USE_BIO_CRF else ev_m
            ev_sel = ev_m if ev_m["f1"] >= ev_c["f1"] else ev_c
        else:
            ev_m = ev_c = ev_sel = dict(precision=0.0, recall=0.0, f1=0.0, f1_theo_y=0.0, safe_mark_rate=0.0)
        m_f1 = float(macro(dev))
        sl = slice_summary(dev, dev_records)
        key = checkpoint_select_key(m_f1, sl, ev_sel["f1"], span_weight=SPAN_WEIGHT, mode=CKPT_SELECT_MODE)
        if key > best_key:
            best_key, best_state = key, trainable(model)
        entry = dict(
            epoch=epoch + 1,
            train_loss=running.item() / max(steps_epoch, 1),
            select_score=key[0],
            macro_f1=m_f1,
            **sl,
            evidence_f1=float(ev_sel["f1"]),
            evidence_marginal_f1=float(ev_m["f1"]),
            evidence_crf_f1=float(ev_c["f1"]),
            evidence_f1_theo_y=float(ev_sel["f1_theo_y"]),
            evidence_precision=float(ev_sel["precision"]),
            evidence_recall=float(ev_sel["recall"]),
            safe_mark_rate=float(ev_sel["safe_mark_rate"]),
            orc_blend=(float(torch.sigmoid(model.orc_blend_logit).item()) if USE_ORC else 0.0),
            seconds=time.perf_counter() - epoch_start,
            peak_cuda_MiB=(torch.cuda.max_memory_allocated(DEVICE) / 2**20 if DEVICE.type == "cuda" else None),
        )
    else:
        # Chế độ final (final_90_10 / final_100_human_cal): KHÔNG nhìn cal để chọn epoch; lấy trọng số cố định sau đủ EPOCHS
        best_key = (float(epoch + 1),)
        best_state = trainable(model)
        entry = dict(
            epoch=epoch + 1,
            train_loss=running.item() / max(steps_epoch, 1),
            mode=f"fixed_epoch_{DATA_SCHEME}",
            orc_blend=(float(torch.sigmoid(model.orc_blend_logit).item()) if USE_ORC else 0.0),
            seconds=time.perf_counter() - epoch_start,
            peak_cuda_MiB=(torch.cuda.max_memory_allocated(DEVICE) / 2**20 if DEVICE.type == "cuda" else None),
        )
    history.append(entry)
    tmp = last.with_suffix(".tmp")
    torch.save(
        dict(
            config=config,
            epoch=epoch,
            trainable=trainable(model),
            optimizer=opt.state_dict(),
            scheduler=scheduler.state_dict(),
            scaler=scaler.state_dict(),
            history=history,
            best=best_state,
            key=best_key,
        ),
        tmp,
    )
    os.replace(tmp, last)
    print(entry, flush=True)

restore(model, best_state)
tmp = RUN / "model_best.tmp"
save_file(best_state, str(tmp))
os.replace(tmp, RUN / "model_best.safetensors")
write_json(RUN / "history.json", history)
free_training_memory(model, globals())
print(pd.DataFrame(history))
if DEVICE.type == "cuda":
    print(f"Sau khi giải phóng optimizer/FGM, VRAM đang giữ: {torch.cuda.memory_allocated(DEVICE) / 2**20:.1f} MiB")

---
<a id="sec-6"></a>
## 6. Hiệu chuẩn xác suất, Đánh giá phát triển & Cổng khóa Bộ Test (`C0 / C1 / C2`)

Ô này thực thi hai bước tách biệt rõ ràng về mặt khoa học: **(i) Hiệu chuẩn & đóng băng ngưỡng trên `cal` rồi báo cáo `dev`**, và **(ii) Cổng khóa cứng 1 lần duy nhất cho Bộ Test người viết (`1.670` mẫu)**.

| Giai đoạn | Quy tắc thực thi & Tệp xuất |
| :--- | :--- |
| **1. Hiệu chuẩn sát phân phối thật (`cal` / `HUMAN_CAL_PATH`)** | Nếu có `HUMAN_CAL_PATH`, hiệu chuẩn trực tiếp trên validation người gán. Nếu dùng `cal` nội bộ (`fold 1`, 1.339 đoạn), nhiệt độ `T` dùng 70% câu thật + 30% câu sinh khó (`cap = 3.0`, biên `T >= 0.74` trên `internal_llm`); ngưỡng bằng chứng cân bằng 5 lớp. Lưu đóng băng vào `calibration.json`. |
| **2. Giải mã bằng chứng chống phình cụm** | So sánh trực tiếp trên `cal` giữa **CRF Forward-Backward Marginal + hậu xử lý** và **Viterbi (`o_bias >= -0.15`)**, phạt phình độ dài (`word_length_ratio > 1.25`), kèm `fallback_min_prob`; báo song song F1 mức từ & **F1 theo ý** ([`DO_DO_BANG_CHUNG.md`](../docs/v2/DO_DO_BANG_CHUNG.md)). |
| **3. Báo cáo `dev` & Biểu đồ chẩn đoán** | Xuất `dev_metrics.json` (kèm Bootstrap 95% CI, `FPR(AN_TOAN)`, `worst_group_region_macro_f1`), `dev_predictions.jsonl` (đủ `probs` 5 lớp) và `dev_diagnostics.png`. |

> **CẢNH BÁO KHÓA TEST (`REAL_TEST_LOCK.json`):** Mặc định `EVALUATE_REAL_TEST = False`. Chỉ đặt `True` **đúng 1 lần duy nhất** sau khi hoàn tất lựa chọn cấu hình trên `dev` và đóng băng `calibration.json` (tự xác minh SHA-256 `calibration.json`, không hiệu chỉnh lại ngưỡng). Khi mở khóa, hệ thống chấm đủ `334 C0 + 668 C1 + 668 C2 = 1.670` mẫu cho cả **Dòng `#5` (thô)** và **Dòng `#6` (+ chuẩn hóa giữ dòng)**, xuất `real_test_all_predictions.jsonl` / `real_test_norm_all_predictions.jsonl` (tương thích 100% `python -m src.v2.eval.evaluate`), ma trận 15 lát cắt `real_test_official_matrix.json` (có gắn minh bạch `seen_in_train_aug` & `strictly_unseen`), và tự động khóa `REAL_TEST_LOCK.json`.

In [ ]:
#@title 6.1. Hiệu chuẩn xác suất/ngưỡng (cal), Đánh giá dev & Chấm Test khóa 1 lần cuối
# ==============================================================================
# 1. CHỐNG RÒ RỈ DỮ LIỆU & TRỌNG SỐ HIỆU CHUẨN SÁT PHÂN PHỐI THẬT
# ==============================================================================
def _norm_dedup_text(s):
    return re.sub(r"[\W_]+", " ", strip_accents(s).lower()).strip()

def read_human(path):
    raw_bytes = Path(path).read_bytes()
    for locked_name in ("test.jsonl", "test_day_du.jsonl", "test_perturb_C1.jsonl", "test_perturb_C2.jsonl"):
        if _matches_blob_sha(raw_bytes, DATA_SHA256[locked_name]):
            raise ValueError(f"KHÓA AN TOÀN: HUMAN_CAL_PATH trùng SHA-256 của tập test thật ({locked_name}) — từ chối để chống rò rỉ!")
    for train_name in ("so_sanh.jsonl", "train.jsonl"):
        if _matches_blob_sha(raw_bytes, DATA_SHA256[train_name]):
            raise ValueError(f"KHÓA AN TOÀN: HUMAN_CAL_PATH trùng SHA-256 của tập train ({train_name}) — từ chối để chống rò rỉ!")
    gold = [json.loads(line) for line in raw_bytes.decode("utf-8-sig").splitlines() if line.strip()]
    training_texts = {r["text"].strip().casefold() for r in rows}
    training_norms = {nt for nt in (_norm_dedup_text(r["text"]) for r in rows) if nt}
    training_ids = {r["id"] for r in rows}
    assert gold and all(isinstance(r, dict) and r.get("id") for r in gold) and len({r["id"] for r in gold}) == len(gold), (
        "HUMAN_CAL_PATH rỗng, sai cấu trúc hoặc trùng id!"
    )
    seen_texts = set()
    for r in gold:
        assert isinstance(r.get("text"), str) and r["text"].strip(), f"Thiếu hoặc rỗng text tại id={r['id']}"
        cf = r["text"].strip().casefold()
        nt = _norm_dedup_text(r["text"])
        assert r["id"] not in training_ids and cf not in training_texts and (not nt or nt not in training_norms), (
            f"Rò rỉ dữ liệu train vào HUMAN_CAL_PATH tại id={r['id']}"
        )
        assert cf not in seen_texts, f"Trùng lặp nội dung bên trong HUMAN_CAL_PATH tại id={r['id']}"
        seen_texts.add(cf)
        bc = r.get("bang_chung")
        assert isinstance(bc, list) and all(isinstance(p, str) and p and "\n" not in p and any(p in ln for ln in r["text"].split("\n")) for p in bc), (
            f"Trường bang_chung không hợp lệ tại id={r['id']}"
        )
        assert r["nhan"] in LABELS
        assert r["nhan"] != "AN_TOAN" or not bc
        assert r.get("nguon") != "sinh_llm" and r.get("cach_gan") != "llm"
        r["so_dong"] = len(r["text"].split("\n"))
        assert 1 <= r["so_dong"] <= 8
        r.setdefault("nguon", "nguoi")
        if not r.get("nhom_trung"):
            r["nhom_trung"] = r["id"]
        meta = dict(r.get("meta") or {})
        meta.update({"loai": "that", "nguon": r["nguon"]})
        r["meta"] = meta
    return gold

def compute_cal_weights(records, mode=CAL_WEIGHT_MODE):
    if mode != "real_aligned" or not records:
        return np.ones(len(records), dtype=np.float64)
    raw = []
    for r in records:
        m = r.get("meta") or {}
        if m.get("loai", "that") == "that":
            raw.append(3.0)
        elif m.get("do_lo") == "an_y":
            raw.append(1.25)
        elif m.get("sinh_doi"):
            raw.append(0.85)
        else:
            raw.append(0.50)
    raw = np.asarray(raw, dtype=np.float64)
    labels = [r["nhan"] for r in records]
    present = sorted(set(labels))
    out = np.zeros_like(raw)
    for lbl in present:
        ix = np.array([i for i, y_l in enumerate(labels) if y_l == lbl], dtype=int)
        sub = raw[ix]
        out[ix] = (sub / max(sub.sum(), 1e-12)) / len(present)
    out = out * len(records)
    for _ in range(3):
        out = np.minimum(out, 4.0)
        out = out / max(out.mean(), 1e-12)
    return np.minimum(out, 4.0)

def compute_temp_cal_weights(records, mode=CAL_WEIGHT_MODE, that_share=0.70):
    """Trọng số hiệu chuẩn nhiệt độ T & UCCI: giữ ~70% tổng khối lượng cho câu thật (tránh 57% câu sinh dễ kéo T về 1)
    và 30% cho câu sinh khó (an_y > sinh_doi > lo_lieu) để vẫn đại diện đủ 5 nhãn gồm LUA_DAO."""
    w_base = compute_cal_weights(records, mode=mode)
    if mode != "real_aligned" or not records:
        return w_base
    is_that = np.array([(r.get("meta") or {}).get("loai", "that") == "that" for r in records], dtype=bool)
    if is_that.any() and (~is_that).any():
        out = w_base.copy()
        out[is_that] = out[is_that] / max(out[is_that].sum(), 1e-12) * float(that_share) * len(records)
        out[~is_that] = out[~is_that] / max(out[~is_that].sum(), 1e-12) * (1.0 - float(that_share)) * len(records)
        for _ in range(3):
            out = np.minimum(out, 3.0)
            out = out / max(out.mean(), 1e-12)
        return np.minimum(out, 3.0)
    return w_base

human_cal = read_human(HUMAN_CAL_PATH) if HUMAN_CAL_PATH else []
cal_encoded = [prepare(r) for r in human_cal] if human_cal else encoded["cal"]
cal_records = [e["row"] for e in cal_encoded]
cal = infer(model, cal_encoded)
cal_source = "human" if human_cal else "internal_llm"
cal_ev_weights = np.ones(len(cal_records), dtype=np.float64) if human_cal else compute_cal_weights(cal_records, mode=CAL_WEIGHT_MODE)
cal_temp_weights = np.ones(len(cal_records), dtype=np.float64) if human_cal else compute_temp_cal_weights(cal_records, mode=CAL_WEIGHT_MODE, that_share=0.70)
cal_weights = cal_temp_weights
if not human_cal:
    that_pct = 100.0 * sum(w for r, w in zip(cal_records, cal_temp_weights) if (r.get("meta") or {}).get("loai") == "that") / max(cal_temp_weights.sum(), 1e-12)
    print(f"Hiệu chuẩn trên cal nội bộ ({len(cal_records)} mẫu) [{CAL_WEIGHT_MODE}]: T dùng {that_pct:.1f}% câu thật; bằng chứng cân bằng 5 lớp.")

def arrays(pred):
    return np.array([p["logits"] for p in pred]), np.array([p["gold"] for p in pred])

def _ev_rank_key(st):
    # Phạt cụm dự đoán vượt quá 1.25x độ dài nhãn chuẩn để tránh tô tràn
    bloat_pen = max(0.0, st["word_length_ratio"] - 1.25) * 0.08
    return (round(st["f1"] - bloat_pen, 4), round(st["precision"], 4), -round(st["safe_mark_rate"], 4))

def _fb_rank_key(st, base_safe_mark):
    # Cho phép giảm trắng bằng chứng nhưng phạt phình cụm và phạt tăng tô oan AN_TOAN
    bloat_pen = max(0.0, st["word_length_ratio"] - 1.25) * 0.08
    safe_pen = max(0.0, st["safe_mark_rate"] - base_safe_mark) * 0.25
    empty_pen = st["harm_empty_evidence_rate"] * 0.04
    score = st["f1"] - bloat_pen - safe_pen - empty_pen
    return (round(score, 4), round(st["f1_theo_y"], 4), round(st["precision"], 4), -round(st["safe_mark_rate"], 4))

crf_np = get_crf_params_np(model)

# ==============================================================================
# 2. KHỚP HOẶC NẠP NGUYÊN VẸN CALIBRATION.JSON (NHIỆT ĐỘ T & GIẢI MÃ BẰNG CHỨNG)
# ==============================================================================
CAL_PATH = RUN / "calibration.json"
TEST_LOCK_PATH = RUN / "REAL_TEST_LOCK.json"
expected_cal_mode = "human_uniform" if cal_source == "human" else CAL_WEIGHT_MODE
expected_cal_ids = [p["id"] for p in cal]

if TEST_LOCK_PATH.exists() and not CAL_PATH.exists():
    raise AssertionError("CẢNH BÁO: REAL_TEST_LOCK.json đã tồn tại nhưng thiếu calibration.json đã đóng băng — từ chối hiệu chuẩn lại!")
if CAL_PATH.exists() and (EVALUATE_REAL_TEST or TEST_LOCK_PATH.exists()):
    cal_payload = json.loads(CAL_PATH.read_text(encoding="utf8"))
    cal_sha256 = digest(CAL_PATH.read_bytes())
    assert cal_payload.get("ids") == expected_cal_ids, "calibration.json đã đóng băng không khớp danh sách ID tập cal!"
    assert cal_payload.get("source") == cal_source and cal_payload.get("cal_weight_mode") == expected_cal_mode, "calibration.json đã đóng băng lệch nguồn/chế độ hiệu chuẩn!"
    if TEST_LOCK_PATH.exists():
        lock_meta = json.loads(TEST_LOCK_PATH.read_text(encoding="utf8"))
        assert lock_meta.get("frozen_calibration_sha256") == cal_sha256, "CẢNH BÁO: SHA-256 calibration.json không khớp REAL_TEST_LOCK.json!"
    temp = float(cal_payload["temperature"])
    temp_unweighted = float(cal_payload.get("temperature_unweighted", temp))
    chosen_decoder = cal_payload["chosen_decoder"]
    o_bias = float(cal_payload["o_bias"])
    base_marginal_threshold = float(cal_payload["base_marginal_threshold"])
    marginal_threshold = float(cal_payload["marginal_threshold"])
    single_word_threshold = float(cal_payload["single_word_threshold"])
    bridge_ratio = float(cal_payload["bridge_ratio"])
    fallback_min_prob = float(cal_payload["fallback_min_prob"])
    print(f"ĐÃ NẠP NGUYÊN VẸN calibration.json ĐÓNG BĂNG (không ghi đè, sha={cal_sha256[:12]}).")
else:
    z, y = arrays(cal)
    def nll_temp(log_t, sample_weight=None):
        return log_loss(y, softmax(z / np.exp(log_t), axis=1), labels=range(len(LABELS)), sample_weight=sample_weight)

    # 1) Nhiệt độ không trọng số (để đối chiếu ablation)
    cal_fit_unw = minimize_scalar(lambda lt: nll_temp(lt, None), bounds=(-2.3, 2.3), method="bounded")
    temp_unweighted = float(np.exp(cal_fit_unw.x)) if (cal_fit_unw.success and cal_fit_unw.fun < nll_temp(0.0, None)) else 1.0

    # 2) Nhiệt độ hiệu chuẩn sát phân phối thật (70% câu thật, cap=3.0, biên T >= 0.74 cho Focal + Label Smoothing)
    t_bounds = (-2.3, 2.3) if cal_source == "human" else (-0.30, 2.3)
    cal_fit = minimize_scalar(lambda lt: nll_temp(lt, cal_temp_weights), bounds=t_bounds, method="bounded")
    if not cal_fit.success or not np.isfinite([cal_fit.x, cal_fit.fun]).all():
        raise RuntimeError(f"Hiệu chuẩn nhiệt độ thất bại: {cal_fit.message}")
    temp = float(np.exp(cal_fit.x)) if cal_fit.fun < nll_temp(0.0, cal_temp_weights) else 1.0

    if SPAN_WEIGHT > 0:
        # A) Dò ngưỡng xác suất biên (CRF Forward-Backward marginals) + hậu xử lý + fallback
        coarse_t = np.linspace(0.15, 0.85, 29)
        t0 = float(max(coarse_t, key=lambda t: (
            _ev_rank_key(evidence_stats(cal, threshold=t, mode="marginal", sample_weights=cal_ev_weights)), -abs(t - 0.5)
        )))
        fine_ts = sorted({round(float(np.clip(t0 + d, 0.125, 0.90)), 3) for d in (-0.05, -0.025, 0.0, 0.025, 0.05)})
        base_marginal_threshold = float(max(fine_ts, key=lambda t: (
            _ev_rank_key(evidence_stats(cal, threshold=t, mode="marginal", sample_weights=cal_ev_weights)), -abs(t - 0.5)
        )))
        best_m_cfg, best_m_key = None, None
        for t in fine_ts:
            for sw_t in (t, round(min(0.95, t + 0.10), 3), round(min(0.95, t + 0.18), 3)):
                for br in (0.0, 0.75):
                    st = evidence_stats(cal, threshold=t, mode="marginal", single_word_threshold=sw_t, bridge_ratio=br, sample_weights=cal_ev_weights)
                    rk = _ev_rank_key(st)
                    if best_m_key is None or rk > best_m_key:
                        best_m_key, best_m_cfg = rk, (t, sw_t, br)
        marginal_threshold, single_word_threshold, bridge_ratio = best_m_cfg
        st_m_nofb = evidence_stats(
            cal, threshold=marginal_threshold, mode="marginal",
            single_word_threshold=single_word_threshold, bridge_ratio=bridge_ratio,
            fallback_min_prob=1.01, sample_weights=cal_ev_weights,
        )
        fb_grid = (0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 1.01)
        best_m_fb, best_m_fb_key = 1.01, None
        for fb in fb_grid:
            st = evidence_stats(
                cal, threshold=marginal_threshold, mode="marginal",
                single_word_threshold=single_word_threshold, bridge_ratio=bridge_ratio,
                fallback_min_prob=fb, sample_weights=cal_ev_weights,
            )
            rk = _fb_rank_key(st, st_m_nofb["safe_mark_rate"])
            if best_m_fb_key is None or rk > best_m_fb_key:
                best_m_fb_key, best_m_fb = rk, fb
        st_m_best = evidence_stats(
            cal, threshold=marginal_threshold, mode="marginal",
            single_word_threshold=single_word_threshold, bridge_ratio=bridge_ratio,
            fallback_min_prob=best_m_fb, sample_weights=cal_ev_weights,
        )

        # B) Nếu dùng BIO-CRF, dò Viterbi có giới hạn chống phình cụm (o_bias >= -0.15, bao gồm 0.0)
        if USE_BIO_CRF:
            coarse_b = np.linspace(-0.15, 2.35, 51)
            b0 = float(max(coarse_b, key=lambda ob: (
                _ev_rank_key(evidence_stats(cal, o_bias=ob, crf_np=crf_np, mode="crf", sample_weights=cal_ev_weights)), -abs(ob)
            )))
            fine_b = sorted({round(float(max(-0.15, b0 + d)), 3) for d in (-0.10, -0.05, 0.0, 0.05, 0.10)})
            o_bias = float(max(fine_b, key=lambda ob: _ev_rank_key(
                evidence_stats(cal, o_bias=ob, crf_np=crf_np, mode="crf", sample_weights=cal_ev_weights)
            )))
            st_c_nofb = evidence_stats(cal, o_bias=o_bias, crf_np=crf_np, mode="crf", fallback_min_prob=1.01, sample_weights=cal_ev_weights)
            best_c_fb, best_c_fb_key = 1.01, None
            for fb in fb_grid:
                st = evidence_stats(cal, o_bias=o_bias, crf_np=crf_np, mode="crf", fallback_min_prob=fb, sample_weights=cal_ev_weights)
                rk = _fb_rank_key(st, st_c_nofb["safe_mark_rate"])
                if best_c_fb_key is None or rk > best_c_fb_key:
                    best_c_fb_key, best_c_fb = rk, fb
            st_c_best = evidence_stats(cal, o_bias=o_bias, crf_np=crf_np, mode="crf", fallback_min_prob=best_c_fb, sample_weights=cal_ev_weights)
        else:
            o_bias, best_c_fb, st_c_best = 0.0, 1.01, st_m_best

        if EVIDENCE_DECODE_MODE == "marginal" or not USE_BIO_CRF:
            chosen_decoder, fallback_min_prob = "marginal", best_m_fb
        elif EVIDENCE_DECODE_MODE == "crf":
            chosen_decoder, fallback_min_prob = "crf", best_c_fb
        else:
            if _ev_rank_key(st_c_best) > _ev_rank_key(st_m_best):
                chosen_decoder, fallback_min_prob = "crf", best_c_fb
            else:
                chosen_decoder, fallback_min_prob = "marginal", best_m_fb
    else:
        chosen_decoder = "marginal"
        base_marginal_threshold = marginal_threshold = single_word_threshold = 1.1
        bridge_ratio, o_bias, fallback_min_prob = 0.0, 0.0, 1.01

    cal_payload = dict(
        temperature=temp,
        temperature_unweighted=temp_unweighted,
        cal_weight_mode=expected_cal_mode,
        temp_cal_that_share=round(float(sum(w for r, w in zip(cal_records, cal_temp_weights) if (r.get("meta") or {}).get("loai", "that") == "that") / max(cal_temp_weights.sum(), 1e-12)), 4),
        chosen_decoder=chosen_decoder,
        use_bio_crf=USE_BIO_CRF,
        o_bias=o_bias,
        base_marginal_threshold=base_marginal_threshold,
        marginal_threshold=marginal_threshold,
        single_word_threshold=single_word_threshold,
        bridge_ratio=bridge_ratio,
        fallback_min_prob=fallback_min_prob,
        nll_weighted_before=float(nll_temp(0.0, cal_temp_weights)),
        nll_weighted_after=float(nll_temp(np.log(temp), cal_temp_weights)),
        nll_unweighted_before=float(nll_temp(0.0, None)),
        nll_unweighted_after=float(nll_temp(np.log(temp), None)),
        source=cal_source,
        ids=expected_cal_ids,
    )
    write_json(CAL_PATH, cal_payload)
    cal_sha256 = digest(CAL_PATH.read_bytes())
print(
    f"Hiệu chuẩn [{cal_payload['cal_weight_mode']}]: T={temp:.4f} (unw={temp_unweighted:.4f}) | "
    f"decoder={chosen_decoder} | marg=({marginal_threshold:.3f}, sw={single_word_threshold:.3f}, br={bridge_ratio}) | "
    f"crf o_bias={o_bias:.3f} | fallback={fallback_min_prob:.2f}"
)


# ==============================================================================
# 3. HÀM ĐÁNH GIÁ TỔNG HỢP (5 NHÃN, LÁT CẮT, BOOTSTRAP 95% CI) & BIỂU ĐỒ
# ==============================================================================
def calibration_stats(prob, y_true):
    confidence = prob.max(1)
    correct = prob.argmax(1) == y_true
    ece = 0.0
    for mask in (np.minimum((confidence * 10).astype(int), 9) == i for i in range(10)):
        if mask.any():
            ece += mask.mean() * abs(correct[mask].mean() - confidence[mask].mean())
    return dict(
        nll=float(log_loss(y_true, prob, labels=range(len(LABELS)))),
        ece10=float(ece),
        brier=float(((prob - np.eye(len(LABELS))[y_true]) ** 2).sum(1).mean()),
    )

def format_output(item, text, cal_cfg=None):
    cfg = cal_cfg or cal_payload
    prob = softmax(np.array(item["logits"]) / cfg["temperature"])
    label = int(prob.argmax())
    runs = []
    if label and SPAN_WEIGHT > 0:
        runs = extract_evidence_runs(
            item, text,
            threshold=cfg["marginal_threshold"],
            o_bias=cfg["o_bias"],
            crf_np=crf_np,
            mode=cfg["chosen_decoder"],
            single_word_threshold=cfg["single_word_threshold"],
            bridge_ratio=cfg["bridge_ratio"],
            fallback_min_prob=cfg["fallback_min_prob"],
        )
    return dict(
        nhan=LABELS[label],
        tin_cay=float(prob[label]),
        probs=[round(float(x), 6) for x in prob],
        rui_ro=round(float(item["severity"]) * 100.0, 2),
        bang_chung=[text[a:b] for a, b in runs],
        vi_tri=runs,
    )

def evaluate(pred, records, cal_cfg=None):
    cfg = cal_cfg or cal_payload
    z_e, y_e = arrays(pred)
    yp = z_e.argmax(1)
    ev_kwargs = dict(
        threshold=cfg["marginal_threshold"],
        o_bias=cfg["o_bias"],
        crf_np=crf_np,
        mode=cfg["chosen_decoder"],
        single_word_threshold=cfg["single_word_threshold"],
        bridge_ratio=cfg["bridge_ratio"],
        fallback_min_prob=cfg["fallback_min_prob"],
    )
    safe_mask_e = (y_e == 0)
    result = dict(
        macro_f1=float(macro(pred)),
        accuracy=float((y_e == yp).mean()),
        binary_f1=float(f1_score(y_e != 0, yp != 0, zero_division=0)),
        fpr_an_toan=(float((yp[safe_mask_e] != 0).mean()) if safe_mask_e.any() else 0.0),
        classes=classification_report(y_e, yp, labels=range(5), target_names=LABELS, zero_division=0, output_dict=True),
        confusion=confusion_matrix(y_e, yp, labels=range(5)).tolist(),
        calibration_before=calibration_stats(softmax(z_e, axis=1), y_e),
        calibration_after=calibration_stats(softmax(z_e / cfg["temperature"], axis=1), y_e),
        evidence=evidence_stats(pred, gate=True, **ev_kwargs),
        evidence_marginal_threshold=evidence_stats(pred, threshold=cfg["base_marginal_threshold"], gate=True, mode="marginal"),
        evidence_crf_viterbi=(evidence_stats(pred, o_bias=cfg["o_bias"], crf_np=crf_np, gate=True, mode="crf") if (USE_BIO_CRF and SPAN_WEIGHT > 0) else None),
        evidence_without_gate=evidence_stats(pred, gate=False, **ev_kwargs),
    )
    that_ix = [i for i, r in enumerate(records) if (r.get("meta") or {}).get("loai") == "that"]
    if that_ix:
        result["calibration_real_before"] = calibration_stats(softmax(z_e[that_ix], axis=1), y_e[that_ix])
        result["calibration_real_after"] = calibration_stats(softmax(z_e[that_ix] / cfg["temperature"], axis=1), y_e[that_ix])

    slices = defaultdict(list)
    for i, r in enumerate(records):
        meta = r.get("meta") or {}
        loai = meta.get("loai") or r.get("nguon", "khac")
        keys = [loai, "1 lượt" if r["so_dong"] == 1 else "nhiều lượt", *r.get("the", [])]
        phan_t = r.get("phan_test") or meta.get("phan_test")
        if phan_t:
            keys.append("phan:" + phan_t)
        vung = r.get("giong_vung") or meta.get("giong_vung")
        if not vung and meta.get("giong") in ("giong_bac", "giong_trung", "giong_nam"):
            vung = meta["giong"].removeprefix("giong_")
        if vung:
            keys.append("vung:" + vung)
        do_lo = r.get("do_lo") or meta.get("do_lo")
        if do_lo:
            keys.append("do_lo:" + do_lo)
        kenh = r.get("kenh") or meta.get("kenh")
        if kenh:
            keys.append("kenh:" + kenh)
        if r.get("cach_gan"):
            keys.append("cach_gan:" + r["cach_gan"])
        bd = r.get("bien_dang") or meta.get("bien_dang")
        if bd:
            keys.append("bien_dang:" + bd)
        nbd = r.get("nhom_bien_dang") or meta.get("nhom_bien_dang")
        if nbd:
            keys.append("nhom_bd:" + nbd)
        kbd = r.get("kieu_bien_dang") or meta.get("kieu_bien_dang")
        if kbd:
            keys.append("kieu_bd:" + kbd)
        for key in keys:
            slices[key].append(i)
    result["slices"] = {}
    for key, ix in slices.items():
        present = np.unique(y_e[ix])
        sm_ix = (y_e[ix] == 0)
        result["slices"][key] = dict(
            n=len(ix),
            labels=[LABELS[j] for j in present],
            macro_present=float(f1_score(y_e[ix], yp[ix], average="macro", labels=present, zero_division=0)),
            binary_f1=float(f1_score(y_e[ix] != 0, yp[ix] != 0, zero_division=0)),
            fpr_an_toan=(float((yp[ix][sm_ix] != 0).mean()) if sm_ix.any() else 0.0),
            accuracy=float((y_e[ix] == yp[ix]).mean()),
            evidence=evidence_stats([pred[i] for i in ix], gate=True, **ev_kwargs),
        )
    reg_f1s = [result["slices"][f"vung:{g}"]["macro_present"] for g in ("bac", "trung", "nam") if f"vung:{g}" in result["slices"]]
    if len(reg_f1s) == 3:
        result["worst_group_region_macro_f1"] = float(min(reg_f1s))
        result["region_gap_max_min"] = float(max(reg_f1s) - min(reg_f1s))
    result["recall_nguon_va_nhan"] = {}
    for source in sorted({(r.get("meta") or {}).get("loai") or r.get("nguon", "khac") for r in records}):
        table = {}
        for j, label in enumerate(LABELS):
            ix = [i for i, r in enumerate(records) if ((r.get("meta") or {}).get("loai") or r.get("nguon", "khac")) == source and y_e[i] == j]
            table[label] = {"n": len(ix), "recall": float((yp[ix] == j).mean()) if ix else None}
        result["recall_nguon_va_nhan"][source] = table

    # Macro-F1 trên 4 lớp (AN_TOAN, QUAY_ROI, DE_DOA, LUA_DAO) — chỉ số phụ trợ song song Macro-F1 5 lớp
    main4 = [LABELS.index(l) for l in ("AN_TOAN", "QUAY_ROI", "DE_DOA", "LUA_DAO")]
    result["macro_f1_4main"] = float(f1_score(y_e, yp, labels=main4, average="macro", zero_division=0))

    group_names, inverse = np.unique([split_group(r) for r in records], return_inverse=True)
    matrices = np.zeros((len(group_names), len(LABELS), len(LABELS)), dtype=np.int64)
    np.add.at(matrices, (inverse, y_e, yp), 1)
    rng = np.random.default_rng(SEED)
    scores, scores_4, scores_bin = [], [], []
    for _ in range(500):
        cm = matrices[rng.integers(0, len(matrices), len(matrices))].sum(0)
        per_c = 2 * cm.diagonal() / np.maximum(cm.sum(0) + cm.sum(1), 1)
        scores.append(per_c.mean())
        scores_4.append(per_c[main4].mean())
        tp_b = cm[1:, 1:].sum()
        fp_b = cm[0, 1:].sum()
        fn_b = cm[1:, 0].sum()
        scores_bin.append(2 * tp_b / max(2 * tp_b + fp_b + fn_b, 1))
    result["macro_f1_bootstrap_95"] = np.quantile(scores, [0.025, 0.975]).tolist()
    result["macro_f1_4main_bootstrap_95"] = np.quantile(scores_4, [0.025, 0.975]).tolist()
    result["binary_f1_bootstrap_95"] = np.quantile(scores_bin, [0.025, 0.975]).tolist()
    return result

def save_diagnostics_plot(pred, out_path, cal_cfg=None):
    cfg = cal_cfg or cal_payload
    import matplotlib.pyplot as plt
    from sklearn.metrics import ConfusionMatrixDisplay, precision_recall_curve, average_precision_score
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.8), layout="constrained")
    z_eval, y_eval = arrays(pred)
    ConfusionMatrixDisplay.from_predictions(
        y_eval, z_eval.argmax(1), labels=range(5), display_labels=LABELS,
        xticks_rotation=40, ax=axes[0], colorbar=False, cmap="Blues",
    )
    prob_raw = softmax(z_eval, axis=1)
    prob_eval = softmax(z_eval / cfg["temperature"], axis=1)
    for j, name in enumerate(LABELS[1:], start=1):
        target = y_eval == j
        if target.any():
            precision, recall, _ = precision_recall_curve(target, prob_eval[:, j])
            ap = average_precision_score(target, prob_eval[:, j])
            axes[1].plot(recall, precision, label=f"{name} · AP {ap:.2f}")
    axes[1].set(xlabel="Recall", ylabel="Precision", xlim=(0, 1), ylim=(0, 1.02), title="Precision–recall từng nhãn nguy hại")
    axes[1].legend(fontsize=8)

    def _rel_curve(prob):
        conf, corr = prob.max(1), prob.argmax(1) == y_eval
        xs, ys = [], []
        for i in range(10):
            m = np.minimum((conf * 10).astype(int), 9) == i
            if m.any():
                xs.append(float(conf[m].mean()))
                ys.append(float(corr[m].mean()))
        return xs, ys

    ece_before = calibration_stats(prob_raw, y_eval)["ece10"]
    ece_after = calibration_stats(prob_eval, y_eval)["ece10"]
    axes[2].plot([0, 1], [0, 1], "k--", lw=1, label="Hoàn hảo")
    axes[2].plot(*_rel_curve(prob_raw), "o-", label=f"Trước (ECE {ece_before:.3f})")
    axes[2].plot(*_rel_curve(prob_eval), "s-", label=f"Sau T={cfg['temperature']:.2f} (ECE {ece_after:.3f})")
    axes[2].set(xlabel="Độ tin cậy", ylabel="Độ chính xác", xlim=(0, 1), ylim=(0, 1.02), title="Độ tin cậy (Reliability)")
    axes[2].legend(fontsize=8)
    fig.savefig(out_path, dpi=140)
    plt.show()
    plt.close(fig)


# ==============================================================================
# 4. XUẤT BÁO CÁO TẬP PHÁT TRIỂN (DEV / HUMAN_CAL / AUDIT)
# ==============================================================================
def report_split(key, data_override=None):
    data = encoded[key] if data_override is None else data_override
    records = [e["row"] for e in data]
    pred = infer(model, data)
    result = evaluate(pred, records)
    if key in ("cal", "human_cal"):
        result["diagnostic_split_warning"] = "in_sample_cal_only_do_not_use_for_model_selection"
        print(f"LƯU Ý ({DATA_SCHEME}): Đang xuất chẩn đoán trên {key}; chỉ dùng để kiểm tra ống suy luận, không dùng chọn mô hình.")
    write_json(RUN / f"{key}_metrics.json", result)
    with (RUN / f"{key}_predictions.jsonl").open("w", encoding="utf8") as f:
        for p, r in zip(pred, records):
            record = dict(id=r["id"], nhan_dung=r["nhan"], nguon=r["nguon"], **format_output(p, r["text"]))
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    print(
        key, "macro-F1", round(result["macro_f1"], 4), "95%", [round(x, 4) for x in result["macro_f1_bootstrap_95"]],
        "| Ev F1 (word / theo_y):", round(result["evidence"]["f1"], 4), "/", round(result["evidence"]["f1_theo_y"], 4),
        "| len_ratio:", round(result["evidence"]["word_length_ratio"], 2),
        "| empty_harm:", f"{100*result['evidence']['harm_empty_evidence_rate']:.2f}%",
    )
    print(pd.DataFrame(result["confusion"], index=LABELS, columns=LABELS))
    print(pd.DataFrame(result["classes"]).T)
    print(pd.DataFrame(result["slices"]).T[["n", "macro_present", "accuracy"]])
    return pred

if EVALUATE_AUDIT and "audit" not in encoded:
    print(f"CẢNH BÁO: EVALUATE_AUDIT=True nhưng DATA_SCHEME={DATA_SCHEME!r} không có split audit; bỏ qua audit.")
if EVALUATE_AUDIT and "audit" in encoded:
    report_key, report_encoded = "audit", encoded["audit"]
elif "dev" in encoded:
    report_key, report_encoded = "dev", encoded["dev"]
elif "cal" in encoded:
    report_key, report_encoded = "cal", encoded["cal"]
else:
    report_key, report_encoded = "human_cal", cal_encoded
final_pred = report_split(report_key, data_override=report_encoded)
save_diagnostics_plot(final_pred, RUN / f"{report_key}_diagnostics.png")

# ==============================================================================
# 5. KHÓA CỨNG BỘ TEST ĐÁNH GIÁ (334 C0 + 668 C1 + 668 C2 = 1.670 MẪU, #5 & #6)
# ==============================================================================
def load_locked_real_test():
    test_full = fetch_data("test_day_du.jsonl")
    test_slim = fetch_data("test.jsonl")
    assert len(test_full) == len(test_slim) == 334
    train_ids = {r["id"] for r in rows}
    train_texts = {r["text"].strip().casefold() for r in rows}
    train_norms = {nt for nt in (_norm_dedup_text(r["text"]) for r in rows) if nt}
    hcal_ids = {r["id"] for r in human_cal}
    hcal_texts = {r["text"].strip().casefold() for r in human_cal}
    hcal_norms = {nt for nt in (_norm_dedup_text(r["text"]) for r in human_cal) if nt}
    assert len({r["id"] for r in test_full}) == len(test_full)
    for r, s in zip(test_full, test_slim):
        assert s == {k: r[k] for k in ("id", "text", "nhan", "bang_chung")}
        nt = _norm_dedup_text(r["text"])
        assert r["id"] not in train_ids and r["text"].strip().casefold() not in train_texts and (not nt or nt not in train_norms)
        assert r["id"] not in hcal_ids and r["text"].strip().casefold() not in hcal_texts and (not nt or nt not in hcal_norms)
        assert r.get("nguon") != "sinh_llm" and r.get("cach_gan") != "llm"
        assert r["nhan"] in LABELS
        assert r["nhan"] != "AN_TOAN" or not r["bang_chung"]
        lines = r["text"].split("\n")
        assert 1 <= len(lines) <= 8 and r["so_dong"] == len(lines)
        assert all(p and "\n" not in p and any(p in ln for ln in lines) for p in r["bang_chung"])
        if not r.get("nhom_trung"):
            r["nhom_trung"] = r["id"]
        r.setdefault("bien_dang", "C0")
        r.setdefault("goc_id", r["id"])
        r["meta"] = {"loai": "that", "phan_test": r.get("phan_test", ""), "giong_vung": r.get("giong_vung", ""), "bien_dang": "C0"}
    return test_full

def load_locked_real_test_perturb(test_c0):
    c0_by_id = {r["id"]: r for r in test_c0}
    train_ids = {r["id"] for r in rows}
    hcal_ids = {r["id"] for r in human_cal}
    seen_ids = set(c0_by_id)
    out = {}
    for tier, fname in (("C1", "test_perturb_C1.jsonl"), ("C2", "test_perturb_C2.jsonl")):
        recs = fetch_data(fname)
        assert len(recs) == 668, f"{fname}: kỳ vọng 668 dòng, nhận {len(recs)}"
        enriched = []
        for r in recs:
            rid = r["id"]
            assert rid not in seen_ids and rid not in train_ids and rid not in hcal_ids, f"Trùng/rò rỉ id {rid} trong {fname}"
            seen_ids.add(rid)
            gid = r["goc_id"]
            assert gid in c0_by_id, f"Thiếu goc_id {gid} cho {rid}"
            src = c0_by_id[gid]
            assert r["nhan"] == src["nhan"] and r["nhan"] in LABELS and r.get("bien_dang") == tier
            assert r["nhan"] != "AN_TOAN" or not r["bang_chung"]
            lines = r["text"].split("\n")
            assert 1 <= len(lines) <= 8
            assert all(p and "\n" not in p and any(p in ln for ln in lines) for p in r["bang_chung"])
            rec = dict(r)
            rec["so_dong"] = len(lines)
            rec["nhom_trung"] = gid
            for k in ("nguon", "cach_gan", "do_lo", "kenh", "the"):
                if k not in rec and k in src:
                    rec[k] = src[k]
            rec["meta"] = {
                "loai": "that",
                "phan_test": rec.get("phan_test", ""),
                "giong_vung": rec.get("giong_vung", ""),
                "bien_dang": tier,
                "kieu_bien_dang": rec.get("kieu_bien_dang", ""),
                "nhom_bien_dang": rec.get("nhom_bien_dang", ""),
            }
            enriched.append(rec)
        out[tier] = enriched
    return out["C1"], out["C2"]

def _compact_eval_summary(m, n):
    out = dict(
        n=int(n),
        macro_f1=round(float(m["macro_f1"]), 4),
        macro_f1_4main=round(float(m["macro_f1_4main"]), 4),
        binary_f1=round(float(m["binary_f1"]), 4),
        fpr_an_toan=round(float(m["fpr_an_toan"]), 4),
        accuracy=round(float(m["accuracy"]), 4),
        macro_f1_bootstrap_95=[round(float(x), 4) for x in m["macro_f1_bootstrap_95"]],
        evidence_f1=round(float(m["evidence"]["f1"]), 4),
        evidence_f1_theo_y=round(float(m["evidence"]["f1_theo_y"]), 4),
        safe_mark_rate=round(float(m["evidence"]["safe_mark_rate"]), 4),
        word_length_ratio=round(float(m["evidence"]["word_length_ratio"]), 4),
        harm_empty_evidence_rate=round(float(m["evidence"]["harm_empty_evidence_rate"]), 4),
    )
    if "worst_group_region_macro_f1" in m:
        out["worst_group_region_macro_f1"] = round(float(m["worst_group_region_macro_f1"]), 4)
        out["region_gap_max_min"] = round(float(m["region_gap_max_min"]), 4)
    return out

def _kieu_seen_in_train_aug(kieu):
    if kieu == "bo_dau":
        return bool(ACCENT_AUG > 0 or SURFACE_AUG > 0)
    if kieu in ("teencode", "keo_dai", "chen_ky_tu", "so_thay_chu"):
        return bool(SURFACE_AUG > 0)
    return False

def build_official_test_matrix(c0_rec, c0_pred, c1_rec, c1_pred, c2_rec, c2_pred, cal_cfg, raw_matrix=None):
    m0 = evaluate(c0_pred, c0_rec, cal_cfg=cal_cfg)
    m1 = evaluate(c1_pred, c1_rec, cal_cfg=cal_cfg)
    m2 = evaluate(c2_pred, c2_rec, cal_cfg=cal_cfg)
    tiers = {"C0": _compact_eval_summary(m0, len(c0_rec)), "C1": _compact_eval_summary(m1, len(c1_rec)), "C2": _compact_eval_summary(m2, len(c2_rec))}
    deltas = dict(
        C1_minus_C0=round(tiers["C1"]["macro_f1"] - tiers["C0"]["macro_f1"], 4),
        C2_minus_C0=round(tiers["C2"]["macro_f1"] - tiers["C0"]["macro_f1"], 4),
        C1_minus_C0_binary=round(tiers["C1"]["binary_f1"] - tiers["C0"]["binary_f1"], 4),
        C2_minus_C0_binary=round(tiers["C2"]["binary_f1"] - tiers["C0"]["binary_f1"], 4),
        C1_minus_C0_ev_theo_y=round(tiers["C1"]["evidence_f1_theo_y"] - tiers["C0"]["evidence_f1_theo_y"], 4),
        C2_minus_C0_ev_theo_y=round(tiers["C2"]["evidence_f1_theo_y"] - tiers["C0"]["evidence_f1_theo_y"], 4),
    )
    slices_15 = {}
    for tier_name, m in (("C0", m0), ("C1", m1), ("C2", m2)):
        for src_k, dst_k in (
            ("vung:bac", "bac"), ("vung:trung", "trung"), ("vung:nam", "nam"),
            ("phan:trong_phan_phoi", "da_thay"), ("phan:chua_thay", "chua_thay"),
        ):
            if src_k in m["slices"]:
                sl = m["slices"][src_k]
                slices_15[f"{tier_name}_{dst_k}"] = dict(
                    n=sl["n"],
                    macro_f1=round(float(sl["macro_present"]), 4),
                    binary_f1=round(float(sl["binary_f1"]), 4),
                    fpr_an_toan=round(float(sl["fpr_an_toan"]), 4),
                    evidence_f1=round(float(sl["evidence"]["f1"]), 4),
                    evidence_f1_theo_y=round(float(sl["evidence"]["f1_theo_y"]), 4),
                )
    delta_slices = {}
    for grp in ("bac", "trung", "nam", "da_thay", "chua_thay"):
        if f"C0_{grp}" in slices_15:
            for tier_name in ("C1", "C2"):
                if f"{tier_name}_{grp}" in slices_15:
                    delta_slices[f"{tier_name}_{grp}_minus_C0"] = round(
                        slices_15[f"{tier_name}_{grp}"]["macro_f1"] - slices_15[f"C0_{grp}"]["macro_f1"], 4
                    )
    by_kieu = {}
    for tier_name, m, recs in (("C1", m1, c1_rec), ("C2", m2, c2_rec)):
        kieu_nhom = {r.get("kieu_bien_dang", ""): r.get("nhom_bien_dang", "") for r in recs}
        for k, sl in sorted(m["slices"].items()):
            if k.startswith("kieu_bd:"):
                kieu = k.removeprefix("kieu_bd:")
                nhom_c0 = kieu_nhom.get(kieu, "")
                seen_aug = _kieu_seen_in_train_aug(kieu)
                by_kieu[kieu] = dict(
                    tier=tier_name,
                    n=sl["n"],
                    nhom_vs_train_c0=nhom_c0,
                    seen_in_train_aug=seen_aug,
                    strictly_unseen=(nhom_c0 == "chua_thay" and not seen_aug),
                    macro_f1=round(float(sl["macro_present"]), 4),
                    binary_f1=round(float(sl["binary_f1"]), 4),
                    fpr_an_toan=round(float(sl["fpr_an_toan"]), 4),
                    evidence_f1=round(float(sl["evidence"]["f1"]), 4),
                    evidence_f1_theo_y=round(float(sl["evidence"]["f1_theo_y"]), 4),
                )
    unseen_recs, unseen_preds = [], []
    for r, p in list(zip(c1_rec, c1_pred)) + list(zip(c2_rec, c2_pred)):
        if r.get("nhom_bien_dang") == "chua_thay" and not _kieu_seen_in_train_aug(r.get("kieu_bien_dang", "")):
            unseen_recs.append(r)
            unseen_preds.append(p)
    strictly_unseen = _compact_eval_summary(evaluate(unseen_preds, unseen_recs, cal_cfg=cal_cfg), len(unseen_recs)) if unseen_recs else None
    matrix = dict(
        data_rev=DATA_REV,
        train_aug_config=dict(accent_aug=ACCENT_AUG, surface_aug=SURFACE_AUG),
        tiers=tiers,
        deltas_vs_C0=deltas,
        strictly_unseen_perturb=strictly_unseen,
        slices_15=slices_15,
        delta_slices_vs_C0=delta_slices,
        by_kieu_bien_dang=by_kieu,
    )
    if raw_matrix is not None:
        gain = {t: round(tiers[t]["macro_f1"] - raw_matrix["tiers"][t]["macro_f1"], 4) for t in ("C0", "C1", "C2")}
        gain["C1_ev_theo_y"] = round(tiers["C1"]["evidence_f1_theo_y"] - raw_matrix["tiers"]["C1"]["evidence_f1_theo_y"], 4)
        gain["C2_ev_theo_y"] = round(tiers["C2"]["evidence_f1_theo_y"] - raw_matrix["tiers"]["C2"]["evidence_f1_theo_y"], 4)
        if strictly_unseen and raw_matrix.get("strictly_unseen_perturb"):
            gain["strictly_unseen"] = round(strictly_unseen["macro_f1"] - raw_matrix["strictly_unseen_perturb"]["macro_f1"], 4)
        gain["by_kieu"] = {
            k: round(by_kieu[k]["macro_f1"] - raw_matrix["by_kieu_bien_dang"][k]["macro_f1"], 4)
            for k in by_kieu if k in raw_matrix["by_kieu_bien_dang"]
        }
        matrix["gain_vs_raw"] = gain
    return matrix, (m0, m1, m2)

def _write_real_test_jsonl(pred_path, err_path, pred_list, rec_list, cal_cfg):
    rows_out, errors_out = [], []
    with pred_path.open("w", encoding="utf8") as f:
        for p, r in zip(pred_list, rec_list):
            out_rec = format_output(p, r["text"], cal_cfg=cal_cfg)
            row_out = dict(
                id=r["id"], nhan_dung=r["nhan"], nguon=r.get("nguon", "nguoi_viet"), cach_gan=r.get("cach_gan", ""),
                bien_dang=r.get("bien_dang", "C0"), goc_id=r.get("goc_id", r["id"]),
                kieu_bien_dang=r.get("kieu_bien_dang", "goc"), nhom_bien_dang=r.get("nhom_bien_dang", "goc"),
                phan_test=r.get("phan_test", ""), giong_vung=r.get("giong_vung", ""),
                kich_ban=r.get("kich_ban", ""), do_lo=r.get("do_lo", ""), kenh=r.get("kenh", ""),
                **out_rec,
            )
            f.write(json.dumps(row_out, ensure_ascii=False) + "\n")
            rows_out.append(row_out)
            if out_rec["nhan"] != r["nhan"]:
                errors_out.append(dict(row_out, text=r["text"], bang_chung_dung=r["bang_chung"]))
    if err_path is not None:
        with err_path.open("w", encoding="utf8") as f:
            for err in errors_out:
                f.write(json.dumps(err, ensure_ascii=False) + "\n")
    return rows_out, errors_out

real_test_records = real_test_encoded = real_test_pred = None
real_test_c1_records = real_test_c2_records = real_test_c1_pred = real_test_c2_pred = None
TEENCODE_DICT_MAP = fetch_teencode_dict() if EVAL_WITH_NORMALIZER else {}

if EVALUATE_REAL_TEST:
    if RUN_MODE != "full":
        raise RuntimeError("KHÓA AN TOÀN: Không chấm tập test thật ở RUN_MODE='smoke'! Đổi RUN_MODE='full' trước.")
    if TEST_LOCK_PATH.exists():
        lock_info = json.loads(TEST_LOCK_PATH.read_text(encoding="utf8"))
        print(f"ĐÃ KHÓA (1-SHOT LOCK): {TEST_LOCK_PATH} đã tồn tại (locked_at={lock_info.get('locked_at_utc')}, macro_f1={lock_info.get('macro_f1'):.4f}) — từ chối chấm lại lần 2!")
    else:
        frozen_cal = json.loads(CAL_PATH.read_text(encoding="utf8"))
        frozen_cal_sha = digest(CAL_PATH.read_bytes())
        real_test_records = load_locked_real_test()
        real_test_encoded = [prepare(r) for r in real_test_records]
        real_test_pred = infer(model, real_test_encoded)
        real_test_metrics = evaluate(real_test_pred, real_test_records, cal_cfg=frozen_cal)
        real_test_metrics["test_dataset_rev"] = DATA_REV
        real_test_metrics["test_day_du_sha256"] = DATA_SHA256["test_day_du.jsonl"]
        real_test_metrics["frozen_calibration_sha256"] = frozen_cal_sha
        write_json(RUN / "real_test_metrics.json", real_test_metrics)
        c0_rows_out, errors_out = _write_real_test_jsonl(
            RUN / "real_test_predictions.jsonl", RUN / "real_test_errors.jsonl",
            real_test_pred, real_test_records, frozen_cal,
        )
        save_diagnostics_plot(real_test_pred, RUN / "real_test_diagnostics.png", cal_cfg=frozen_cal)
        lock_payload = dict(
            locked_at_utc=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            data_rev=DATA_REV,
            test_day_du_sha256=DATA_SHA256["test_day_du.jsonl"],
            test_perturb_c1_sha256=(DATA_SHA256["test_perturb_C1.jsonl"] if RUN_REAL_TEST_PERTURB else None),
            test_perturb_c2_sha256=(DATA_SHA256["test_perturb_C2.jsonl"] if RUN_REAL_TEST_PERTURB else None),
            frozen_calibration_sha256=frozen_cal_sha,
            n_test=len(real_test_records),
            n_c1=0,
            n_c2=0,
            macro_f1=real_test_metrics["macro_f1"],
            macro_f1_4main=real_test_metrics["macro_f1_4main"],
            binary_f1=real_test_metrics["binary_f1"],
            fpr_an_toan=real_test_metrics["fpr_an_toan"],
            evidence_f1=real_test_metrics["evidence"]["f1"],
            evidence_f1_theo_y=real_test_metrics["evidence"]["f1_theo_y"],
            n_errors=len(errors_out),
        )
        raw_matrix = None
        if RUN_REAL_TEST_PERTURB:
            real_test_c1_records, real_test_c2_records = load_locked_real_test_perturb(real_test_records)
            real_test_c1_pred = infer(model, [prepare(r) for r in real_test_c1_records])
            real_test_c2_pred = infer(model, [prepare(r) for r in real_test_c2_records])
            raw_matrix, (_, m_c1, m_c2) = build_official_test_matrix(
                real_test_records, real_test_pred,
                real_test_c1_records, real_test_c1_pred,
                real_test_c2_records, real_test_c2_pred,
                frozen_cal,
            )
            write_json(RUN / "real_test_c1_metrics.json", m_c1)
            write_json(RUN / "real_test_c2_metrics.json", m_c2)
            write_json(RUN / "real_test_official_matrix.json", raw_matrix)
            c1_rows_out, _ = _write_real_test_jsonl(
                RUN / "real_test_c1_predictions.jsonl", RUN / "real_test_c1_errors.jsonl",
                real_test_c1_pred, real_test_c1_records, frozen_cal,
            )
            c2_rows_out, _ = _write_real_test_jsonl(
                RUN / "real_test_c2_predictions.jsonl", RUN / "real_test_c2_errors.jsonl",
                real_test_c2_pred, real_test_c2_records, frozen_cal,
            )
            with (RUN / "real_test_all_predictions.jsonl").open("w", encoding="utf8") as f:
                for row_out in c0_rows_out + c1_rows_out + c2_rows_out:
                    f.write(json.dumps(row_out, ensure_ascii=False) + "\n")
            lock_payload.update(
                n_c1=len(real_test_c1_records),
                n_c2=len(real_test_c2_records),
                c1_macro_f1=raw_matrix["tiers"]["C1"]["macro_f1"],
                c2_macro_f1=raw_matrix["tiers"]["C2"]["macro_f1"],
                delta_c1_minus_c0=raw_matrix["deltas_vs_C0"]["C1_minus_C0"],
                delta_c2_minus_c0=raw_matrix["deltas_vs_C0"]["C2_minus_C0"],
                c1_evidence_f1_theo_y=raw_matrix["tiers"]["C1"]["evidence_f1_theo_y"],
                c2_evidence_f1_theo_y=raw_matrix["tiers"]["C2"]["evidence_f1_theo_y"],
            )
        if EVAL_WITH_NORMALIZER:
            rt_norm_c0_pred = infer_normalized(model, real_test_records, TEENCODE_DICT_MAP)
            nc0_rows_out, _ = _write_real_test_jsonl(
                RUN / "real_test_norm_predictions.jsonl", None,
                rt_norm_c0_pred, real_test_records, frozen_cal,
            )
            if RUN_REAL_TEST_PERTURB:
                rt_norm_c1_pred = infer_normalized(model, real_test_c1_records, TEENCODE_DICT_MAP)
                rt_norm_c2_pred = infer_normalized(model, real_test_c2_records, TEENCODE_DICT_MAP)
                norm_matrix, (m0_norm, m1_norm, m2_norm) = build_official_test_matrix(
                    real_test_records, rt_norm_c0_pred,
                    real_test_c1_records, rt_norm_c1_pred,
                    real_test_c2_records, rt_norm_c2_pred,
                    frozen_cal, raw_matrix=raw_matrix,
                )
                write_json(RUN / "real_test_norm_metrics.json", m0_norm)
                write_json(RUN / "real_test_norm_c1_metrics.json", m1_norm)
                write_json(RUN / "real_test_norm_c2_metrics.json", m2_norm)
                write_json(RUN / "real_test_norm_official_matrix.json", norm_matrix)
                nc1_rows_out, _ = _write_real_test_jsonl(
                    RUN / "real_test_norm_c1_predictions.jsonl", None,
                    rt_norm_c1_pred, real_test_c1_records, frozen_cal,
                )
                nc2_rows_out, _ = _write_real_test_jsonl(
                    RUN / "real_test_norm_c2_predictions.jsonl", None,
                    rt_norm_c2_pred, real_test_c2_records, frozen_cal,
                )
                with (RUN / "real_test_norm_all_predictions.jsonl").open("w", encoding="utf8") as f:
                    for row_out in nc0_rows_out + nc1_rows_out + nc2_rows_out:
                        f.write(json.dumps(row_out, ensure_ascii=False) + "\n")
                lock_payload.update(
                    norm_c0_macro_f1=norm_matrix["tiers"]["C0"]["macro_f1"],
                    norm_c1_macro_f1=norm_matrix["tiers"]["C1"]["macro_f1"],
                    norm_c2_macro_f1=norm_matrix["tiers"]["C2"]["macro_f1"],
                    norm_gain_c1=norm_matrix["gain_vs_raw"]["C1"],
                )
            else:
                norm_c0_m = evaluate(rt_norm_c0_pred, real_test_records, cal_cfg=frozen_cal)
                write_json(RUN / "real_test_norm_metrics.json", norm_c0_m)
                lock_payload["norm_c0_macro_f1"] = round(float(norm_c0_m["macro_f1"]), 4)
        write_json(TEST_LOCK_PATH, lock_payload)
        print(
            f"=== KẾT QUẢ TEST THẬT ĐÃ KHÓA (C0={len(real_test_records)}, C1={lock_payload['n_c1']}, C2={lock_payload['n_c2']}) ===\n"
            f"C0 macro-F1 (5 lớp): {real_test_metrics['macro_f1']:.4f} | FPR(AN_TOAN): {real_test_metrics['fpr_an_toan']:.4f} | "
            f"binary-F1: {real_test_metrics['binary_f1']:.4f} | Ev F1 (word/theo_y): "
            f"{real_test_metrics['evidence']['f1']:.4f} / {real_test_metrics['evidence']['f1_theo_y']:.4f} | "
            f"safe_mark: {real_test_metrics['evidence']['safe_mark_rate']:.4f} | empty_harm: {100*real_test_metrics['evidence']['harm_empty_evidence_rate']:.2f}%"
        )
        if raw_matrix is not None:
            print("Ma trận chính thức C0 / C1 / C2 (Dòng #5 thô):", raw_matrix["tiers"])
            print("Độ tụt ΔF1 so với C0:", raw_matrix["deltas_vs_C0"])
        if EVAL_WITH_NORMALIZER and RUN_REAL_TEST_PERTURB:
            print("Lợi ích chuẩn hóa Dòng #6 − #5:", norm_matrix["gain_vs_raw"])
        print(pd.DataFrame(real_test_metrics["slices"]).T[["n", "macro_present", "binary_f1", "accuracy"]])
else:
    print("Bộ test thật (334 C0 + 668 C1 + 668 C2 = 1.670 mẫu) đang ĐÓNG KHÓA (EVALUATE_REAL_TEST = False).")

---
<a id="sec-7"></a>
## 7. Kiểm thử độ bền (Stress Tests), Ngữ cảnh đa lượt & Benchmark hiệu năng

Ô này đo độ bền biến dạng trên 5 bộ phép thử tất định đồng bộ 100% với Nhánh 1, kiểm tra đóng góp của ngữ cảnh hội thoại và đo chi phí suy luận sạch trên GPU:

| Nhóm kiểm thử | Bộ phép thử / Chỉ số | Trạng thái lúc huấn luyện |
| :--- | :--- | :---: |
| **Stress Tests (5 bộ tất định)** | `accent_strip`, `c1_in_dist` | Đã thấy nếu bật `ACCENT_AUG` / `SURFACE_AUG` |
| **Stress Tests (Chưa thấy)** | `c1_heldout_controlled`, `blindspot_homoglyph`, `blindspot_zero_width` | **Chưa thấy (`seen_in_train = False`)** |
| **Xem trước Chuẩn hóa `#6`** | `{report_key}_norm_stress_tests.json` (`#6 − #5` trên `clean_c0` và 5 bộ stress) | Không chạm bộ test khóa |
| **Kiểm chứng ngữ cảnh** | Length-only Decision Tree baseline & So sánh giữ đủ hội thoại vs chỉ giữ lượt cuối | `{report_key}_context_checks.json` |
| **Benchmark VRAM & Độ trễ** | Dọn cache CUDA; ghi tách bạch `model_weights_cuda_MiB`, `inference_baseline_cuda_MiB`, `session_peak_cuda_MiB`, `p50/p95 ms` | `benchmark.json` |

Cuối ô cung cấp hàm suy luận tương tác `predict_text(net, text, use_normalizer=False)` cho phép thử nhanh cả hai chế độ `#5` (thô) và `#6` (chuẩn hóa + chiếu ngược verbatim).

In [ ]:
#@title 7.1. Kiểm thử độ bền (5 bộ + #6), Bỏ ngữ cảnh, Đo VRAM/Độ trễ & Demo predict_text
# ==============================================================================
# 1. ĐỊNH NGHĨA 5 BỘ KIỂM THỬ BIẾN DẠNG TẤT ĐỊNH
# ==============================================================================
def _stress_eval(label, transform_fn, seen_in_train, eval_encoded=None, eval_pred=None):
    eval_encoded = report_encoded if eval_encoded is None else eval_encoded
    eval_pred = final_pred if eval_pred is None else eval_pred
    changed, original = [], []
    for e, p in zip(eval_encoded, eval_pred):
        r = e["row"]
        variant = transform_fn(r["text"])
        if variant != r["text"]:
            changed.append(prepare(dict(r, text=variant, bang_chung=[])))
            original.append(p)
    if not changed:
        return None
    stress = infer(model, changed)
    a, truth = arrays(original)
    b, _ = arrays(stress)
    orig_f1, pert_f1 = float(macro(original)), float(macro(stress))
    main4 = [LABELS.index(l) for l in ("AN_TOAN", "QUAY_ROI", "DE_DOA", "LUA_DAO")]
    orig_f1_4 = float(f1_score(truth, a.argmax(1), labels=main4, average="macro", zero_division=0))
    pert_f1_4 = float(f1_score(truth, b.argmax(1), labels=main4, average="macro", zero_division=0))
    orig_bin = float(f1_score(truth != 0, a.argmax(1) != 0, zero_division=0))
    pert_bin = float(f1_score(truth != 0, b.argmax(1) != 0, zero_division=0))
    return dict(
        suite=label,
        seen_in_train=bool(seen_in_train),
        n=len(stress),
        original_f1=orig_f1,
        perturbed_f1=pert_f1,
        delta_f1=pert_f1 - orig_f1,
        original_f1_4main=orig_f1_4,
        perturbed_f1_4main=pert_f1_4,
        original_binary_f1=orig_bin,
        perturbed_binary_f1=pert_bin,
        flip_rate=float((a.argmax(1) != b.argmax(1)).mean()),
        newly_missed_harm=int(((truth != 0) & (a.argmax(1) != 0) & (b.argmax(1) == 0)).sum()),
    )

def _sample_rng(offset, text):
    return random.Random(f"{SEED}:{offset}:{text}")

def _eval_c1_in_dist(text):
    rng = _sample_rng(77, text)
    lines = []
    for line in text.split("\n"):
        row_stub = augment_surface(dict(id="eval", text=line, nhan="AN_TOAN", bang_chung=[]), rng)
        lines.append(row_stub["text"])
    return "\n".join(lines)

_HELDOUT_PHRASES = {
    "bây giờ": ("bg", "bi h"),
    "bao giờ": ("bh",),
    "như thế nào": ("ntn",),
    "được không": ("đc ko",),
    "tại sao": ("tsao",),
    "anh em": ("ae",),
    "bình thường": ("bth",),
    "nói chuyện": ("nc",),
    "mọi người": ("các fen",),
    "điện thoại": ("đt",),
    "tin nhắn": ("tn",),
}
_HELDOUT_WORDS = {
    "không": ("hổng",),
    "vào": ("dzô",),
    "về": ("dìa",),
    "thế": ("rứa",),
    "đâu": ("mô",),
    "bạn": ("fen",),
    "facebook": ("fb",),
}
_HELDOUT_LOOKALIKE = {"s": ("5",), "g": ("9",), "b": ("8",)}
_HELDOUT_SEPS = ("·",)
_QWERTY_NEIGHBOURS = {
    "a": "sqw", "b": "vghn", "c": "xdfv", "d": "serfcx", "e": "wsdr",
    "g": "ftyhbv", "h": "gyujnb", "i": "ujko", "k": "ijolm", "m": "njk",
    "n": "bhjm", "o": "iklp", "p": "ol", "q": "wa", "r": "edft",
    "s": "awedxz", "t": "rfgy", "u": "yhji", "v": "cfgb", "x": "zsdc", "y": "tghu",
}

def _eval_c1_heldout(text):
    rng = _sample_rng(78, text)
    out_lines = []
    for line in text.split("\n"):
        cur = line
        for src_p, alts in _HELDOUT_PHRASES.items():
            pat = re.compile(rf"(?<!\w){re.escape(src_p)}(?!\w)", re.IGNORECASE)
            if pat.search(cur):
                cur = pat.sub(lambda m: rng.choice(alts), cur, count=1)
        words_out = []
        mode = rng.choice(("lookalike", "sep", "typo"))
        edited = False
        for m in re.finditer(r"\S+|\s+", cur):
            tok_str = m.group(0)
            if tok_str.isspace():
                words_out.append(tok_str)
                continue
            lead, core, trail = _split_affix(tok_str)
            w = core
            if w and w.lower() in _HELDOUT_WORDS and rng.random() < 0.7:
                rep = rng.choice(_HELDOUT_WORDS[w.lower()])
                w = rep.capitalize() if w[0].isupper() else rep
            elif not edited and len(w) >= 4 and _ALPHA_CORE_RE.match(w):
                if mode == "lookalike":
                    ixs = [i for i in range(1, len(w)) if w[i].lower() in _HELDOUT_LOOKALIKE]
                    if ixs:
                        pick = rng.choice(ixs)
                        w = w[:pick] + rng.choice(_HELDOUT_LOOKALIKE[w[pick].lower()]) + w[pick + 1:]
                        edited = True
                elif mode == "sep":
                    cut = rng.randrange(1, len(w))
                    w = w[:cut] + rng.choice(_HELDOUT_SEPS) + w[cut:]
                    edited = True
                elif mode == "typo" and len(w) >= 5:
                    ixs = [i for i in range(1, len(w) - 1) if w[i] in _QWERTY_NEIGHBOURS]
                    if ixs:
                        pick = rng.choice(ixs)
                        w = w[:pick] + rng.choice(_QWERTY_NEIGHBOURS[w[pick]]) + w[pick + 1:]
                        edited = True
            words_out.append(lead + w + trail)
        out_lines.append("".join(words_out))
    return "\n".join(out_lines)

def _eval_blindspot_homoglyph(text):
    cmap = {"a": "а", "e": "е", "o": "о", "c": "с", "p": "р", "y": "у", "x": "х"}
    rng = _sample_rng(88, text)
    return "".join(cmap[ch] if ch in cmap and rng.random() < 0.40 else ch for ch in text)

def _eval_blindspot_zw(text):
    rng = _sample_rng(99, text)
    spans = [m.span() for m in re.finditer(r"[a-zA-ZÀ-ỹ]{3,}", text)]
    if not spans:
        return text
    picks = sorted(rng.sample(spans, min(3, len(spans))))
    out, prev = [], 0
    for sa, sb in picks:
        out.append(text[prev:sa])
        w = text[sa:sb]
        cut = rng.randrange(1, len(w))
        out.append(w[:cut] + "\u200b" + w[cut:])
        prev = sb
    out.append(text[prev:])
    return "".join(out)


# ==============================================================================
# 2. CHẠY STRESS TESTS (#5 THÔ) & XEM TRƯỚC CHUẨN HÓA (#6)
# ==============================================================================
stress_suites = {}
for name, fn, seen_flag in (
    ("accent_strip", strip_accents, ACCENT_AUG > 0 or SURFACE_AUG > 0),
    ("c1_in_dist", _eval_c1_in_dist, SURFACE_AUG > 0),
    ("c1_heldout_controlled", _eval_c1_heldout, False),
    ("blindspot_homoglyph", _eval_blindspot_homoglyph, False),
    ("blindspot_zero_width", _eval_blindspot_zw, False),
):
    res = _stress_eval(name, fn, seen_flag)
    if res is not None:
        stress_suites[name] = res

if "accent_strip" in stress_suites:
    s = stress_suites["accent_strip"]
    write_json(
        RUN / f"{report_key}_accent_stress.json",
        dict(
            n=s["n"],
            seen_in_train=s["seen_in_train"],
            original_f1=s["original_f1"],
            stripped_f1=s["perturbed_f1"],
            flip_rate=s["flip_rate"],
            newly_missed_harm=s["newly_missed_harm"],
        ),
    )
write_json(RUN / f"{report_key}_stress_tests.json", stress_suites)
print(pd.DataFrame(stress_suites).T)

if EVAL_WITH_NORMALIZER:
    norm_stress_suites = {}
    report_records_c0 = [e["row"] for e in report_encoded]
    raw_c0_eval = evaluate(final_pred, report_records_c0)
    norm_c0_pred = infer_normalized(model, report_records_c0, TEENCODE_DICT_MAP)
    norm_c0_eval = evaluate(norm_c0_pred, report_records_c0)
    norm_stress_suites["clean_c0"] = dict(
        suite="clean_c0",
        seen_in_train=True,
        n=len(norm_c0_pred),
        perturbed_raw_f1=float(raw_c0_eval["macro_f1"]),
        perturbed_norm_f1=float(norm_c0_eval["macro_f1"]),
        gain_vs_raw_f1=float(norm_c0_eval["macro_f1"] - raw_c0_eval["macro_f1"]),
        perturbed_norm_binary_f1=float(norm_c0_eval["binary_f1"]),
        fpr_an_toan=float(norm_c0_eval["fpr_an_toan"]),
        evidence_f1=float(norm_c0_eval["evidence"]["f1"]),
        evidence_f1_theo_y=float(norm_c0_eval["evidence"]["f1_theo_y"]),
        gain_vs_raw_ev_theo_y=float(norm_c0_eval["evidence"]["f1_theo_y"] - raw_c0_eval["evidence"]["f1_theo_y"]),
    )
    for name, fn, seen_flag in (
        ("accent_strip", strip_accents, ACCENT_AUG > 0 or SURFACE_AUG > 0),
        ("c1_in_dist", _eval_c1_in_dist, SURFACE_AUG > 0),
        ("c1_heldout_controlled", _eval_c1_heldout, False),
        ("blindspot_homoglyph", _eval_blindspot_homoglyph, False),
        ("blindspot_zero_width", _eval_blindspot_zw, False),
    ):
        changed_rows, orig_p = [], []
        for e, p in zip(report_encoded, final_pred):
            r = e["row"]
            v = fn(r["text"])
            if v != r["text"]:
                changed_rows.append(dict(r, text=v, bang_chung=[]))
                orig_p.append(p)
        if not changed_rows:
            continue
        sp = infer_normalized(model, changed_rows, TEENCODE_DICT_MAP)
        a_arr, y_arr = arrays(orig_p)
        b_arr, _ = arrays(sp)
        pf1 = float(macro(sp))
        raw_pf1 = stress_suites[name]["perturbed_f1"] if name in stress_suites else pf1
        norm_stress_suites[name] = dict(
            suite=name,
            seen_in_train=bool(seen_flag),
            n=len(sp),
            perturbed_raw_f1=raw_pf1,
            perturbed_norm_f1=pf1,
            gain_vs_raw_f1=pf1 - raw_pf1,
            perturbed_norm_binary_f1=float(f1_score(y_arr != 0, b_arr.argmax(1) != 0, zero_division=0)),
            newly_missed_harm=int(((y_arr != 0) & (a_arr.argmax(1) != 0) & (b_arr.argmax(1) == 0)).sum()),
        )
    write_json(RUN / f"{report_key}_norm_stress_tests.json", norm_stress_suites)
    print(f"=== Xem trước chuẩn hóa Dòng #6 trên {report_key} ===")
    print(pd.DataFrame(norm_stress_suites).T)


# ==============================================================================
# 3. STRESS TESTS BỔ SUNG TRÊN BỘ TEST THẬT (KHI ĐÃ MỞ KHÓA)
# ==============================================================================
if real_test_encoded is not None and real_test_pred is not None and RUN_REAL_TEST_PERTURB:
    rt_stress = {}
    for name, fn, seen_flag in (
        ("accent_strip", strip_accents, ACCENT_AUG > 0 or SURFACE_AUG > 0),
        ("c1_in_dist", _eval_c1_in_dist, SURFACE_AUG > 0),
        ("c1_heldout_controlled", _eval_c1_heldout, False),
        ("blindspot_homoglyph", _eval_blindspot_homoglyph, False),
        ("blindspot_zero_width", _eval_blindspot_zw, False),
    ):
        res = _stress_eval(name, fn, seen_flag, eval_encoded=real_test_encoded, eval_pred=real_test_pred)
        if res is not None:
            rt_stress[name] = res
    write_json(RUN / "real_test_stress_tests.json", rt_stress)
    print("=== Độ bền biến dạng bổ sung trên TEST THẬT (334 câu) ===")
    print(pd.DataFrame(rt_stress).T)


# ==============================================================================
# 4. MỐC ĐỐI CHIẾU ĐỘ DÀI (LENGTH BASELINE) & PHÉP THỬ BỎ NGỮ CẢNH ĐA LƯỢT
# ==============================================================================
from sklearn.tree import DecisionTreeClassifier
def lengths(e):
    t = e["row"]["text"]
    return [len(t.splitlines()), len(t), len(t.split())]
length_model = DecisionTreeClassifier(max_depth=4, min_samples_leaf=5, class_weight="balanced", random_state=SEED)
length_model.fit([lengths(e) for e in encoded["fit"]], [e["label"] for e in encoded["fit"]])
comparison_data = report_encoded
length_pred = length_model.predict([lengths(e) for e in comparison_data])
diag = {"length_only_macro_f1": float(f1_score([e["label"] for e in comparison_data], length_pred, average="macro", labels=range(5), zero_division=0))}

mult = [(e, p) for e, p in zip(comparison_data, final_pred) if e["row"]["so_dong"] > 1]
if mult:
    last_turn = [prepare(dict(e["row"], text=e["row"]["text"].split("\n")[-1], bang_chung=[])) for e, _ in mult]
    truncated_pred = infer(model, last_turn)
    diag.update(
        multi_n=len(mult),
        full_context_multi_macro_f1=float(macro([p for _, p in mult])),
        last_turn_multi_macro_f1=float(macro(truncated_pred)),
    )
write_json(RUN / f"{report_key}_context_checks.json", diag)
print(diag)


# ==============================================================================
# 5. BENCHMARK VRAM & ĐỘ TRỄ SUY LUẬN SẠCH (P50 / P95)
# ==============================================================================
@torch.inference_mode()
def benchmark(net, data, device, limit=100):
    net.eval()
    net.zero_grad(set_to_none=True)
    gc.collect()
    samples = data[:limit]
    def sync():
        if device.type == "cuda":
            torch.cuda.synchronize(device)
    def call(e):
        with (amp() if device.type == "cuda" else nullcontext()):
            return net(to_device(collate([e]), device))
    if device.type == "cuda":
        torch.cuda.empty_cache()
        sync()
        weights_mib = (
            sum(p.numel() * p.element_size() for p in net.parameters() if p.is_cuda)
            + sum(b.numel() * b.element_size() for b in net.buffers() if b.is_cuda)
        ) / 2**20
        baseline_mib = torch.cuda.memory_allocated(device) / 2**20
        if baseline_mib > weights_mib * 1.35 + 64.0:
            print(f"CẢNH BÁO VRAM: baseline ({baseline_mib:.1f} MiB) lớn hơn đáng kể trọng số mô hình ({weights_mib:.1f} MiB)!")
    else:
        weights_mib = baseline_mib = None
    for e in samples[:5]:
        call(e)
    sync()
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
    latency, prep = [], []
    for e in samples:
        start = time.perf_counter()
        prepare(e["row"])
        prep.append(1000 * (time.perf_counter() - start))
        start = time.perf_counter()
        call(e)
        sync()
        latency.append(1000 * (time.perf_counter() - start))
    peak_mib = torch.cuda.max_memory_allocated(device) / 2**20 if device.type == "cuda" else None
    return dict(
        n=len(samples),
        batch=1,
        model_and_collate_p50_ms=float(np.median(latency)),
        model_and_collate_p95_ms=float(np.percentile(latency, 95)),
        prepare_p50_ms=float(np.median(prep)),
        model_weights_cuda_MiB=(round(weights_mib, 2) if weights_mib is not None else None),
        inference_baseline_cuda_MiB=(round(baseline_mib, 2) if baseline_mib is not None else None),
        session_peak_cuda_MiB=(round(peak_mib, 2) if peak_mib is not None else None),
        activation_overhead_cuda_MiB=(round(max(0.0, peak_mib - baseline_mib), 2) if peak_mib is not None else None),
    )

perf = benchmark(model, report_encoded, DEVICE)
write_json(RUN / "benchmark.json", perf)
print(perf)


# ==============================================================================
# 6. HÀM SUY LUẬN MẪU PREDICT_TEXT() & CHẠY THỬ
# ==============================================================================
def predict_text(net, text, device=DEVICE, use_normalizer=False):
    if not text.strip() or not 1 <= len(text.split("\n")) <= 8:
        raise ValueError("Nhập 1–8 lượt, mỗi lượt một dòng")
    stub = dict(id="demo", text=text, nhan="AN_TOAN", bang_chung=[])
    if use_normalizer:
        tmap = TEENCODE_DICT_MAP or fetch_teencode_dict()
        item = infer_normalized(net, [stub], tmap, infer_fn=lambda m, d: infer(m, d, device=device))[0]
    else:
        item = infer(net, [prepare(stub)], device=device)[0]
    return format_output(item, text)

restore(model, load_file(str(RUN / "model_best.safetensors")))
print(predict_text(model, "Chiều nay nhóm mình họp lúc mấy giờ?\nBa giờ ở thư viện nhé."))

---
<a id="sec-8"></a>
## 8. Lượng tử hóa INT8 trên CPU & Xuất đồ thị ONNX (Triển khai On-Device)

Ô tùy chọn phục vụ đóng gói và đo chi phí trên thiết bị đích (Laptop CPU & Web Demo ONNX Runtime):

| Cờ điều khiển | Chức năng & Kết quả kỳ vọng |
| :--- | :--- |
| **`RUN_INT8_CPU = True`** | Lượng tử hóa cả **`nn.Embedding` lớn** (`PerRowInt8Embedding` — INT8 đối xứng theo hàng + scale FP32, giữ nguyên `padding_idx` và giữ `position_embeddings <= 2050` ở FP32, nén ~4× bảng từ vựng: `750 → 188 MiB` trên `mmbert_base`, `60 → 15 MiB` trên `bamibert`, `44 → 11 MiB` trên `visobert`) lẫn các lớp **`nn.Linear`** (`qint8`). Xuất `int8_comparison.json` so sánh dung lượng đĩa, độ trễ p50/p95 CPU và chênh lệch Macro-F1 / Evidence-F1. |
| **`EXPORT_ONNX = True`** | Xuất nhánh suy luận cốt lõi sang đồ thị ONNX opset 17 (`dualorc_core.onnx`, cần gói `onnx`/`onnxruntime`) kèm toàn bộ tham số hậu xử lý và ma trận chuyển trạng thái CRF trong `crf_params.json`. |

In [ ]:
#@title 8.1. Lượng tử hóa INT8 (PerRowInt8Embedding + Linear) & Xuất đồ thị ONNX
# ==============================================================================
# 1. LỚP PER-ROW INT8 EMBEDDING (NÉN 4×, SCALE FP32)
# ==============================================================================
class PerRowInt8Embedding(nn.Module):
    """Lượng tử hóa đối xứng INT8 theo từng hàng từ vựng (nén ~4x, giải nén cục bộ chỉ trên token đang tra cứu)."""
    def __init__(self, emb: nn.Embedding):
        super().__init__()
        w = emb.weight.detach().float().cpu()
        self.num_embeddings = emb.num_embeddings
        self.embedding_dim = emb.embedding_dim
        self.padding_idx = emb.padding_idx
        scale = w.abs().amax(dim=-1, keepdim=True).clamp_min(1e-8) / 127.0
        qweight = (w / scale).round().clamp(-127, 127).to(torch.int8)
        if self.padding_idx is not None:
            qweight[self.padding_idx].zero_()
            scale[self.padding_idx].zero_()
        self.register_buffer("qweight", qweight)
        self.register_buffer("scale", scale.to(torch.float32))

    def forward(self, input_ids):
        q = F.embedding(input_ids, self.qweight).float()
        s = F.embedding(input_ids, self.scale)
        return q * s

def quantize_embeddings_int8(module, min_num_embeddings=4096):
    replaced = 0
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Embedding) and child.num_embeddings >= min_num_embeddings:
            setattr(module, name, PerRowInt8Embedding(child))
            replaced += 1
        else:
            replaced += quantize_embeddings_int8(child, min_num_embeddings=min_num_embeddings)
    return replaced


# ==============================================================================
# 2. ĐÁNH GIÁ NÉN INT8 TRÊN CPU (DUNG LƯỢNG, ĐỘ TRỄ & F1)
# ==============================================================================
if RUN_INT8_CPU:
    import io
    cpu = torch.device("cpu")
    if "x86" in torch.backends.quantized.supported_engines:
        torch.backends.quantized.engine = "x86"
    subset = report_encoded[:min(256, len(report_encoded))]
    cpu_net = copy.deepcopy(model.to(cpu)).float().eval()
    model.to(DEVICE)
    if hasattr(cpu_net.backbone, "gradient_checkpointing_disable"):
        cpu_net.backbone.gradient_checkpointing_disable()
    fp_buf = io.BytesIO()
    torch.save(cpu_net.state_dict(), fp_buf)
    fp32_mib = len(fp_buf.getvalue()) / 2**20
    del fp_buf
    fp = infer(cpu_net, subset, device=cpu)
    fp_perf = benchmark(cpu_net, subset, cpu)
    emb_replaced = quantize_embeddings_int8(cpu_net.backbone, min_num_embeddings=4096) if QUANTIZE_EMBEDDINGS else 0
    from torch.ao.quantization import quantize_dynamic
    quantize_dynamic(cpu_net.backbone, {nn.Linear}, dtype=torch.qint8, inplace=True)
    qi = infer(cpu_net, subset, device=cpu)
    tmp = RUN / "int8.tmp"
    torch.save(cpu_net.state_dict(), tmp)
    os.replace(tmp, RUN / "int8_cpu.pt")
    quant_result = dict(
        n=len(subset),
        engine=torch.backends.quantized.engine,
        quantize_embeddings=QUANTIZE_EMBEDDINGS,
        embeddings_quantized=emb_replaced,
        fp32_serialized_MiB=round(fp32_mib, 2),
        serialized_MiB=round((RUN / "int8_cpu.pt").stat().st_size / 2**20, 2),
        fp32=evaluate(fp, [e["row"] for e in subset]),
        int8=evaluate(qi, [e["row"] for e in subset]),
        fp32_perf=fp_perf,
        int8_perf=benchmark(cpu_net, subset, cpu),
    )
    write_json(RUN / "int8_comparison.json", quant_result)
    print(
        f"FP32 / INT8 macro-F1: {macro(fp):.4f} / {macro(qi):.4f} | "
        f"INT8 CPU p50: {quant_result['int8_perf']['model_and_collate_p50_ms']:.1f} ms | "
        f"Size: {quant_result['fp32_serialized_MiB']:.1f} -> {quant_result['serialized_MiB']:.1f} MiB (emb_q={emb_replaced})"
    )
    del cpu_net
    gc.collect()


# ==============================================================================
# 3. XUẤT NHÁNH SUY LUẬN CỐT LÕI SANG ONNX (OPSET 17)
# ==============================================================================
if EXPORT_ONNX:
    class _OnnxWrapper(nn.Module):
        def __init__(self, net):
            super().__init__()
            self.net = net
        def forward(self, ids, attn_mask, word_sub_idx, word_sub_mask, word_mask, word_turn, word_recency, word_bytes):
            b = dict(
                ids=ids, attn_mask=attn_mask, word_sub_idx=word_sub_idx,
                word_sub_mask=word_sub_mask, word_mask=word_mask, word_turn=word_turn,
                word_recency=word_recency, word_bytes=word_bytes,
            )
            out = self.net(b, with_aux=False)
            return out["logits"], out["emissions"], out["severity"]

    cpu = torch.device("cpu")
    ex_net = copy.deepcopy(model.to(cpu)).float().eval()
    model.to(DEVICE)
    if hasattr(ex_net.backbone, "gradient_checkpointing_disable"):
        ex_net.backbone.gradient_checkpointing_disable()
    sample_b = collate([report_encoded[0]])
    onnx_path = RUN / "dualorc_crf.onnx"
    torch.onnx.export(
        _OnnxWrapper(ex_net),
        (
            sample_b["ids"], sample_b["attn_mask"], sample_b["word_sub_idx"],
            sample_b["word_sub_mask"], sample_b["word_mask"], sample_b["word_turn"],
            sample_b["word_recency"], sample_b["word_bytes"],
        ),
        str(onnx_path),
        opset_version=17,
        input_names=["ids", "attn_mask", "word_sub_idx", "word_sub_mask", "word_mask", "word_turn", "word_recency", "word_bytes"],
        output_names=["logits", "emissions", "severity"],
        dynamic_axes={
            "ids": {0: "B", 1: "L"}, "attn_mask": {0: "B", 1: "L"},
            "word_sub_idx": {0: "B", 1: "W", 2: "S"}, "word_sub_mask": {0: "B", 1: "W", 2: "S"},
            "word_mask": {0: "B", 1: "W"}, "word_turn": {0: "B", 1: "W"},
            "word_recency": {0: "B", 1: "W"}, "word_bytes": {0: "B", 1: "W"},
            "logits": {0: "B"}, "emissions": {0: "B", 1: "W"}, "severity": {0: "B"},
        },
    )
    export_meta = dict(
        chosen_decoder=cal_payload["chosen_decoder"],
        temperature=cal_payload["temperature"],
        o_bias=cal_payload["o_bias"],
        marginal_threshold=cal_payload["marginal_threshold"],
        single_word_threshold=cal_payload["single_word_threshold"],
        bridge_ratio=cal_payload["bridge_ratio"],
        fallback_min_prob=cal_payload["fallback_min_prob"],
    )
    if crf_np is not None:
        export_meta.update({k: v.tolist() for k, v in crf_np.items()})
    write_json(RUN / "crf_params.json", export_meta)
    del ex_net
    gc.collect()
    print(f"Đã xuất ONNX: {onnx_path} ({onnx_path.stat().st_size / 2**20:.1f} MiB)")

---
<a id="sec-9"></a>
## 9. Đối chiếu liên nhánh (`GuardMerge`) & Mô phỏng `VietSafe-HierBi-Cascade`

Đặt `GUARDMERGE_RUN_DIR` (ở Ô 1.2b) trỏ tới thư mục kết quả Nhánh 1 ([`v2_guardmerge_colab.ipynb`](v2_guardmerge_colab.ipynb)) để tự động thực hiện đối chiếu thống kê và mô phỏng kiến trúc tầng đôi **không rò rỉ**:

1. **Kiểm định McNemar ghép cặp:** So sánh cặp-đối-cặp trên `dev` (hoặc `audit`), thống kê số ca Nhánh 2 dự đoán đúng mà Nhánh 1 sai và ngược lại kèm p-value hai phía.
2. **Bộ định tuyến UCCI (`IsotonicRegression`) không rò rỉ:** Khớp mô hình rủi ro đẳng trương có trọng số `cal_weights` duy nhất trên tập **`cal`** của Tầng 1, sau đó đánh giá đường cong tốc độ–chất lượng trên `dev`.
3. **Đánh giá trên `real_test` (khi cả hai nhánh đã mở khóa):** Áp nguyên bộ định tuyến UCCI đã đóng băng trên `cal` sang bộ test khóa (tuyệt đối không khớp lại trên test).

In [ ]:
#@title 9.1. Kiểm định McNemar & Mô phỏng bộ định tuyến UCCI Cascade (không rò rỉ)
# ==============================================================================
# 1. HÀM PHÂN VỊ CÓ TRỌNG SỐ, ĐIỂM ĐỊNH TUYẾN UCCI & ĐỐI CHIẾU MCNEMAR
# ==============================================================================
def _weighted_quantile(values, weights, q):
    values = np.asarray(values, dtype=np.float64)
    weights = np.asarray(weights, dtype=np.float64)
    order = np.argsort(values)
    v_s, w_s = values[order], weights[order]
    cdf = np.cumsum(w_s) / max(w_s.sum(), 1e-12)
    return float(np.interp(q, cdf, v_s))

def _ucci_route_score(iso_router, margins):
    # Phá hòa trong vùng rủi ro đẳng trương bằng 0 (tránh ngưỡng 0.0 làm chuyển tầng 100% mẫu)
    return iso_router.predict(margins) + 1e-6 * (1.0 - margins)

def _run_mcnemar_cascade(split_label, eval_pred, gm_pred_path, b2_pred_path, iso_router, cal_score_ref, cal_w_ref, cal_temp):
    gm_preds = {r["id"]: r for r in (json.loads(l) for l in gm_pred_path.read_text(encoding="utf8").splitlines() if l.strip())}
    b2_preds = {r["id"]: r for r in (json.loads(l) for l in b2_pred_path.read_text(encoding="utf8").splitlines() if l.strip())}
    assert set(gm_preds) == set(b2_preds), f"Tập ID giữa Nhánh 1 và Nhánh 2 trên {split_label} không trùng khớp!"
    b_fix = c_brk = both_ok = both_err = 0
    for sid, r2 in b2_preds.items():
        r1 = gm_preds[sid]
        ok1 = r1["nhan"] == r1["nhan_dung"]
        ok2 = r2["nhan"] == r2["nhan_dung"]
        if ok1 and ok2: both_ok += 1
        elif (not ok1) and ok2: b_fix += 1
        elif ok1 and (not ok2): c_brk += 1
        else: both_err += 1
    pval = float(binomtest(b_fix, b_fix + c_brk, 0.5).pvalue) if (b_fix + c_brk) else 1.0
    gm_f1 = float(f1_score([LABELS.index(gm_preds[p["id"]]["nhan_dung"]) for p in eval_pred],
                           [LABELS.index(gm_preds[p["id"]]["nhan"]) for p in eval_pred],
                           labels=range(5), average="macro", zero_division=0))
    b2_f1 = float(macro(eval_pred))
    print(f"McNemar ({split_label}): Nhánh 1 F1={gm_f1:.4f} | Nhánh 2 F1={b2_f1:.4f} | B2 đúng/B1 sai={b_fix}, B1 đúng/B2 sai={c_brk}, p={pval:.5f}")

    z_ev, y_ev = arrays(eval_pred)
    p_ev = softmax(z_ev / cal_temp, axis=1)
    top2_ev = np.sort(p_ev, axis=1)[:, -2:]
    margin_ev = top2_ev[:, 1] - top2_ev[:, 0]
    ev_score = _ucci_route_score(iso_router, margin_ev)
    t1_pred_idx = p_ev.argmax(1)
    t2_pred_idx = np.array([LABELS.index(gm_preds[p["id"]]["nhan"]) for p in eval_pred])

    cascade_rows = []
    for target_rate in (0.10, 0.15, 0.20, 0.25, 0.30):
        thr = _weighted_quantile(cal_score_ref, cal_w_ref, 1.0 - target_rate)
        escalate = ev_score >= thr
        casc_pred = np.where(escalate, t2_pred_idx, t1_pred_idx)
        c_f1 = float(f1_score(y_ev, casc_pred, labels=range(5), average="macro", zero_division=0))
        c_acc = float((casc_pred == y_ev).mean())
        cascade_rows.append(dict(
            target_esc_cal=target_rate,
            actual_esc_eval=round(float(escalate.mean()), 4),
            risk_threshold=round(thr, 4),
            cascade_macro_f1=round(c_f1, 4),
            cascade_acc=round(c_acc, 4),
            delta_vs_t1=round(c_f1 - b2_f1, 4),
            delta_vs_t2=round(c_f1 - gm_f1, 4),
        ))
    summary = dict(
        split=split_label,
        mcnemar=dict(both_ok=both_ok, b2_fixed=b_fix, b2_broke=c_brk, both_err=both_err, exact_p=pval),
        t1_macro_f1=b2_f1,
        t2_macro_f1=gm_f1,
        cascade_sweep=cascade_rows,
    )
    write_json(RUN / f"{split_label}_cascade_ucci.json", summary)
    print(pd.DataFrame(cascade_rows))


# ==============================================================================
# 2. THỰC THI ĐỐI CHIẾU LIÊN NHÁNH (DEV / REAL_TEST)
# ==============================================================================
if GUARDMERGE_RUN_DIR:
    gm_dir = Path(GUARDMERGE_RUN_DIR)
    # Khớp UCCI Isotonic Router DUY NHẤT trên cal (dùng cal_temp_weights & nhiệt độ đã đóng băng)
    active_temp = float(cal_payload["temperature"])
    z_cal, y_cal = arrays(cal)
    p_cal = softmax(z_cal / active_temp, axis=1)
    top2_cal = np.sort(p_cal, axis=1)[:, -2:]
    margin_cal = top2_cal[:, 1] - top2_cal[:, 0]
    err_cal = (p_cal.argmax(1) != y_cal).astype(np.float64)
    iso = IsotonicRegression(y_min=0.0, y_max=1.0, increasing=False, out_of_bounds="clip")
    iso.fit(margin_cal, err_cal, sample_weight=cal_temp_weights)
    cal_score_ref = _ucci_route_score(iso, margin_cal)

    if report_key in ("dev", "audit"):
        gm_pred_path = gm_dir / f"{report_key}_predictions.jsonl"
        assert gm_pred_path.exists(), f"Không thấy {gm_pred_path}"
        _run_mcnemar_cascade(report_key, final_pred, gm_pred_path, RUN / f"{report_key}_predictions.jsonl", iso, cal_score_ref, cal_temp_weights, active_temp)

    for rt_tag, rt_pred_obj, pred_fname in (
        ("real_test", real_test_pred, "real_test_predictions.jsonl"),
        ("real_test_c1", real_test_c1_pred, "real_test_c1_predictions.jsonl"),
        ("real_test_c2", real_test_c2_pred, "real_test_c2_predictions.jsonl"),
    ):
        gm_rt_path = gm_dir / pred_fname
        b2_rt_path = RUN / pred_fname
        if rt_pred_obj is not None and gm_rt_path.exists() and b2_rt_path.exists():
            _run_mcnemar_cascade(rt_tag, rt_pred_obj, gm_rt_path, b2_rt_path, iso, cal_score_ref, cal_temp_weights, active_temp)
else:
    print("Bỏ qua đối chiếu Cascade (chưa đặt GUARDMERGE_RUN_DIR).")

---
<a id="sec-10"></a>
## 10. Phụ lục: Quy trình thực nghiệm chuẩn, Đối chiếu API & Tái kiểm chứng CLI

### 10.1. Quy trình thực nghiệm 3 giai đoạn

1. **Giai đoạn 1 — So sánh ứng viên song song (`DATA_SCHEME = "dev_80_10_10"`, `EVALUATE_REAL_TEST = False`):**
   - Chạy `RUN_MODE = "smoke"` (~15 giây) để xác nhận GPU, kiểm định CRF/LEG mức từ, padding và cơ chế lưu checkpoint.
   - Chuyển `RUN_MODE = "full"` và chạy song song các backbone ứng viên: `BACKBONE_PRESET = "visobert"`, `"bamibert"` (`lr_backbone=5e-5`), `"mmbert_base"` (`lr_backbone=4.5e-5`), `"mmbert_small"`, cùng Nhánh 1 `v2_guardmerge_colab.ipynb` (khớp 100% `fit` 10.713, `dev` 1.340, `cal` 1.339; dùng chung tiêu chí chọn checkpoint `real_aligned` với trọng số `0.50/0.15/0.25/0.10`).
   - **Ma trận Ablation (giữ `SEED = 42`, tắt lần lượt từng thành phần):**
     - Kiến trúc: `USE_BYTE_CNN = False` · `USE_HAT_ZOOM = False` · `USE_BIO_CRF = False` · `USE_ORC = False` · `NUM_SUBCENTERS = 1`.
     - Huấn luyện bền vững: `CONSISTENCY_WEIGHT = 0.0` / `FGM_EPS = 0.0` · và tắt tăng cường `ACCENT_AUG = 0.0, SURFACE_AUG = 0.0`.
2. **Giai đoạn 2 — Huấn luyện trên tập dữ liệu mở rộng (tùy chọn sau khi xác định kiến trúc & `EPOCHS` trên `dev`):**
   - Đổi `DATA_SCHEME = "final_90_10"` (`fit = 12.053`, `cal = 1.339`) hoặc `DATA_SCHEME = "final_100_human_cal"` (`fit = 13.392` + `HUMAN_CAL_PATH` do người gán), huấn luyện đủ `EPOCHS` cố định rồi hiệu chuẩn và đóng băng `calibration.json`.
3. **Giai đoạn 3 — Đánh giá một lần duy nhất trên Bộ Test khóa (`334 C0 + 668 C1 + 668 C2 = 1.670` mẫu):**
   - Bật `EVALUATE_REAL_TEST = True` trên đúng `RUN` đã đóng băng `calibration.json` (không đổi `RUN` hash, tự động chạy cả Dòng `#5` thô lẫn Dòng `#6` chuẩn hóa đầu vào khi `EVAL_WITH_NORMALIZER = True`).
   - Tái kiểm chứng tổng hợp bằng bộ chấm CLI chuẩn của kho mã nguồn:
     ```bash
     python -m src.v2.eval.evaluate \
       --gold data/v2/test_day_du.jsonl data/v2/test_perturb_C1.jsonl data/v2/test_perturb_C2.jsonl \
       --pred <RUN>/real_test_all_predictions.jsonl \
       --meta data/v2/test_day_du.jsonl \
       --out results/v2/<ten_cau_hinh>/metrics.json
     ```

### 10.2. Đối chiếu API & Điều kiện kiểm chứng thực nghiệm

| Hạng mục | Cách áp dụng trong notebook | Tài liệu tham chiếu |
| :--- | :--- | :--- |
| **Môi trường & Lưu trữ** | Xác thực phiên bản gói đang nạp trong kernel (`transformers==4.57.6`, `tokenizers==0.22.2`, `safetensors==0.6.2`, `huggingface-hub==0.36.0`, `scikit-learn==1.9.1`); ghi `config.json` & `environment.json` ngay khi tạo `RUN`. | [Colab FAQ](https://research.google.com/colaboratory/faq.html) · [Transformers 4.57.6](https://pypi.org/project/transformers/4.57.6/) |
| **Dữ liệu & Chống rò rỉ** | Ghim commit `DATA_REV`, kiểm tra HTTP retry và SHA-256 (tương thích LF/CRLF/BOM). `StratifiedGroupKFold` giữ nguyên nhóm trùng và cách ly 100% 6 tệp khóa khỏi `HUMAN_CAL_PATH`. | [StratifiedGroupKFold](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedGroupKFold.html) · [SafeTensors](https://huggingface.co/docs/safetensors/api/torch) |
| **Kiến trúc & Độ chính xác số** | Backbone chạy AMP BF16/FP16; toàn bộ head (Byte-CNN, HAT, BIO-CRF, ORC) chạy FP32 tường minh; tự kiểm định CRF với vét cạn $3^W$ và kiểm định bất biến padding trước khi train. | [PyTorch AMP](https://docs.pytorch.org/docs/stable/notes/amp_examples.html) · [SDPA](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html) |
| **Tối ưu & Khôi phục** | Depth-Normalized LLRD chuẩn hóa tỷ lệ LR tầng đáy/đỉnh theo chiều sâu; `last.pt` lưu trọng số, trạng thái AdamW, scheduler, scaler và xác minh khớp `config` khi resume. | [AdamW](https://docs.pytorch.org/docs/stable/generated/torch.optim.AdamW.html) · [Checkpointing](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html) |
| **Hiệu chuẩn & Thống kê** | SciPy `minimize_scalar` tối ưu nhiệt độ $T$; bootstrap phân tầng nhóm cho 95% CI; kiểm định McNemar hai phía (`scipy.stats.binomtest`) và UCCI `IsotonicRegression` chỉ khớp trên `cal`. | [SciPy Optimize](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize_scalar.html) · [IsotonicRegression](https://scikit-learn.org/stable/modules/generated/sklearn.isotonic.IsotonicRegression.html) |
| **Benchmark & Nén INT8** | Đồng bộ CUDA (`torch.cuda.synchronize`) trước khi đọc thời gian; đo tách bạch trọng số vs đỉnh VRAM; nén INT8 cả `nn.Embedding` (`PerRowInt8Embedding`, scale FP32) lẫn `nn.Linear`. | [CUDA Memory](https://docs.pytorch.org/docs/stable/generated/torch.cuda.max_memory_allocated.html) · [Quantization](https://docs.pytorch.org/docs/stable/quantization.html) |